# CareMatch: find the right care in India, with the evidence behind every answer

**CMU × Databricks Datathon, Fall 2026** · Code: https://github.com/DDaniel-code/carematch · Built from commit `c78b23e + local changes to carematch/`

A patient needs a specific kind of care. CareMatch takes a **service** (dialysis, C-section, NICU, trauma care, emergency
care, burn care, cardiac surgery, MRI, CT) and a **place** (city, district, PIN code or coordinates) and returns up to three
facilities from the Virtue Foundation dataset. Every recommendation quotes the facility text it rests on, says how strong
that evidence is and how far away the facility is, lists what is still uncertain, and gives the questions to ask by phone.
When the records do not support a recommendation, it says so instead of padding the list.

For planners it answers the next question: **which districts are far from any facility with credible evidence, and do
those districts also show high need** in NFHS-5?

| Judging criterion | Where in this notebook |
| --- | --- |
| Exploration | Part 3: what a record is, how far each field can be trusted, what the claims can and cannot prove |
| Modeling | Parts 1, 4, 5, 7: claim classification, evidence tiers, location confidence, banded ranking, quality checks, evaluation, sensitivity; a region-held-out lead model |
| Impact | Parts 6, 7: district access gaps × NFHS need, four recommended actions, and which 3–10 facilities to verify first for the most new coverage |

**How to run.** On Databricks Free Edition (serverless): import this file and *Run all*; Part 2 reads the three Unity
Catalog tables through Spark. Locally: put the exported CSVs in `data/` and run all cells. The outputs saved in this file
come from a local run on the full exported tables (`notebooks/00_export_data.py` in the repo); the same cells run
unchanged on Databricks, and Part 2 prints which source was used.

**How this file is made.** Part 1 is the repo's `carematch` package inlined module by module by
`scripts/build_notebook.py`, so this notebook, the Databricks App and the unit tests run the same code.
The same engine is deployed as a Databricks App (Streamlit) for health workers; see the video.


In [1]:
import csv, json, math, os, re, statistics, sys, time
from collections import Counter
from types import ModuleType

# Use a private package so importing these modules cannot shadow an installed CareMatch.
package = ModuleType('_carematch_submission')
package.__path__ = []
sys.modules[package.__name__] = package

def load_embedded(name, source):
    fullname = package.__name__ + '.' + name
    module = ModuleType(fullname)
    module.__package__ = package.__name__
    module.__file__ = os.path.join(os.getcwd(), 'carematch', name + '.py')
    sys.modules[fullname] = module
    setattr(package, name, module)
    exec(compile(source, module.__file__, 'exec'), module.__dict__)

ON_DATABRICKS = 'spark' in globals()
print('Running on', 'Databricks (Spark)' if ON_DATABRICKS else 'local Python', sys.version.split()[0])


def show(rows, cols, headers=None, limit=None):
    """Print a list of dicts as an aligned text table."""
    headers = headers or cols
    rows = rows[:limit] if limit else rows
    cells = [[('' if r.get(c) is None else str(r.get(c))) for c in cols] for r in rows]
    widths = [max([len(h)] + [len(row[i]) for row in cells]) for i, h in enumerate(headers)]
    line = lambda vals: '  '.join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(headers))
    print(line(['-' * w for w in widths]))
    for row in cells:
        print(line(row))


Running on local Python 3.14.6


## Part 1 · The engine

The 3,230 lines below embed the same modules used by the app. The patient ranking uses deterministic rules.
Part 7 also evaluates a separate statistical model for selecting facilities to verify by phone.


**`carematch/textutil.py`** · Parsing helpers: the dataset stores lists as JSON text; coordinates are checked against India; straight-line distance.

In [2]:
load_embedded('textutil', r'''"""Parsing and geometry helpers.

Adapted from the feature/search branch of Sa1kyo121/carematch-india (Charlie's search MVP):
JSON-array field parsing, URL extraction, coordinate validation and haversine distance.
"""
import hashlib
import json
import math
import re
import unicodedata
from urllib.parse import urlsplit

EMPTY_STRINGS = {'', '[]', 'null', 'none', 'nan'}


def text(value):
    return '' if value is None else str(value).strip()


def normalize(value):
    """Unicode-normalize, casefold and collapse whitespace."""
    return ' '.join(unicodedata.normalize('NFKC', text(value)).casefold().split())


def is_empty(value):
    return normalize(value) in EMPTY_STRINGS


def split_camel(value):
    """'obstetricsAndGynecology' -> 'obstetrics And Gynecology'."""
    return re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text(value))


def stable_id(value, length=16):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(payload.encode()).hexdigest()[:length]


def fragments(value):
    """Return the string items of a JSON-array-as-string field; never JSON object keys.

    Null elements and empty strings are dropped; exact duplicates are removed in order.
    """
    out = []

    def walk(v):
        if v is None:
            return
        if isinstance(v, dict):
            for item in v.values():
                walk(item)
            return
        if isinstance(v, (list, tuple)):
            for item in v:
                walk(item)
            return
        if isinstance(v, (bool, int, float)):
            return  # numbers inside JSON are not evidence text
        raw = text(v)
        if is_empty(raw):
            return
        if raw[:1] in '[{"':
            try:
                parsed = json.loads(raw)
            except (ValueError, TypeError):
                parsed = raw
            if parsed != raw:
                walk(parsed)
                return
        out.append(raw)

    walk(value)
    seen, unique = set(), []
    for item in out:
        if item not in seen:
            seen.add(item)
            unique.append(item)
    return unique


def source_urls(value):
    """Explicit http(s) URLs only; no inferred domains, no network calls."""
    urls = set()
    for part in fragments(value):
        for url in re.findall(r'https?://[^\s<>"\',;\]\}]+', part):
            url = url.rstrip(').')
            try:
                parsed = urlsplit(url)
            except ValueError:
                continue
            if parsed.hostname and not parsed.username and not parsed.password:
                urls.add(url)
    return sorted(urls)


def valid_coords(lat, lon):
    """(lat, lon) floats if finite and inside a generous India bounding box, else None."""
    try:
        if isinstance(lat, bool) or isinstance(lon, bool):
            return None
        lat, lon = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if not (math.isfinite(lat) and math.isfinite(lon)):
        return None
    if not (6.0 <= lat <= 37.5 and 68.0 <= lon <= 98.5):
        return None
    return lat, lon


def haversine_km(a, b):
    lat1, lon1, lat2, lon2 = map(math.radians, (*a, *b))
    h = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 6371.0088 * 2 * math.asin(math.sqrt(min(1.0, max(0.0, h))))
''')

**`carematch/services.py`** · The nine services: which phrases are direct evidence, which are only related departments, which contexts are false hits ("dental trauma"), negation and referral patterns, what each service's evidence can and cannot show, and what to ask by phone.

In [3]:
load_embedded('services', r'''"""Service vocabulary: what text counts as evidence for each service.

Patterns are case-insensitive regexes matched against single claim fragments (one JSON-array item).
  direct    - names the service itself ("Provides hemodialysis", "14 dialysis machines")
  related   - department / prerequisite terms that make the service plausible but do not prove it
  exclude   - contexts that make a direct hit a false positive (e.g. "dental trauma")
Specialty enum values (camelCase in `specialties`) are listed separately and only ever count as related.

Calibrated on the full facilities table on 2026-11-21; see docs/evidence_calibration.md.
"""

SERVICES = {
    'dialysis': {
        'label': 'Dialysis',
        'scope': 'Records show a dialysis unit or dialysis treatment; whether a slot is free today must be confirmed.',
        'direct': [r'\b(ha?emo|peritoneal\s+)?dialy[sz](is|es)\b', r'\bartificial kidney\b', r'\bcrrt\b', r'\bcapd\b'],
        'related': [r'\bnephrolog', r'\bkidney transplant', r'\brenal (care|unit|replacement)'],
        'related_specialties': ['nephrology', 'pediatricNephrology', 'renalTransplantationUrology'],
        'exclude': [],
        'needs_inpatient': False,
        'ask': ['Is the dialysis unit running today, and is a slot free?',
                'Is it haemodialysis, peritoneal dialysis, or both?',
                'Is it free or covered under PM-JAY / the national dialysis programme?'],
    },
    'c_section': {
        'label': 'C-section (caesarean delivery)',
        'scope': 'Records show caesarean deliveries are performed. They cannot show emergency readiness (surgeon, anaesthetist and blood on duty now); confirm by phone.',
        'direct': [r'\bca?esarean', r'\bc[- ]?sections?\b', r'\blscs\b', r'\bcemonc\b',
                   r'\bemergency obstetric (care|surgery)'],
        'related': [r'\bobstetric', r'\blabou?r (room|ward|suite)', r'\bmaternity\b', r'\bhigh[- ]risk pregnanc',
                    r'\bblood bank\b', r'\banaesthe|\banesthe'],
        'related_specialties': ['gynecologyAndObstetrics', 'obstetricsAndMaternityCare', 'obstetricsAndGynaecology',
                                'obstetricsAndGynecology', 'maternalFetalMedicineOrPerinatology', 'anesthesia'],
        'exclude': [r'\bveterinar'],
        'needs_inpatient': True,
        'ask': ['Can you do an emergency C-section right now (surgeon and anaesthetist on duty)?',
                'Is blood available on site?', 'Is there a newborn care unit if the baby needs it?'],
    },
    'nicu': {
        'label': 'NICU (newborn intensive care)',
        'scope': 'Records show a NICU or neonatal intensive care; bed availability must be confirmed.',
        'direct': [r'\bnicu\b', r'\bneonatal intensive', r'\bnewborn intensive', r'\bsncu\b',
                   r'\bspecial newborn care', r'\blevel[- ](ii|iii|2|3)\b.{0,20}neonat'],
        'related': [r'\bneonat', r'\bincubator', r'\bradiant warmer', r'\bphototherapy', r'\bpicu\b', r'\bpaediatric icu|\bpediatric icu'],
        'related_specialties': ['neonatologyPerinatalMedicine', 'pediatricCriticalCareMedicine'],
        'exclude': [],
        'needs_inpatient': True,
        'ask': ['Is a NICU bed free right now?', 'Do you have a ventilator / CPAP for newborns?',
                'Is a paediatrician or neonatologist on duty 24 hours?'],
    },
    'trauma': {
        'label': 'Trauma care',
        'scope': 'Records show trauma or accident care; whether the team can receive a patient now must be confirmed.',
        'direct': [r'\btrauma\b', r'\bpolytrauma', r'\baccident (and|&) emergency', r'\baccident care\b', r'\btraumatology'],
        'related': [r'\bemergency\b', r'\bcasualty\b', r'\bicu\b', r'\bintensive care', r'\bneurosurg', r'\borthopa?edic', r'\bblood bank'],
        'related_specialties': ['emergencyMedicine', 'criticalCareMedicine', 'neurosurgery', 'orthopedicSurgery',
                                'burnAndTraumaPlasticSurgery', 'generalSurgery'],
        'exclude': [r'\bdental|\btooth|\bteeth|\bfacial|\bmaxillofacial|\beye\b|\bocular|\bpsycholog|\bemotional|\bptsd\b'
                    r'|\bbirth trauma|\bcounsel|\bnasal|\bnose\b|\bear\b|\bsports injur'],
        'needs_inpatient': True,
        'ask': ['Can you receive a road-accident patient now (surgeon, CT and blood available)?',
                'Is there an ICU bed free?', 'Do you have a neurosurgeon / orthopaedic surgeon on call?'],
    },
    'emergency': {
        'label': 'Emergency care',
        'scope': 'Records show emergency care is offered. Round-the-clock cover is only shown when the quote says 24x7; confirm by phone.',
        'direct': [r'\bemergency (department|room|ward|services?|care|unit|medicine|wing)', r'\bcasualty\b',
                   r'\baccident (and|&) emergency', r'\b24\s*[x×/*]\s*7 emergency', r'\bemergency 24\s*[x×/*]\s*7'],
        'related': [r'\bambulance', r'\b24\s*[x×/*]\s*7', r'\bicu\b', r'\bintensive care', r'\bround[- ]the[- ]clock'],
        'related_specialties': ['emergencyMedicine', 'criticalCareMedicine', 'pediatricEmergencyMedicine'],
        'exclude': [r'\bdental|\btooth|\bteeth|\beye\b|\bocular|\bcontracept|\bveterinar|\bpharmac'],
        'needs_inpatient': True,
        'ask': ['Is the emergency department open now, and is a doctor on duty?', 'Do you have an ambulance?'],
    },
    'burn': {
        'label': 'Burn care',
        'scope': 'Records show burn treatment or a burn unit; bed availability must be confirmed.',
        'direct': [r'\bburns?\s+(unit|ward|care|centre|center|management|treatment|surgery|patients?|injur|icu|wound)',
                   r'\b(treatment|management|care|surgery) of burns?\b', r'\bburns? (and|&) (plastic|trauma|reconstruct)',
                   r'\bpost[- ]?burn', r'\bplastic surgery including burns'],
        'related': [r'\bplastic surg', r'\bskin graft', r'\breconstructive surg'],
        'related_specialties': ['burnAndTraumaPlasticSurgery', 'plasticSurgery', 'reconstructivePlasticSurgery'],
        'exclude': [r'\bburnout', r'\bheart ?burn', r'\bsun ?burn', r'\bacid burn|\brazor burn'],
        'needs_inpatient': True,
        'ask': ['Do you have a burn unit or burn ward with a free bed?', 'Is a plastic surgeon available?'],
    },
    'cardiac_surgery': {
        'label': 'Cardiac surgery',
        'scope': 'Records show heart surgery (bypass, valve, cardiothoracic); scheduling and cost must be confirmed.',
        'direct': [r'\bcardiac surg', r'\bheart surg', r'\bopen[- ]heart', r'\bcabg\b', r'\bcoronary artery bypass',
                   r'\bcardio[- ]?thoracic', r'\bvalve (replacement|repair|surgery)', r'\bctvs\b'],
        'related': [r'\bcath(eteri[sz]ation)? ?lab', r'\bangioplast', r'\bangiograph', r'\bcardiolog', r'\bicu\b|\bccu\b'],
        'related_specialties': ['cardiacSurgery', 'cardiothoracicSurgery', 'interventionalCardiology', 'cardiology',
                                'pediatricCardiology', 'vascularSurgery'],
        'exclude': [],
        'needs_inpatient': True,
        'ask': ['Do you perform heart surgery (bypass / valve) on site, and when is the next slot?',
                'Is it covered under PM-JAY or a state scheme?'],
    },
    'mri': {
        'label': 'MRI scan',
        'scope': 'Records show an MRI scanner or MRI scans; whether the machine is working today must be confirmed.',
        'direct': [r'\bmri\b', r'\bmagnetic resonance'],
        'related': [r'\bradiolog', r'\bct scan|\bct\b', r'\bimaging\b'],
        'related_specialties': ['radiology', 'diagnosticRadiology', 'neuroradiology', 'interventionalRadiology'],
        'exclude': [],
        'needs_inpatient': False,
        'ask': ['Is the MRI machine working today, and what is the waiting time?', 'What does a scan cost?'],
    },
    'ct_scan': {
        'label': 'CT scan',
        'scope': 'Records show a CT scanner or CT scans; night-time availability must be confirmed.',
        'direct': [r'\bct\b(?!\s*(?:g|scan(?:ning)? report))', r'\bcomputed tomography', r'\bcat scan', r'\b\d+[- ]slice'],
        'related': [r'\bradiolog', r'\bmri\b', r'\bimaging\b'],
        'related_specialties': ['radiology', 'diagnosticRadiology', 'neuroradiology'],
        'exclude': [r'\bct\s*simulator'],
        'needs_inpatient': False,
        'ask': ['Is the CT scanner working right now (including at night)?', 'What does a scan cost?'],
    },
}

# A direct hit in the same fragment as one of these is not an in-house claim.
INDIRECT = (r'\btie[- ]?up|\boutsourc|\breferr(al|ed|ing)\b|\brefer(s)? (patients )?to\b'
            r'|\bvia\b|\bthrough (a |an )?(partner|associate)|\bvisible in (the )?(photo|image)|\bphotos? (show|indicate)'
            r'|\bplanned\b|\bproposed\b|\bcoming soon\b|\bupcoming\b|\bunder construction\b|\bwill (be|soon)\b'
            r'|\bto be (started|launched|set up)\b|\bwas located\b|\bformerly\b|\bpreviously\b')

# Run jointly with another organisation ("PET-CT in partnership with Neuberg Diagnostics"): still an on-site claim,
# not outsourcing, but the card asks the user to confirm the equipment or unit is there and working.
PARTNER = r'\bpartner(ed|ship|s)?\b|\bin association with\b|\bin collaboration with\b|\bcollaborat'

# Negation must sit right next to the service term (checked in a short window before / after the hit).
NEG_BEFORE = r'\b(no|without|lacks?|lacking|not|excluding|except|other than)\s+(\w+\s+){0,2}$'
NEG_AFTER = (r'^\W{0,3}(\w+\W+){0,3}?(is |are )?(not (available|offered|provided|done|performed)|unavailable'
             r'|discontinued|closed (down|permanently)|suspended)\b')

# A fragment that enumerates many departments ("Specialties include A, B, C, D ...") names the
# service only as a list item: it counts as a direct claim for 'moderate' but not as corroboration.
LIST_MIN_COMMAS = 4

# Facility types that cannot plausibly provide any of these services.
INCOMPATIBLE_TYPES = {'dentist', 'pharmacy', 'farmacy', 'doctor'}
# facilityTypeId is often just "clinic"; a name like "Patra Dental Clinic" tells us more.
SPECIALTY_NAME = (r'\b(dental|dentist|dentistry|teeth|tooth|orthodont|eye|netra|vision|ophthalm|optical|skin|derma'
                  r'|cosmetic|hair|homoeo|homeo|ayurved|unani|siddha|physiotherap|rehab|de-?addiction|nasha mukti'
                  r'|psychiatr|pathology lab|diagnostic lab|laborator)\w*\b')


def service_keys():
    return list(SERVICES)


SERVICE_ALIASES = {
    'emergency': 'emergency', 'er': 'emergency', 'casualty': 'emergency', 'emergency room': 'emergency',
    'heart surgery': 'cardiac_surgery', 'bypass surgery': 'cardiac_surgery',
    'ct': 'ct_scan', 'ct scan': 'ct_scan', 'cat scan': 'ct_scan',
    'c section': 'c_section', 'caesarean': 'c_section', 'cesarean': 'c_section', 'high-risk delivery': 'c_section',
    'burns': 'burn', 'burn care': 'burn', 'burn unit': 'burn',
    'newborn icu': 'nicu', 'neonatal icu': 'nicu', 'kidney dialysis': 'dialysis', 'accident': 'trauma',
}


def resolve_service(text):
    """Map free text ('C-section', 'dialysis', 'nicu') to a service key, or None."""
    import re
    t = ' '.join((text or '').strip().lower().replace('_', ' ').split())
    if t.replace(' ', '_') in SERVICES:
        return t.replace(' ', '_')
    if t in SERVICE_ALIASES:
        return SERVICE_ALIASES[t]
    for key, spec in SERVICES.items():
        if t == spec['label'].lower() or any(re.search(p, t) for p in spec['direct']):
            return key
    return None
''')

**`carematch/evidence.py`** · Every claim (one item of a `procedure` / `equipment` / `capability` list, or one description sentence) is classified on its own: direct, listed, indirect (referral, outsourced, planned), negative, mention, related. Claims become a tier: Strong, Moderate, Weak or None, with caps for facility type, small clinics and contradictions. Also detects claims about another facility.

In [4]:
load_embedded('evidence', r'''"""Turn one facility record into cited evidence and an evidence tier for one service.

Every evidence item is a single claim fragment copied verbatim from a named column, so any
statement we make can be traced back to the facility text.
"""
import re
from difflib import SequenceMatcher

from .services import (INCOMPATIBLE_TYPES, INDIRECT, LIST_MIN_COMMAS, NEG_AFTER, NEG_BEFORE, PARTNER, SERVICES,
                       SPECIALTY_NAME)
from .textutil import fragments, normalize, split_camel, stable_id, text

CLAIM_FIELDS = ('procedure', 'equipment', 'capability')   # structured claim lists
TEXT_FIELDS = ('description',)                            # free prose
TIERS = ('strong', 'moderate', 'weak', 'none')

_compiled = {}


def _patterns(service):
    if service not in _compiled:
        spec = SERVICES[service]
        c = lambda ps: [re.compile(p, re.I) for p in ps]
        _compiled[service] = {
            'direct': c(spec['direct']), 'related': c(spec['related']), 'exclude': c(spec['exclude']),
            'specialties': {s.lower() for s in spec['related_specialties']},
        }
    return _compiled[service]


_INDIRECT = re.compile(INDIRECT, re.I)
_NEG_BEFORE = re.compile(NEG_BEFORE, re.I)
_NEG_AFTER = re.compile(NEG_AFTER, re.I)


def classify_fragment(fragment, service, field):
    """Return (kind, matched_text) for one fragment, or None.

    kind: direct | listed | indirect | negative | mention | related
    """
    pats = _patterns(service)
    for pat in pats['direct']:
        m = pat.search(fragment)
        if not m:
            continue
        if any(x.search(fragment) for x in pats['exclude']):
            continue
        if _NEG_BEFORE.search(fragment[max(0, m.start() - 40):m.start()]) or _NEG_AFTER.search(fragment[m.end():m.end() + 40]):
            return 'negative', m.group(0)
        if _INDIRECT.search(fragment):
            return 'indirect', m.group(0)
        if field in TEXT_FIELDS:
            return 'mention', m.group(0)
        if fragment.count(',') >= LIST_MIN_COMMAS:
            return 'listed', m.group(0)
        return 'direct', m.group(0)
    for pat in pats['related']:
        m = pat.search(fragment)
        if m and not any(x.search(fragment) for x in pats['exclude']):
            return 'related', m.group(0)
    return None


def extract_evidence(row, service):
    """All evidence items for one facility row and one service, strongest first."""
    fid = row.get('unique_id') or stable_id(row)
    items = []
    for field in CLAIM_FIELDS + TEXT_FIELDS:
        for frag in fragments(row.get(field)):
            hit = classify_fragment(frag, service, field)
            if hit:
                kind, term = hit
                items.append({'evidence_id': 'ev-' + stable_id([fid, field, frag], 12),
                              'field': field, 'text': frag, 'matched': term, 'kind': kind})
    wanted = _patterns(service)['specialties']
    listed = sorted({s for s in fragments(row.get('specialties')) if s.lower() in wanted})
    for spec in listed:
        items.append({'evidence_id': 'ev-' + stable_id([fid, 'specialties', spec], 12),
                      'field': 'specialties', 'text': split_camel(spec), 'raw': spec, 'matched': spec, 'kind': 'related'})
    order = {'direct': 0, 'listed': 1, 'negative': 2, 'indirect': 3, 'mention': 4, 'related': 5}
    field_order = {'procedure': 0, 'equipment': 1, 'capability': 2, 'description': 3, 'specialties': 4}
    items.sort(key=lambda e: (order[e['kind']], field_order[e['field']], len(e['text'])))
    return items


GENERIC_NAME_WORDS = {
    'hospital', 'hospitals', 'clinic', 'clinics', 'nursing', 'home', 'medical', 'centre', 'center', 'care', 'health',
    'healthcare', 'multispeciality', 'multispecialty', 'multi', 'super', 'speciality', 'specialty', 'the', 'and', 'of',
    'dr', 'research', 'institute', 'dental', 'eye', 'children', 'child', 'women', 'womens', 'general', 'city',
    'government', 'govt', 'district', 'civil', 'memorial', 'trust', 'charitable', 'maternity', 'surgical', 'heart',
    'kidney', 'cancer', 'diagnostic', 'diagnostics', 'pvt', 'ltd', 'private', 'limited', 'new', 'sri', 'shri', 'st',
    'fertility', 'ivf', 'dialysis', 'ortho', 'orthopaedic', 'orthopedic', 'neuro', 'skin', 'ent', 'mother', 'baby',
}
_FACILITY_NAME_RE = re.compile(
    r"\b((?:[A-Z][\w&'.-]*\.?\s+){1,5})(Hospitals?|Clinic|Nursing Home|Medical (?:Centre|Center|College)|Health ?Care)\b")


def _name_tokens(name):
    return {t for t in re.findall(r'[a-z]+', normalize(name)) if t not in GENERIC_NAME_WORDS and len(t) > 1}


def foreign_mentions(evidence, row, other_cities, canon=lambda c: c):
    """Signs that a claim describes a different facility (records that merged two sources).

    other_cities: compiled regex of well-known city names, or None; canon maps spelling variants to one key
    (Bangalore/Bengaluru). Returns a list of short reasons.
    """
    own_city = canon(normalize(row.get('address_city')))
    own = _name_tokens(row.get('name'))
    home = {normalize(row.get(k)) for k in ('address_city', 'address_stateOrRegion', 'address_line2', 'address_line3')}
    home_text = ' '.join(home)
    reasons = []
    for e in evidence:
        if e['kind'] not in ('direct', 'listed'):
            continue
        for m in _FACILITY_NAME_RE.finditer(e['text']):
            tokens = _name_tokens(m.group(0))
            similar = SequenceMatcher(None, ''.join(sorted(tokens)), ''.join(sorted(own))).ratio() >= 0.75
            if tokens and own and not tokens & own and not similar:
                reasons.append(f'a claim names another facility ("{m.group(0).strip()}")')
        if other_cities is not None:
            for m in other_cities.finditer(normalize(e['text'])):
                city = m.group(0)
                if canon(city) != own_city and city not in home_text and city not in normalize(row.get('name')):
                    reasons.append(f'a claim places the facility in {city.title()}')
    return sorted(set(reasons))


_BRANCH_SPLIT = re.compile(r'\s*(?:,|\s-\s|\s–\s|\()\s*')


def record_mixing(row, other_cities, canon=lambda c: c, limit=3):
    """Whole-record check: does the record carry text about other facilities, branches or cities?

    Looks at every claim and the description, not only the service evidence. Returns up to `limit` reasons.
    """
    name = text(row.get('name'))
    parts = _BRANCH_SPLIT.split(name, maxsplit=1)
    brand, branch = parts[0], (parts[1] if len(parts) > 1 else '')
    brand_re = re.compile(re.escape(brand) + r'\s*(?:,|-|–)\s*([A-Z][\w .]{2,30})') if len(brand) >= 5 else None
    branch_tokens = _name_tokens(branch)
    frags = [f for field in ('procedure', 'equipment', 'capability', 'description') for f in fragments(row.get(field))]
    own = _name_tokens(name)
    initials = ''.join(w[0] for w in re.findall(r'[a-z]+', normalize(name)) if w not in ('of', 'and', 'the'))
    own_city = canon(normalize(row.get('address_city')))
    home_text = ' '.join(normalize(row.get(k)) for k in ('address_city', 'address_stateOrRegion', 'address_line1',
                                                          'address_line2', 'address_line3', 'name'))
    names, cities, reasons = {}, {}, set()
    for f in frags:
        seen = set()
        for m in _FACILITY_NAME_RE.finditer(f):
            words = re.findall(r'[a-z]+', normalize(m.group(1)))
            tokens = _name_tokens(m.group(0))
            if not tokens or not own or tokens & own or (words and words[0] in GENERIC_LEADS):
                continue
            if initials.startswith(''.join(sorted(tokens))) or ''.join(sorted(tokens)) in initials:
                continue   # "CIMS Hospital" is Care Institute of Medical Sciences itself
            if SequenceMatcher(None, ''.join(sorted(tokens)), ''.join(sorted(own))).ratio() >= 0.75:
                continue
            seen.add((' '.join(sorted(tokens)), m.group(0).strip()))
        for key, label in seen:
            names.setdefault(key, {'label': label, 'frags': []})['frags'].append(f)
        if other_cities is not None:
            for city in {m.group(0) for m in other_cities.finditer(normalize(f))}:
                if canon(city) != own_city and city not in home_text:
                    cities.setdefault(canon(city), []).append(f)
        if brand_re and branch_tokens:
            for m in brand_re.finditer(f):
                other = _name_tokens(m.group(1)) - LOCALITY_WORDS
                first = m.group(1).split()[0].lower()
                if other and not other & (branch_tokens - LOCALITY_WORDS) and first not in BRANCH_STOP \
                        and len(m.group(1).split()) <= 4:
                    reasons.add(f'the record also describes another branch ("{brand}, {m.group(1).strip()}")')
    for entry in names.values():
        if len(entry['frags']) >= 3:
            reasons.add(f'{len(entry["frags"])} statements describe another facility ("{entry["label"]}")')
    for city, fs in cities.items():
        if len(fs) >= 3:
            reasons.add(f'{len(fs)} statements place the facility in {city.title()}')
    return sorted(reasons)[:limit]


GENERIC_LEADS = {'top', 'find', 'best', 'related', 'associated', 'special', 'general', 'government', 'govt', 'medical',
                 'super', 'multi', 'first', 'leading', 'nearby', 'other', 'private', 'public', 'all', 'private', 'list',
                 'empanelled', 'ayushman', 'district', 'civil', 'teaching', 'partner', 'affiliated', 'tertiary',
                 'breast', 'liver', 'heart', 'kidney', 'eye', 'skin', 'dental', 'cancer', 'fertility', 'women', 'child',
                 'medindia', 'practo', 'justdial', 'lybrate', 'credihealth', 'more', 'hexahealth', 'docindia', 'sehat'}
LOCALITY_WORDS = {'road', 'rd', 'nagar', 'street', 'colony', 'sector', 'layout', 'main', 'cross', 'marg', 'chowk',
                  'branch', 'campus', 'unit', 'east', 'west', 'north', 'south', 'extension', 'phase'}
BRANCH_STOP = {'providing', 'provides', 'offers', 'offering', 'is', 'has', 'with', 'located', 'a', 'an', 'the', 'and',
               'india', 'pvt', 'private', 'limited', 'ltd'}


def _number(value):
    m = re.search(r'\d+(\.\d+)?', str(value or ''))
    return float(m.group(0)) if m else None


def assess(row, service, evidence=None):
    """Evidence tier for (facility, service) plus plain-language reasons and flags.

    Rules (see docs/design.md). A "claim" is a distinct sentence; the same sentence copied into two
    columns counts once. "Listed" = the service appears only inside a long department list.
      strong   - >= 2 distinct specific claims found in >= 2 columns, or >= 2 distinct specific claims plus a related
                 department/specialty
      moderate - at least one specific or listed claim
      weak     - only indirect claims (referral/outsourced/planned), prose mentions, or related departments
      none     - nothing relevant
    Caps: incompatible facility type -> weak; small clinic for an inpatient service -> moderate;
          a claim contradicted by a negative one -> moderate.
    """
    spec = SERVICES[service]
    ev = extract_evidence(row, service) if evidence is None else evidence
    specific = [e for e in ev if e['kind'] == 'direct']
    listed = [e for e in ev if e['kind'] == 'listed']
    texts = {}
    for e in specific:
        texts.setdefault(normalize(e['text']), set()).add(e['field'])
    n_specific = len(texts)
    specific_fields = set().union(*texts.values()) if texts else set()
    related = [e for e in ev if e['kind'] == 'related']
    negative = [e for e in ev if e['kind'] == 'negative']
    indirect = [e for e in ev if e['kind'] in ('indirect', 'mention')]

    reasons, flags = [], []
    if n_specific >= 2 and len(specific_fields) >= 2:
        tier = 'strong'
        reasons.append(f"{n_specific} distinct claims across {', '.join(sorted(specific_fields))}")
    elif n_specific >= 2 and (related or listed):
        tier = 'strong'
        reasons.append(f"{n_specific} distinct claims in {', '.join(sorted(specific_fields))}, plus related department evidence")
    elif n_specific >= 1:
        tier = 'moderate'
        reasons.append(f"one specific claim ({', '.join(sorted(specific_fields))})" if n_specific == 1
                       else f"{n_specific} claims in {', '.join(sorted(specific_fields))} with no corroborating department")
    elif listed:
        tier = 'moderate'
        reasons.append('named only inside a department list')
    elif indirect:
        tier = 'weak'
        kinds = sorted({'referral/outsourced/planned' if e['kind'] == 'indirect' else 'description only' for e in indirect})
        reasons.append('only ' + ' and '.join(kinds) + ' mentions')
    elif related:
        tier = 'weak'
        reasons.append('only a related department is listed, the service itself is never named')
    else:
        tier = 'none'

    def cap(level, why, flag):
        nonlocal tier
        if TIERS.index(tier) < TIERS.index(level):
            tier = level
            reasons.append(why)
        flags.append(flag)

    if negative:
        if specific or listed:
            cap('moderate', 'some text says the service is not available', 'contradictory_claims')
        else:
            reasons.append('text says the service is not available')
            flags.append('negative_claim')
            tier = 'none'
    ftype = normalize(row.get('facilityTypeId'))
    capacity = _number(row.get('capacity'))
    name_hit = re.search(SPECIALTY_NAME, text(row.get('name')), re.I)
    if name_hit and service in ('mri', 'ct_scan') and re.match(r'(lab|patholog|diagnostic)', name_hit.group(0), re.I):
        name_hit = None   # diagnostic labs can legitimately run scanners
    if ftype in INCOMPATIBLE_TYPES and tier in ('strong', 'moderate'):
        cap('weak', f"listed as a {ftype}, which is unlikely to provide {spec['label'].lower()}", 'facility_type_mismatch')
    elif name_hit and not re.search(r'\bhospital|\bmedical college|\binstitute of medical', text(row.get('name')), re.I) \
            and tier in ('strong', 'moderate'):
        cap('weak', f'the name suggests a {name_hit.group(0).lower()} specialist, unlikely to provide '
                    f"{spec['label'].lower()}", 'facility_type_mismatch')
    elif (spec['needs_inpatient'] and ftype == 'clinic' and (capacity is None or capacity < 20)
          and tier in ('strong', 'moderate')):
        cap('moderate', 'listed as a small clinic; inpatient capability unconfirmed', 'small_clinic_claim')
    if indirect and tier in ('strong', 'moderate'):
        flags.append('also_has_indirect_claims')
    if tier in ('strong', 'moderate') and any(re.search(PARTNER, e['text'], re.I) for e in specific + listed):
        flags.append('partner_claim')

    return {'service': service, 'tier': tier, 'reasons': reasons, 'flags': sorted(set(flags)), 'evidence': ev}
''')

**`carematch/geo.py`** · Turns the user's place (city, district, PIN, or "lat, lon") into a point with a stated precision, and judges each facility's position by comparing its coordinates, its PIN centre and its city centre.

In [5]:
load_embedded('geo', r'''"""Locations: turn user input into a point, and judge how far to trust each facility's position.

User input can be "lat, lon", a 6-digit PIN code, or a city / district name (optionally "name, state").
Facility positions come from the record's latitude/longitude, cross-checked against the India Post PIN
directory and against the other facilities listed in the same city.
"""
import re
import statistics

from .textutil import haversine_km, normalize, text, valid_coords

STATES = ['Andaman And Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
          'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu And Kashmir', 'Jharkhand',
          'Karnataka', 'Kerala', 'Ladakh', 'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya',
          'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana',
          'The Dadra And Nagar Haveli And Daman And Diu', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal']
_STATE_KEYS = {normalize(s): s for s in STATES}
_STATE_ALIASES = {
    'nct of delhi': 'Delhi', 'new delhi': 'Delhi', 'delhi ncr': 'Delhi', 'tamilnadu': 'Tamil Nadu', 'orissa': 'Odisha',
    'up': 'Uttar Pradesh', 'u.p.': 'Uttar Pradesh', 'mp': 'Madhya Pradesh', 'punjab region': 'Punjab',
    'j&k': 'Jammu And Kashmir', 'jammu & kashmir': 'Jammu And Kashmir', 'pondicherry': 'Puducherry',
    'uttaranchal': 'Uttarakhand', 'chattisgarh': 'Chhattisgarh', 'dadra and nagar haveli': 'The Dadra And Nagar Haveli And Daman And Diu',
    'daman and diu': 'The Dadra And Nagar Haveli And Daman And Diu', 'telengana': 'Telangana', 'west bangal': 'West Bengal',
}

# Facility city spellings that refer to the same place (verified counts in docs/data_profile.md).
CITY_ALIASES = {
    'bangalore': 'bengaluru', 'bengaluru': 'bengaluru', 'gurgaon': 'gurugram', 'gurugram': 'gurugram',
    'new delhi': 'delhi', 'delhi': 'delhi', 'trivandrum': 'thiruvananthapuram', 'cochin': 'kochi',
    'vizag': 'visakhapatnam', 'bombay': 'mumbai', 'madras': 'chennai', 'calcutta': 'kolkata',
    'poona': 'pune', 'mysore': 'mysuru', 'baroda': 'vadodara', 'benares': 'varanasi', 'banaras': 'varanasi',
    'allahabad': 'prayagraj', 'calicut': 'kozhikode', 'mangalore': 'mangaluru', 'belgaum': 'belagavi',
}

PIN_RE = re.compile(r'\b([1-9]\d{5})\b')
COORD_RE = re.compile(r'^\s*(-?\d{1,2}(?:\.\d+)?)\s*[, ]\s*(-?\d{2,3}(?:\.\d+)?)\s*$')

# Trust thresholds (km)
PIN_SPREAD_OK = 60        # a PIN whose post offices span more than this is a poor reference point
COORD_PIN_CONFLICT = 50   # facility coordinates this far from their PIN centroid -> location conflict
CITY_OUTLIER = 100        # this far from the median of other facilities in the same city -> outlier


def canonical_state(value):
    key = normalize(value).replace('state of ', '')
    return _STATE_KEYS.get(key) or _STATE_ALIASES.get(key)


def city_key(value):
    key = normalize(value)
    return CITY_ALIASES.get(key, key)


def clean_pin(value):
    m = PIN_RE.search(text(value).replace(' ', ''))
    return m.group(1) if m else None


def _num(v):
    try:
        f = float(v)
        return f if f == f else None
    except (TypeError, ValueError):
        return None


class Gazetteer:
    """PIN, district and city lookups built from the PIN directory and the facilities table."""

    def __init__(self, pincode_rows, facility_rows):
        self.pins = {}
        districts = {}
        for r in pincode_rows:
            pin = clean_pin(r.get('pincode'))
            if not pin:
                continue
            centre = valid_coords(r.get('lat'), r.get('lon'))
            box = [_num(r.get(k)) for k in ('lat_min', 'lat_max', 'lon_min', 'lon_max')]
            spread = None
            if centre and None not in box:
                spread = haversine_km((box[0], box[2]), (box[1], box[3]))
            state = canonical_state(r.get('statename'))
            entry = {'pin': pin, 'district': text(r.get('district')).title(), 'state': state,
                     'centre': centre, 'spread_km': spread, 'n_offices': int(_num(r.get('n_offices')) or 0)}
            self.pins.setdefault(pin, []).append(entry)
        self._drop_misgeocoded_pins()
        for entries in self.pins.values():
            for entry in entries:
                if entry['centre'] and entry['spread_km'] is not None and entry['spread_km'] <= PIN_SPREAD_OK and entry['district']:
                    districts.setdefault((normalize(entry['district']), entry['state']), []).append(entry['centre'])
        self.districts = {}
        for (dname, state), pts in districts.items():
            self.districts.setdefault(dname, []).append(self._cluster(pts, state, f'{dname.title()} district'))

        cities = {}
        for r in facility_rows:
            c = valid_coords(r.get('latitude'), r.get('longitude'))
            key = city_key(r.get('address_city'))
            if c and key and key not in ('null', 'none'):
                state = canonical_state(r.get('address_stateOrRegion'))
                cities.setdefault((key, state), []).append(c)
        self.cities = {}
        for (key, state), pts in cities.items():
            if len(pts) >= 2:
                self.cities.setdefault(key, []).append(self._cluster(pts, state, key.title()))

    def _drop_misgeocoded_pins(self):
        """Some post offices were geocoded to a same-named place in another state (e.g. Aurangabad, Bihar
        PINs 824xxx placed in Aurangabad, Maharashtra). PINs sharing leading digits are in the same postal
        region, so a centre far from its region's median is discarded."""
        self.dropped_pins = 0
        for digits, limit in ((2, 400.0), (3, 150.0)):
            groups = {}
            for pin, entries in self.pins.items():
                for e in entries:
                    if e['centre']:
                        groups.setdefault(pin[:digits], []).append(e['centre'])
            medians = {k: (statistics.median(p[0] for p in v), statistics.median(p[1] for p in v))
                       for k, v in groups.items() if len(v) >= 5}
            for pin, entries in self.pins.items():
                ref = medians.get(pin[:digits])
                for e in entries:
                    if e['centre'] and ref and haversine_km(e['centre'], ref) > limit:
                        e['centre'], e['spread_km'] = None, None
                        self.dropped_pins += 1

    @staticmethod
    def _cluster(pts, state, label):
        centre = (statistics.median(p[0] for p in pts), statistics.median(p[1] for p in pts))
        radius = statistics.median(haversine_km(centre, p) for p in pts) if len(pts) > 1 else 5.0
        return {'label': label, 'state': state, 'centre': centre, 'radius_km': radius, 'n': len(pts)}

    # ---- PIN ---------------------------------------------------------------
    def pin_info(self, pin):
        """Best entry for a PIN plus whether it spans several districts/states."""
        entries = self.pins.get(pin)
        if not entries:
            return None
        good = [e for e in entries if e['centre']]
        best = max(good or entries, key=lambda e: e['n_offices'])
        return {**best, 'districts': sorted({e['district'] for e in entries}),
                'states': sorted({e['state'] for e in entries if e['state']}),
                'reliable': bool(best['centre']) and best['spread_km'] is not None and best['spread_km'] <= PIN_SPREAD_OK}

    # ---- user input --------------------------------------------------------
    def resolve(self, query):
        """Return a location dict, or a dict with 'error'."""
        raw = text(query)
        if not raw:
            return {'error': 'Please enter a place: city, district, 6-digit PIN code, or "lat, lon".'}
        m = COORD_RE.match(raw)
        if m:
            c = valid_coords(m.group(1), m.group(2))
            if not c:
                return {'error': f'"{raw}" is not a coordinate inside India.'}
            return self._loc(raw, 'coordinates', f'{c[0]:.4f}, {c[1]:.4f}', c, 0.1)
        pin = clean_pin(raw) if re.fullmatch(r'\s*\d[\d ]{4,6}\d?\s*', raw) else None
        if pin:
            return self._resolve_pin(raw, pin)
        name, _, state_hint = raw.partition(',')
        state = canonical_state(state_hint) if state_hint.strip() else None
        lookups = (('city', self.cities, city_key(name)), ('district', self.districts, normalize(name)))
        # With a state hint, first look for an exact state match in either index, then relax.
        passes = (True, False) if state else (False,)
        for strict in passes:
            for kind, index, key in lookups:
                found = index.get(key)
                if not found and kind == 'district' and len(key) >= 3:
                    # word-level fallback: "leh" -> "leh ladakh", "kanpur" -> "kanpur nagar"
                    found = [o for k, opts in index.items() if key in k.split() for o in opts]
                options = [o for o in found or [] if o['state']] or (found or [])
                if strict:
                    options = [o for o in options if o['state'] == state]
                if not options:
                    continue
                options = sorted(options, key=lambda o: -o['n'])
                best = options[0]
                label = best['label'] + (f", {best['state']}" if best['state'] else '')
                loc = self._loc(raw, kind, label, best['centre'], max(3.0, best['radius_km']), state=best['state'])
                if not strict and state and best['state'] != state:
                    loc['notes'].append(f'No "{name.strip()}" found in {state}; showing {label}.')
                others = {o['state'] for o in options[1:] if o['state']}
                if kind == 'city':   # a district elsewhere can share the city's name (Aurangabad, Bihar)
                    others |= {o['state'] for o in self.districts.get(normalize(name)) or [] if o['state']}
                others = sorted(others - {best['state']})
                if others and not state:
                    loc['alternatives'] = [f'{name.strip()}, {s}' for s in others]
                    loc['notes'].append(f'"{name.strip()}" also exists in {", ".join(others)}; showing {best["state"]}. '
                                        f'Type "{name.strip()}, {others[0]}" to switch.')
                if kind == 'city':
                    loc['notes'].append('Distances are measured from the centre of the listed facilities in this city.')
                return loc
        return {'error': f'Could not find "{raw}". Try a 6-digit PIN code, a district name, or "lat, lon".'}

    def _resolve_pin(self, raw, pin):
        info = self.pin_info(pin)
        if info and info['reliable']:
            loc = self._loc(raw, 'pin', f"PIN {pin} ({info['district']}, {info['state']})", info['centre'],
                            max(2.0, info['spread_km'] / 2), state=info['state'], district=info['district'])
            if len(info['districts']) > 1:
                loc['notes'].append(f"PIN {pin} covers several districts: {', '.join(info['districts'])}.")
            return loc
        # Fall back to nearby PINs sharing the first 4 digits (same postal sub-region).
        near = [e['centre'] for p, es in self.pins.items() if p.startswith(pin[:4]) for e in es
                if e['centre'] and e['spread_km'] is not None and e['spread_km'] <= PIN_SPREAD_OK]
        if near:
            c = (statistics.median(p[0] for p in near), statistics.median(p[1] for p in near))
            loc = self._loc(raw, 'pin_area', f'PIN area {pin[:4]}xx', c, 25.0)
            loc['notes'].append(f'PIN {pin} has no reliable location in the India Post directory; '
                                f'using the centre of nearby PINs {pin[:4]}xx (approximate).')
            return loc
        return {'error': f'PIN {pin} was not found in the India Post directory.'}

    @staticmethod
    def _loc(query, kind, label, centre, precision, state=None, district=None):
        return {'query': query, 'kind': kind, 'label': label, 'lat': round(centre[0], 5), 'lon': round(centre[1], 5),
                'precision_km': round(precision, 1), 'state': state, 'district': district, 'notes': [], 'alternatives': []}


def facility_location(row, gaz):
    """Position used for distance + a confidence level and plain flags for one facility record.

    Three independent signals are compared: the record's coordinates, the centre of its PIN code,
    and the centre of other facilities listed in the same city. When two agree and one does not,
    the odd one out is treated as the error (e.g. coordinates 500 km away while PIN and city agree).
    """
    coord = valid_coords(row.get('latitude'), row.get('longitude'))
    pin = clean_pin(row.get('address_zipOrPostcode'))
    info = gaz.pin_info(pin) if pin else None
    pin_pt = info['centre'] if info and info['reliable'] else None
    state_raw = text(row.get('address_stateOrRegion'))
    state = canonical_state(state_raw)
    city = [c for c in gaz.cities.get(city_key(row.get('address_city'))) or []
            if c['n'] >= 5 and (not state or c['state'] == state)]
    city_best = max(city, key=lambda c: c['n']) if city else None
    city_pt = city_best['centre'] if city_best else None
    # "Same city" radius scales with how spread out the city's facilities are (25-60 km).
    city_km = min(CITY_OUTLIER, max(25.0, 4 * city_best['radius_km'])) if city_best else CITY_OUTLIER
    flags = []

    if not pin:
        flags.append('pin_missing_or_malformed')
    elif not info:
        flags.append('pin_not_in_directory')
    if state_raw and normalize(state_raw) not in ('null', 'none') and not state:
        flags.append('state_field_not_a_state')
    if state and info and info['states'] and state not in info['states']:
        flags.append('state_pin_mismatch')

    near = lambda a, b, km: a is not None and b is not None and haversine_km(a, b) <= km
    position, basis, confidence = coord, 'facility coordinates', 'high'
    alternatives = []      # other plausible positions; distance uses the farthest (conservative)
    if coord:
        coord_pin = None if pin_pt is None else near(coord, pin_pt, COORD_PIN_CONFLICT)
        coord_city = None if city_pt is None else near(coord, city_pt, city_km)
        pin_city = None if pin_pt is None or city_pt is None else near(pin_pt, city_pt, city_km)
        if coord_pin is False and coord_city is False and pin_city:
            position, basis, confidence = pin_pt, 'PIN code centre (record coordinates contradict PIN and city)', 'approximate'
            flags.append('coords_wrong_used_pin')
        elif coord_pin is False and coord_city:
            flags.append('pin_inconsistent')          # coordinates agree with the city; the PIN looks wrong
            confidence = 'medium'
        elif coord_pin is False or coord_city is False:
            flags.append('location_conflict')         # cannot tell which signal is right
            confidence = 'low'
            alternatives = [p for p in (pin_pt, city_pt) if p is not None and not near(p, coord, 10)]
    elif pin_pt:
        position, basis, confidence = pin_pt, 'PIN code centre (approximate)', 'approximate'
        flags.append('position_from_pin')
    elif city_pt:
        position, basis, confidence = city_pt, 'city centre (approximate)', 'low'
        flags.append('position_from_city')
    else:
        position, basis, confidence = None, None, 'low'
        flags.append('no_location')
    if confidence == 'high' and flags:
        confidence = 'medium'
    return {'position': position, 'alternatives': alternatives, 'basis': basis, 'confidence': confidence, 'flags': flags,
            'pin': pin, 'state': state or (info['states'][0] if info and len(info['states']) == 1 else None),
            'district': info['district'] if info else None}
''')

**`carematch/price.py`** · Service-specific prices, bounds, conditions and exact source text.

In [6]:
load_embedded('price', r'''"""Service-price evidence: documented patient prices for one service, quoted verbatim, never inferred.

A price is read only from a single claim fragment that (1) names the service itself and (2) states an amount in INR.
Each amount is classified before it can be used:
  patient price        what the patient is charged: exact, range, upper bound ("Rs. 350 or less"), lower bound
                       ("starting at"), or explicitly free
  insurer coverage     a scheme or insurer limit ("covers up to INR 5 lakh per family under PM-JAY"): not a price
  capital / equipment  what the facility spent ("machines worth Rs. 40 lakh", "built at a cost of 4.38 crore"): not a price

Never inferred: a public operator is not "free"; a package is not "per session"; an equipment cost is not a patient fee;
one quote is not "the cheapest". Prices are compared only when service, unit and conditions match (see comparable()).
"""
import re

from .services import SERVICES
from .textutil import fragments, normalize, text

FIELDS = ('procedure', 'capability', 'equipment', 'description')

AMOUNT = re.compile(
    r'(?:₹|\brs\.?|\binr\b|\brupees?\b)\s*(\d[\d,]*(?:\.\d+)?)\s*(lakhs?|lacs?|crores?|cr\b|k\b)?'
    r'|(\d[\d,]*(?:\.\d+)?)\s*(lakhs?|lacs?|crores?)?\s*(?:rupees|/-|\binr\b)', re.I)
CAPITAL = re.compile(r'\bworth\b|\bcost of\b|\bbuilt at\b|\bbudget|\ballocat|\binvest|\bestimated cost|\bproject\b'
                     r'|\bsanction|\bgrant(ed)?\b|\bdonat|\bpurchase|\bprocured|\brestoration|\binaugurat|\bfoundation stone'
                     r'|\bconstruct|\bcapital|\bturnover|\brevenue|\bfunds?\b|\bcrores?\b|\bcr\b', re.I)
INSURER = re.compile(r'\bcovers?\b.{0,20}\bup ?to\b|\bcoverage\b|\bcover of\b|\bsum insured|\bper family\b|\bper year\b'
                     r'|\bpm-?jay\b|\bayushman|\bscheme benefit|\bbenefits? (of |up ?to)|\binsur|\bcashless|\bclaim', re.I)
UPPER = re.compile(r'\bor less\b|\bup ?to\b|\bmax(imum|\.)?\b|\bnot (more|exceeding) than\b|\bnot exceeding\b|\bbelow\b'
                   r'|\bunder\b|\bwithin\b|\bat most\b', re.I)
LOWER = re.compile(r'\bstart(s|ing)?( at| from)?\b|\bfrom\b|\bonwards\b|\bminimum\b|\bat least\b|\band above\b', re.I)
RANGE = re.compile(r'\d\s*(?:-|–|to)\s*(?:₹|rs\.?|inr)?\s*\d', re.I)
# "free" must be about money: "infection-free dialysis", "hassle-free", "pain-free", "germ-free" are not prices.
FREE = re.compile(r'(?<![-\w])free of (cost|charge)\b|(?<![-\w])free (dialysis|treatment|of charges)\b|\bat no cost\b'
                  r'|\bwithout (any )?charge', re.I)
NEGATED_FREE = re.compile(r'\b(?:no|not|never|without)\s+(?:\w+\s+){0,2}$', re.I)
NOT_MONEY_FREE = re.compile(r'\b(infection|germ|pain|hassle|stress|worry|risk|cash|paper|smoke|tobacco|drug|toxin|error'
                            r'|complication|contamination|cross[- ]infection|barrier|wait|queue|latex|mercury|radiation)'
                            r'[- ]free\b', re.I)
CLAUSE = re.compile(r';|\|| - (?=[A-Z])|\.\s+(?=[A-Z])')
DISCOUNT = re.compile(r'\b(regular|original|mrp|actual) (price|rate|cost)\b|\b\d+% off\b|\bdiscount', re.I)
UNITS = [('per_session', r'\bper (session|sitting|cycle)\b|/\s*session\b|\beach session\b'),
         ('per_scan', r'\bper (scan|study|test|investigation)\b|/\s*scan\b'),
         ('per_day', r'\bper (day|night|bed[- ]day)\b|/\s*day\b'),
         ('per_procedure', r'\bper (procedure|surgery|operation|delivery)\b'),
         ('per_month', r'\bper month\b|\bmonthly\b|/\s*month\b'),
         ('package', r'\bpackage\b')]
CONDITIONS = [('concessional / subsidised rate', r'\bconcession(al|ary)?\b|\bsubsidi[sz]ed\b|\bcharitable\b|\bfor the (poor|needy)\b|\bbpl\b'),
              ('discounted offer', DISCOUNT.pattern),
              ('scheme / insurance condition', r'\bunder\b.{0,30}\b(scheme|yojana|pm-?jay|ayushman|insurance|esi|cghs)\b'),
              ('restricted eligibility', r'\bfor (bpl|members|employees|staff|students|senior citizens|card ?holders'
                                         r'|beneficiaries|registered|eligible)\b|\bonly for\b|\bentitled\b'),
              ('inclusions stated', r'\binclu(ding|sive of|des)\b'),
              ('dated price', r'\bas of (19|20)\d\d\b|\bin (19|20)\d\d\b')]


def _amount(num, mult):
    value = float(num.replace(',', ''))
    m = (mult or '').lower()
    if m.startswith(('lakh', 'lac')):
        value *= 1e5
    elif m.startswith(('crore', 'cr')):
        value *= 1e7
    elif m == 'k':
        value *= 1e3
    return int(round(value))


RANGE_TAIL = re.compile(r'\s*(?:-|–|to)\s*(?:₹|rs\.?|inr)?\s*(\d[\d,]*(?:\.\d+)?)\s*(lakhs?|lacs?|crores?|k\b)?', re.I)


def amounts(fragment):
    out = []
    for m in AMOUNT.finditer(fragment):
        num, mult = (m.group(1), m.group(2)) if m.group(1) else (m.group(3), m.group(4))
        if num and re.search(r'\d', num):
            out.append((_amount(num, mult), m.start(), m.end()))
            tail = RANGE_TAIL.match(fragment, m.end())       # "Rs 1,200 - 1,800": second number has no currency sign
            if tail and not any(t[1] == tail.start(1) for t in out):
                out.append((_amount(tail.group(1), tail.group(2) or mult), tail.start(1), tail.end()))
    return sorted(set(out), key=lambda t: t[1])


def classify(fragment):
    """(kind, details) for one fragment that contains an amount; kind in patient_price / insurer_coverage /
    capital_or_equipment."""
    found = amounts(fragment)
    if not found:
        return None
    if CAPITAL.search(fragment) and not re.search(r'\bcharge|\bfee|\bper (session|scan|visit)|\bpriced|\brate', fragment, re.I):
        return 'capital_or_equipment', {'amounts': [a for a, _, _ in found]}
    if INSURER.search(fragment) and not re.search(r'\bcharge[sd]?\b|\bpriced\b|\bfee\b|\bper session\b', fragment, re.I):
        return 'insurer_coverage', {'amounts': [a for a, _, _ in found]}
    first = found[0]
    window = fragment[max(0, first[1] - 25):first[2] + 15]
    if len(found) >= 2 and RANGE.search(fragment[found[0][1]:found[1][2]]) and not DISCOUNT.search(fragment):
        bound, lo, hi = 'range', min(found[0][0], found[1][0]), max(found[0][0], found[1][0])
    elif UPPER.search(window):
        bound, lo, hi = 'upper_bound', None, first[0]
    elif LOWER.search(window):
        bound, lo, hi = 'lower_bound', first[0], None
    else:
        bound, lo, hi = 'exact', first[0], first[0]
    unit = next((u for u, p in UNITS if re.search(p, fragment, re.I)), 'unspecified')
    conditions = [label for label, p in CONDITIONS if re.search(p, fragment, re.I)]
    return 'patient_price', {'bound': bound, 'amount_min_inr': lo, 'amount_max_inr': hi, 'unit': unit,
                             'conditions': conditions}


def price_evidence(row, service):
    """(price_info or None, context list). price_info is the single best documented patient price for the service;
    context lists insurer / capital amounts that are NOT prices (shown so they are never mistaken for one)."""
    direct = [re.compile(p, re.I) for p in SERVICES[service]['direct']]
    prices, context = [], []
    for field in FIELDS:
        for frag in fragments(row.get(field)):
            if not any(p.search(frag) for p in direct):
                continue
            # Read money only in the clause(s) that name the service: in "Trauma care ...; medicines supplied free of
            # cost by the TB Division" the "free" is about TB medicines. The quote shown stays the whole fragment.
            scoped = ' ; '.join(c for c in CLAUSE.split(frag) if any(p.search(c) for p in direct))
            if not scoped:
                continue
            financial_text = NOT_MONEY_FREE.sub(' ', scoped)
            free_claim = FREE.search(financial_text)
            if free_claim and not NEGATED_FREE.search(financial_text[:free_claim.start()]) and not amounts(scoped):
                prices.append({'kind': 'patient_price', 'bound': 'free', 'amount_min_inr': 0, 'amount_max_inr': 0,
                               'unit': 'unspecified', 'conditions': [l for l, p in CONDITIONS if re.search(p, frag, re.I)],
                               'scope': service, 'field': field, 'quote': frag})
                continue
            c = classify(scoped)
            if not c:
                continue
            kind, d = c
            if kind == 'patient_price':
                prices.append({'kind': kind, **d, 'scope': service, 'field': field, 'quote': frag})
            else:
                context.append({'kind': kind, 'amounts_inr': d['amounts'], 'field': field, 'quote': frag,
                                'note': 'not a patient price'})
    order = {'exact': 0, 'range': 1, 'upper_bound': 2, 'lower_bound': 3, 'free': 4}
    prices.sort(key=lambda p: (order[p['bound']], p['unit'] == 'unspecified', len(p['quote'])))
    info = prices[0] if prices else None
    if info:
        info = {**info, 'currency': 'INR', 'other_price_quotes': len(prices) - 1}
    return info, context


def interval(info):
    """[low, high] in INR for comparison; None means unbounded on that side."""
    return info['amount_min_inr'], info['amount_max_inr']


def comparison_key(price):
    conditions = tuple(sorted(price.get('conditions', ())))
    # A broad label like "restricted eligibility" can refer to two different patient groups.
    # For qualified offers require matching wording apart from the currency amounts.
    wording = normalize(AMOUNT.sub(' INR ', price.get('quote', ''))) if conditions else ''
    return price['unit'], conditions, wording


def comparable(a, b):
    """Two price_info dicts can be compared only for the same service and the same stated unit (never 'unspecified')."""
    return (a and b and a['scope'] == b['scope'] and comparison_key(a) == comparison_key(b) and a['unit'] != 'unspecified'
            and a['bound'] != 'free' and b['bound'] != 'free')


def surely_cheaper(a, b):
    """True only if every price a allows is below every price b allows (e.g. '<= 350' vs exactly 500)."""
    a_hi, b_lo = a['amount_max_inr'], b['amount_min_inr']
    return a_hi is not None and b_lo is not None and a_hi < b_lo
''')

**`carematch/decision.py`** · Patient-selected preferences and their documented evidence.

In [7]:
load_embedded('decision', r'''"""Optional preference ranking: re-order the credible options by what the user says matters.

The evidence gate, the search radius and the honest "nothing within 300 km" answer are unchanged: this module only
re-orders facilities that recommend() already considers recommendable (Strong or Moderate).

Preferences
  urgency  'default'        no preference model (recommend() keeps its evidence-then-distance order)
           'time_critical'  every minute on the road counts (e.g. bleeding in labour, road accident)
           'can_travel'     the patient can travel further for stronger evidence and better readiness
  cost     'any'            cost is shown but does not change the order
           'low_cost'       prefer facilities whose record documents free / government-scheme / insured care

Score = weighted sum of components in [0, 1]. Every component carries a plain reason and, where it rests on the record,
the verbatim quote and its column. The components are ranking weights built from stated assumptions; none of them is a
probability of being treated or of survival, and the app never presents them that way.
"""
import math
import re

from .price import NOT_MONEY_FREE
from .textutil import fragments, normalize, text

URGENCY = ('default', 'time_critical', 'can_travel')
COST = ('any', 'low_cost')

# Assumptions used to turn straight-line km into an estimated drive time (shown to the user with every estimate).
ROAD_FACTOR = 1.4                     # road distance ~ 1.4 x straight line
SPEED_KMH = {'plains': 40.0, 'hills': 25.0}
HILL_STATES = {'Jammu And Kashmir', 'Ladakh', 'Himachal Pradesh', 'Uttarakhand', 'Sikkim', 'Arunachal Pradesh',
               'Meghalaya', 'Nagaland', 'Manipur', 'Mizoram'}
# How quickly the travel component falls with EXTRA drive time compared with the nearest credible option:
# weight = exp(-extra_minutes / tau). Relative, so the nearest option always scores 1.0 even in remote districts
# where every option is hours away (an absolute decay would flatten to zero there and stop discriminating).
TAU_MINUTES = {'time_critical': 45.0, 'default': 120.0, 'can_travel': 480.0}
# Where even the nearest option is hours away, an extra hour matters relatively less than in a city, and a fixed tau
# would flatten every option to ~0. The decay scale therefore grows with the nearest option's drive time.
TAU_SCALE = {'time_critical': 0.5, 'default': 1.0, 'can_travel': 2.0}

WEIGHTS = {   # evidence, travel, readiness (cost added separately when requested)
    'time_critical': {'evidence': 0.35, 'travel': 0.45, 'readiness': 0.20},
    'default':       {'evidence': 0.45, 'travel': 0.30, 'readiness': 0.25},
    'can_travel':    {'evidence': 0.55, 'travel': 0.10, 'readiness': 0.35},
}
COST_WEIGHT = 0.30
TIER_VALUE = {'strong': 1.0, 'moderate': 0.6}

READINESS = {
    'c_section': ['anaesthesia', 'blood', 'theatre', 'round_the_clock', 'newborn_care'],
    'trauma': ['icu', 'blood', 'ct', 'round_the_clock', 'ambulance', 'neuro_ortho'],
    'emergency': ['round_the_clock', 'ambulance', 'icu'],
    'nicu': ['round_the_clock', 'ventilation', 'paediatrician'],
    'burn': ['icu', 'plastic_surgery', 'round_the_clock'],
    'dialysis': ['nephrologist', 'round_the_clock', 'machines'],
    'cardiac_surgery': ['icu', 'cath_lab', 'blood'],
    'mri': ['round_the_clock', 'scanner_spec'],
    'ct_scan': ['round_the_clock', 'scanner_spec'],
}
SIGNALS = {
    'anaesthesia': (r'\ban(a)?esthe', 'anaesthesia'),
    'blood': (r'\bblood bank|\bblood storage|\bblood (component|transfusion)', 'blood bank / storage'),
    'theatre': (r'\boperation theat(re|er)|\bmodular (ot|operation)|\boperating room', 'operation theatre'),
    'round_the_clock': (r'\b24\s*[x×/*]\s*7\b|\b24 hours\b|\bround[- ]the[- ]clock|\baround the clock', '24x7 service'),
    'newborn_care': (r'\bnicu\b|\bneonatal|\bnewborn care|\bsncu\b', 'newborn care'),
    'icu': (r'\bicu\b|\bintensive care|\bccu\b|\bcritical care', 'ICU / critical care'),
    'ct': (r'\bct\b|\bcomputed tomography', 'CT scanner'),
    'ambulance': (r'\bambulance', 'ambulance'),
    'neuro_ortho': (r'\bneurosurg|\borthopa?edic surg', 'neuro / orthopaedic surgery'),
    'ventilation': (r'\bventilat|\bcpap\b|\bhfov\b', 'newborn ventilation / CPAP'),
    'paediatrician': (r'\bpa?ediatrician|\bneonatologist', 'paediatrician / neonatologist'),
    'plastic_surgery': (r'\bplastic surg|\bskin graft', 'plastic surgery'),
    'nephrologist': (r'\bnephrologist', 'nephrologist'),
    'machines': (r'\b\d+\s*(bed(ded)?\s*)?(dialysis|haemodialysis|hemodialysis) (machines|units|beds)|\b(machines|units)\b.{0,20}dialysis',
                 'number of dialysis machines'),
    'cath_lab': (r'\bcath(eteri[sz]ation)? ?lab', 'cath lab'),
    'scanner_spec': (r'\b\d(\.\d)?\s*t(esla)?\b|\b\d+[- ]slice', 'scanner specification'),
}
COVERED_TEXT = re.compile(r'\bayushman|\bpm-?jay\b|\bpmjay\b|\bjan arogya|\baarogyasri|\barogyasri|\bmjpjay|\bphule jan'
                          r'|\bcmchis\b|\bchiranjeevi|\bswasthya sathi|\bbiju swasthya|\besic?\b|\bcghs\b|\bemployees.? state insurance'
                          r'|\bfree of cost\b|\bfree (treatment|services?|consultation|surgery|delivery|dialysis)|\bcharitable'
                          r'|\bsubsidi[sz]ed|\bno cost\b|\bnominal (fee|charge)', re.I)
INSURED_TEXT = re.compile(r'\bcashless|\btpa\b|\binsurance', re.I)
# A 24x7 pharmacy or laboratory is not clinical readiness for the service.
NON_CLINICAL = re.compile(r'\blab\b|\blaborator|\bpharmac|\bchemist|\bcanteen|\bcafeteria|\bparking|\batm\b|\bcollection centre'
                          r'|\bmicrobiolog|\bpatholog|\bblood test|\bhome (sample|collection)', re.I)
NEGATED = re.compile(r'\b(no|not|without|isn.?t|doesn.?t|does not|non)\W+(\w+\W+){0,3}$', re.I)
CLAIM_FIELDS = ('procedure', 'equipment', 'capability', 'description')


def normalize_preferences(prefs):
    prefs = prefs or {}
    urgency = prefs.get('urgency') or 'default'
    cost = prefs.get('cost') or 'any'
    if urgency not in URGENCY:
        raise ValueError(f'urgency must be one of {URGENCY}')
    if cost not in COST:
        raise ValueError(f'cost must be one of {COST}')
    return {'urgency': urgency, 'cost': cost, 'applied': urgency != 'default' or cost != 'any'}


def _find(row, pattern, clinical_only=False):
    """First fragment (column, verbatim text) matching pattern without a nearby negation."""
    rx = re.compile(pattern, re.I) if isinstance(pattern, str) else pattern
    for field in CLAIM_FIELDS:
        for frag in fragments(row.get(field)):
            searchable = NOT_MONEY_FREE.sub(lambda match: " " * len(match[0]), frag) if rx is COVERED_TEXT else frag
            m = rx.search(searchable)
            if m and not NEGATED.search(frag[max(0, m.start() - 30):m.start()]) \
                    and not (clinical_only and NON_CLINICAL.search(frag)):
                return field, frag
    return None


def cost_info(row):
    """Documented affordability: covered (public / scheme / charitable), insured (cashless / insurance), or unknown."""
    op = normalize(row.get('operatorTypeId'))
    if op in ('public', 'government'):
        return {'category': 'covered', 'value': 1.0, 'reason': 'operator listed as public', 'field': 'operatorTypeId',
                'quote': text(row.get('operatorTypeId'))}
    hit = _find(row, COVERED_TEXT)
    if hit:
        return {'category': 'covered', 'value': 1.0, 'reason': 'record mentions free care or a government scheme',
                'field': hit[0], 'quote': hit[1]}
    hit = _find(row, INSURED_TEXT)
    if hit:
        return {'category': 'insured', 'value': 0.6, 'reason': 'record mentions cashless / insurance payment',
                'field': hit[0], 'quote': hit[1]}
    return {'category': 'unknown', 'value': None, 'reason': 'the record says nothing about cost or payment schemes',
            'field': None, 'quote': None}


def readiness(row, service):
    signals = READINESS.get(service, [])
    found, missing = [], []
    for key in signals:
        pattern, label = SIGNALS[key]
        hit = _find(row, pattern, clinical_only=True)
        if hit:
            found.append({'signal': label, 'field': hit[0], 'quote': hit[1]})
        else:
            missing.append(label)
    return {'value': len(found) / len(signals) if signals else 0.0, 'found': found, 'missing': missing}


def drive_minutes(km, state, origin_state=None):
    hills = state in HILL_STATES or origin_state in HILL_STATES
    speed = SPEED_KMH['hills'] if hills else SPEED_KMH['plains']
    return km * ROAD_FACTOR / speed * 60.0, speed


def score(card, row, service, prefs, origin_state=None, baseline_minutes=0.0):
    """Preference score with itemised components for one result card."""
    urgency = prefs['urgency']
    weights = dict(WEIGHTS[urgency])
    cost = cost_info(row)
    use_cost = prefs['cost'] == 'low_cost' and cost['value'] is not None
    if prefs['cost'] == 'low_cost':
        weights = {k: v * (1 - COST_WEIGHT) for k, v in weights.items()}
        weights['cost'] = COST_WEIGHT
    ev = card['evidence'][0] if card['evidence'] else None
    evidence_value = TIER_VALUE.get(card['tier'], 0.0)
    minutes, speed = drive_minutes(card['distance_km'] or 0.0, card.get('state'), origin_state)
    extra = max(0.0, minutes - baseline_minutes)
    tau = max(TAU_MINUTES[urgency], TAU_SCALE[urgency] * baseline_minutes)
    travel_value = math.exp(-extra / tau)
    ready = readiness(row, service)
    comps = [
        {'name': 'evidence', 'value': round(evidence_value, 3), 'weight': round(weights['evidence'], 3),
         'reason': f"{card['tier']} evidence: {card['tier_reasons'][0] if card['tier_reasons'] else ''}",
         'field': ev['field'] if ev else None, 'quote': ev['quote'] if ev else None},
        {'name': 'travel', 'value': round(travel_value, 3), 'weight': round(weights['travel'], 3),
         'reason': (f"{card['distance_km']} km straight-line ≈ {minutes:.0f} min by road"
                    + (f", {extra:.0f} min more than the nearest credible option" if extra >= 1 else ', the nearest credible option')
                    + f" (assumes road ≈ {ROAD_FACTOR}× straight line at {speed:.0f} km/h; an estimate, not a route)"),
         'field': None, 'quote': None},
        {'name': 'readiness', 'value': round(ready['value'], 3), 'weight': round(weights['readiness'], 3),
         'reason': (f"{len(ready['found'])} of {len(ready['found']) + len(ready['missing'])} readiness signals in the record"
                    + (f"; not mentioned: {', '.join(ready['missing'])}" if ready['missing'] else '')),
         'signals': ready['found']},
    ]
    if prefs['cost'] == 'low_cost':
        comps.append({'name': 'cost', 'value': cost['value'], 'weight': COST_WEIGHT if use_cost else 0.0,
                      'reason': cost['reason'], 'field': cost['field'], 'quote': cost['quote']})
    total_w = sum(c['weight'] for c in comps)
    total = sum(c['value'] * c['weight'] for c in comps if c['value'] is not None) / total_w if total_w else 0.0
    return {'score': round(total, 4), 'components': comps, 'cost': cost, 'drive_minutes_estimate': round(minutes),
            'note': 'A ranking weight built from stated assumptions, not a probability of treatment or survival.'}


def assumptions(prefs):
    return {
        'urgency': prefs['urgency'], 'cost': prefs['cost'],
        'weights': {**WEIGHTS[prefs['urgency']], **({'cost': COST_WEIGHT} if prefs['cost'] == 'low_cost' else {})},
        'travel': f'drive time = km × {ROAD_FACTOR} / speed ({SPEED_KMH["plains"]:.0f} km/h; {SPEED_KMH["hills"]:.0f} km/h in hill '
                  f'states); travel weight = exp(-extra minutes vs the nearest credible option / tau), '
                  f'tau = max({TAU_MINUTES[prefs["urgency"]]:.0f} min, {TAU_SCALE[prefs["urgency"]]:g} × nearest option drive time)',
        'evidence': 'Strong = 1.0, Moderate = 0.6',
        'readiness': 'share of service-specific readiness signals mentioned in the record (24x7, blood bank, anaesthesia, ...)',
        'cost': 'covered (public, government scheme, free/charitable) = 1.0; cashless/insurance = 0.6; '
                'not documented = listed separately, never scored as expensive',
        'not_a_probability': 'Scores order the options; they are not chances of being treated or of survival.',
    }
''')

**`carematch/review.py`** · Eight automatic quality checks run on every answer, including citation fidelity: every quote must appear word for word in the named column of the facility record.

In [8]:
load_embedded('review', r'''"""Automatic quality review run on every recommendation result.

Each check is deterministic and is shown to the user in the app ("Quality checks: 7/7 passed").
The most important one is citation fidelity: every quoted sentence must appear verbatim in the named
column of that facility's source record, so the system can never show a claim the data does not contain.
"""
import re

from .evidence import TIERS
from .price import comparison_key
from .textutil import fragments, normalize, text


def _quote_in_record(index, card, ev):
    for i in index.by_id.get(text(card['facility_id']), []):
        raw = index.rows[i].get(ev['field'])
        if ev['field'] == 'description':
            if ev['verbatim'] and ev['verbatim'] in text(raw):
                return True
        elif ev['verbatim'] in fragments(raw):
            return True
    return False


def _check(name, passed, detail=''):
    return {'name': name, 'passed': bool(passed), 'detail': '' if passed else detail}


def review(index, result):
    results = result.get('results', [])
    extras = [c for c in (result.get('farther_stronger'), result.get('nearest_beyond')) if c] + result.get('weak_nearby', [])
    extras += result.get('cost_unknown_options', [])
    prefs_on = (result.get('preferences') or {}).get('applied', False)
    every = results + extras + result.get('runner_ups', [])
    checks = []

    bad_tier = [c['name'] for c in results if c['tier'] not in ('strong', 'moderate')]
    checks.append(_check('At most 3 recommendations, all with Strong or Moderate evidence',
                         len(results) <= 3 and not bad_tier, f'{len(results)} results; weak/none: {bad_tier}'))

    keys = [re.sub(r'[^a-z0-9]+', ' ', normalize(c['name'])).strip() for c in results]
    checks.append(_check('No facility listed twice', len(keys) == len(set(keys)),
                         'duplicate names: ' + ', '.join(sorted({k for k in keys if keys.count(k) > 1}))))

    missing = [(c['name'], e['field'], e['quote'][:60]) for c in every for e in c['evidence']
               if not _quote_in_record(index, c, e)]
    n_quotes = sum(len(c['evidence']) for c in every)
    checks.append(_check(f'Every quote appears word for word in the facility record ({n_quotes} quotes)',
                         not missing, f'not found: {missing[:3]}'))

    no_quote = [c['name'] for c in results if not c['evidence']]
    checks.append(_check('Every recommendation cites at least one quote', not no_quote, f'no quote: {no_quote}'))

    bad_dist = [c['name'] for c in every if c['distance_km'] is not None and c['distance_km'] <= 0]
    checks.append(_check('Distances are positive or explicitly unknown (never a fake 0 km)', not bad_dist,
                         f'bad distance: {bad_dist}'))

    band = result.get('band_km')
    inside = all(c['distance_km'] is not None and band is not None
                 and c['distance_km'] <= max(band, c.get('found_within_km') or band) + 0.05 for c in results)
    sort_mode = result.get('sort_mode', 'default')
    sort = result.get('sort') or {}
    if sort_mode in ('nearest', 'price'):
        bad_rank = [c['name'] for c in results
                    if c['location_confidence'] == 'low' or 'possible_merged_record' in c['flags']]
        abstained_ok = bool(results) or bool(sort.get('no_ranked_reason')) or not sort
        checks.append(_check(f'{sort_mode.capitalize()} mode: only branch-resolved options with a certain location are ranked; '
                             'if none qualifies, nothing is ranked and the reason is given',
                             not bad_rank and abstained_ok, f'unresolved but ranked: {bad_rank}'))
    if sort_mode == 'nearest':
        d = [c['distance_km'] for c in results]
        moved = [c for c in results if c.get('safety_reordered')]
        ordered = all(a <= b + 0.05 for a, b in zip(d[len(moved):], d[len(moved) + 1:]))
        checks.append(_check('Nearest mode: ordered by straight-line distance among branch-resolved options with a certain location',
                             ordered and inside, f'distances={d}'))
    elif sort_mode == 'price':
        pc = sort.get('price_comparison', {})
        rest = [c for c in results if not c.get('safety_pinned')]
        pinned_ok = all(not c.get('safety_pinned') or i == 0 for i, c in enumerate(results))
        if pc.get('ranked_by') == 'price':
            hi = [c['price_info']['amount_max_inr'] if c['price_info']['amount_max_inr'] is not None else float('inf')
                  for c in rest if c.get('price_comparable')]
            keys = {comparison_key(c['price_info']) for c in rest if c.get('price_comparable')}
            ordered = all(a <= b for a, b in zip(hi, hi[1:])) and len(keys) <= 1 and \
                all(k[0] != 'unspecified' for k in keys)
        else:
            d = [c['distance_km'] for c in rest]
            ordered = all(a <= b + 0.05 for a, b in zip(d, d[1:]))
        budget = pc.get('budget_km')
        in_budget = all(budget is not None and c['distance_km'] <= budget + 0.05 for c in results)
        honest = not pc.get('cheapest_claim') or pc.get('comparable_quotes', 0) >= 2
        checks.append(_check('Price mode: only like-for-like documented prices (same unit and conditions) are ranked, within '
                             'the extra-distance budget; urgent services keep the nearest safe option first; no "cheapest" claim '
                             'from fewer than two comparable quotes', ordered and in_budget and honest and pinned_ok,
                             f"ranked_by={pc.get('ranked_by')}, in_budget={in_budget}, cheapest_claim={pc.get('cheapest_claim')}"))
    elif prefs_on:
        scores = [c['preference']['score'] for c in results]
        ordered = all(a >= b for a, b in zip(scores, scores[1:]))
        checks.append(_check('Ordered by preference score (highest first); all within the search radius',
                             ordered and inside, f'scores={scores}, inside={inside}'))
        bad = []
        for c in results + result.get('runner_ups', []) + result.get('cost_unknown_options', []):
            rows = [index.rows[i] for i in index.by_id.get(text(c['facility_id']), [])]
            cites = [(x.get('field'), x.get('quote')) for x in c['preference']['components'] if x.get('quote')]
            cites += [(s['field'], s['quote']) for x in c['preference']['components'] for s in x.get('signals', [])]
            for field, quote in cites:
                ok = any((text(r.get(field)) == quote) if field == 'operatorTypeId'
                         else (quote in fragments(r.get(field)) or (field == 'description' and quote in text(r.get(field))))
                         for r in rows)
                if not ok:
                    bad.append((c['name'], field, quote[:50]))
        checks.append(_check('Every preference reason that cites the record quotes it word for word', not bad,
                             f'not found: {bad[:3]}'))
    else:
        # nearest search radius first; inside one radius, tier then distance
        keyseq = [(c.get('found_within_km', band) or 0, TIERS.index(c['tier']),
                   c['distance_km'] if c['distance_km'] is not None else float('inf')) for c in results]
        ordered = all(a[0] < b[0] or (a[0] == b[0] and (a[1] < b[1] or (a[1] == b[1] and a[2] <= b[2] + 0.05)))
                      for a, b in zip(keyseq, keyseq[1:]))
        checks.append(_check('Ordered nearest search radius first, then evidence tier, then distance; all within the radius',
                             ordered and inside, f'order ok={ordered}, inside {band} km={inside}'))

    priced = [c for c in every + [x for key in ('branch_unresolved', 'location_uncertain', 'price_unknown_nearby',
                                                 'price_not_comparable') for x in sort.get(key, [])]
              if c.get('price_info') or c.get('price_context')]
    bad_price = []
    for c in priced:
        rows = [index.rows[i] for i in index.by_id.get(text(c['facility_id']), [])]
        quotes = ([c['price_info']] if c.get('price_info') else []) + list(c.get('price_context') or [])
        for q in quotes:
            if not any(q['quote'] in fragments(r.get(q['field'])) for r in rows):
                bad_price.append((c['name'], q['field'], q['quote'][:50]))
    if priced:
        checks.append(_check('Every price quote appears word for word in the facility record', not bad_price,
                             f'not found: {bad_price[:3]}'))

    ranked_extra = [c['name'] for c in extras + result.get('runner_ups', []) if 'rank' in c]
    numbered = [c.get('rank') for c in results] == list(range(1, len(results) + 1))
    checks.append(_check('Only recommendations are numbered; extras and runner-ups are labelled, not ranked',
                         numbered and not ranked_extra, f'ranks={[c.get("rank") for c in results]}, ranked extras={ranked_extra}'))

    no_ask = [c['name'] for c in results if not c.get('ask_when_calling')]
    checks.append(_check('Every recommendation says what to confirm by phone, and the disclaimer is present',
                         not no_ask and bool(result.get('disclaimer')), f'missing questions: {no_ask}'))

    return {'passed': all(c['passed'] for c in checks), 'checks': checks}
''')

**`carematch/recommend.py`** · The ranking: only Strong or Moderate evidence is recommended; take the nearest distance band (25/75/150/300 km) that has any; inside it, stronger evidence first, then distance. Builds cards with quotes, uncertainties, phone, questions to ask, runner-ups with the reason they lost, and an honest no-answer.

In [9]:
load_embedded('recommend', r'''"""Service + location -> up to 3 facilities, each with quoted evidence, distance and stated uncertainty.

Ranking rule (docs/design.md):
  1. Only facilities with Strong or Moderate evidence are recommended.
  2. Distance bands from the user: 25, 75, 150, 300 km. Use the nearest band that has any recommendable facility.
  3. Inside the band: evidence tier first, then distance, then amount of evidence.
  4. If the band has only Moderate options and the next band has a Strong one, add it as "farther, stronger evidence".
  5. Nothing within 300 km -> no recommendation; show the nearest option beyond and nearby weak mentions, labelled.
Every result also lists runner-ups with the reason each was not chosen, and carries an automatic quality review.
"""
import csv
import re
import sys

from .decision import assumptions as decision_assumptions
from .decision import cost_info, drive_minutes, normalize_preferences
from .decision import score as preference_score
from .evidence import TIERS, assess, foreign_mentions, record_mixing
from .price import comparable as price_comparable
from .price import comparison_key, price_evidence, surely_cheaper
from .geo import CITY_ALIASES, Gazetteer, city_key, facility_location
from .services import SERVICES, resolve_service
from .textutil import fragments, haversine_km, normalize, source_urls, text

CONTRACT_VERSION = '1.1'
BANDS_KM = (25, 75, 150, 300)
MAX_RESULTS = 3
MAX_RUNNER_UPS = 3
DEDUP_KM = 2.0
SORT_MODES = ('default', 'nearest', 'price')
EMERGENCY_SERVICES = {'c_section', 'trauma', 'emergency', 'nicu', 'burn'}
CAN_TRAVEL_EXTRA_KM = 100            # 'can travel' looks at most 2x the nearest option's distance, or +100 km
CITY_MENTION_MIN_FACILITIES = 15     # cities this well represented are used to spot claims about another place

FLAG_TEXT = {
    'contradictory_claims': 'the record contains a statement that this service is not available',
    'small_clinic_claim': 'listed as a small clinic; inpatient capability is unconfirmed',
    'facility_type_mismatch': 'facility type does not usually offer this service',
    'also_has_indirect_claims': 'some text describes referral or outsourcing for this service',
    'coords_wrong_used_pin': 'map coordinates contradicted the PIN code and city, so the PIN code centre is used',
    'pin_inconsistent': 'the PIN code does not match the facility location',
    'location_conflict': 'the address, PIN code and map position disagree; distance uses the farthest of them — confirm the address by phone',
    'position_from_pin': 'no map coordinates; distance is from the PIN code centre',
    'position_from_city': 'no map coordinates or PIN; distance is from the city centre',
    'state_pin_mismatch': 'the state in the record does not match its PIN code',
    'possible_duplicate_listing': 'the same facility appears more than once in the data; listings were merged',
    'possible_merged_record': 'some claims may describe a different facility (the record may merge two sources)',
    'partner_claim': 'a claim says this is run with a partner organisation (not outsourced); confirm the equipment or unit is on site and working',
}
AGGREGATORS = ('practo.com', 'justdial.com', 'facebook.com', 'medindia.net', 'onefivenine.com', 'lybrate.com')


def load_csv(path):
    csv.field_size_limit(sys.maxsize)
    with open(path, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))


def _int(value, upper=5000):
    """First integer in the field, or None if missing or implausible (e.g. '200000' beds)."""
    m = re.search(r'\d+', text(value))
    n = int(m.group(0)) if m else None
    return n if n is not None and 0 < n <= upper else None


def _distance(o, loc):
    """Straight-line km from origin; when location signals conflict, the farthest candidate (conservative)."""
    if loc['position'] is None:
        return None
    return max(haversine_km(o, p) for p in [loc['position']] + loc.get('alternatives', []))


def _name_key(name):
    return re.sub(r'[^a-z0-9]+', ' ', normalize(name)).strip()


class CareIndex:
    """Facilities + gazetteer, with per-service assessments computed once and cached."""

    def __init__(self, facility_rows, pincode_rows):
        self.rows = facility_rows
        self.gaz = Gazetteer(pincode_rows, facility_rows)
        self.locations = [facility_location(r, self.gaz) for r in facility_rows]
        self.by_id = {}
        for i, r in enumerate(facility_rows):
            self.by_id.setdefault(text(r.get('unique_id')), []).append(i)
        big = {k for k, opts in self.gaz.cities.items() if sum(o['n'] for o in opts) >= CITY_MENTION_MIN_FACILITIES}
        names = {n for n in big | {a for a, k in CITY_ALIASES.items() if k in big} if len(n) >= 4}
        self.city_regex = re.compile(r'\b(' + '|'.join(sorted(map(re.escape, names), key=len, reverse=True)) + r')\b') \
            if names else None
        self.mixing = [record_mixing(r, self.city_regex, city_key) for r in facility_rows]
        self.dup_group = self._duplicate_groups()
        self._assessed = {}

    def _duplicate_groups(self):
        """Rows that are the same facility: same normalized name and (same PIN or positions within DEDUP_KM)."""
        parent = list(range(len(self.rows)))

        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i
        by_name = {}
        for i, r in enumerate(self.rows):
            key = _name_key(r.get('name'))
            if key and key not in ('null', 'none'):
                by_name.setdefault(key, []).append(i)
        for idxs in by_name.values():
            for a_pos, a in enumerate(idxs):
                for b in idxs[a_pos + 1:]:
                    la, lb = self.locations[a], self.locations[b]
                    same_pin = la['pin'] and la['pin'] == lb['pin']
                    close = la['position'] and lb['position'] and haversine_km(la['position'], lb['position']) <= DEDUP_KM
                    if same_pin or close:
                        parent[find(b)] = find(a)
        groups = [find(i) for i in range(len(self.rows))]
        self._members = {}
        for i, g in enumerate(groups):
            self._members.setdefault(g, []).append(i)
        return groups

    def duplicates_of(self, i):
        return [j for j in self._members[self.dup_group[i]] if j != i]

    @classmethod
    def from_csv(cls, facilities_path='data/facilities.csv', pincodes_path='data/pincodes.csv'):
        return cls(load_csv(facilities_path), load_csv(pincodes_path))

    def assessments(self, service):
        if service not in self._assessed:
            out = []
            for r in self.rows:
                a = assess(r, service)
                if a['tier'] in ('strong', 'moderate'):
                    merged = foreign_mentions(a['evidence'], r, self.city_regex, city_key)
                    if merged:
                        a['flags'] = sorted(set(a['flags']) | {'possible_merged_record'})
                        a['merge_reasons'] = merged
                out.append(a)
            self._assessed[service] = out
        return self._assessed[service]


def _card(index, i, a, dist):
    row, loc = index.rows[i], index.locations[i]
    phones = fragments(row.get('phone_numbers'))
    official = text(row.get('officialPhone'))
    website = text(row.get('officialWebsite'))
    urls = source_urls(row.get('source_urls'))
    urls = sorted(urls, key=lambda u: (any(d in u for d in AGGREGATORS), len(u)))
    quotes = [e for e in a['evidence'] if e['kind'] in ('direct', 'listed')][:3] or a['evidence'][:2]
    dups = index.duplicates_of(i)
    mixing = index.mixing[i]
    flags = set(a['flags']) | set(f for f in loc['flags'] if f in FLAG_TEXT)
    if dups:
        flags.add('possible_duplicate_listing')
    if mixing:
        flags.add('possible_merged_record')
    flags = sorted(flags)
    uncertainties = [FLAG_TEXT[f] for f in flags if f in FLAG_TEXT]
    uncertainties += [r[0].upper() + r[1:] for r in sorted(set(a.get('merge_reasons', [])) | set(mixing))]
    return {
        'facility_id': row.get('unique_id'),
        'name': text(row.get('name')) or '(name missing)',
        'type': text(row.get('facilityTypeId')) or None,
        'operator': text(row.get('operatorTypeId')) or None,
        'address': ', '.join(x for x in (text(row.get(f'address_line{k}')) for k in (1, 2, 3)) if x and x != 'null'),
        'city': text(row.get('address_city')), 'state': loc['state'], 'pin': loc['pin'], 'district': loc['district'],
        'phone': official if official and official != 'null' else (phones[0] if phones else None),
        'website': website if website and website != 'null' else None,
        'beds': _int(row.get('capacity')), 'doctors': _int(row.get('numberDoctors')),
        'distance_km': max(0.1, round(dist, 1)) if dist is not None else None,
        'distance_basis': loc['basis'], 'location_confidence': loc['confidence'],
        'tier': a['tier'], 'tier_reasons': a['reasons'],
        'evidence': [{'evidence_id': e['evidence_id'], 'field': e['field'], 'quote': e['text'],
                      'verbatim': e.get('raw', e['text']), 'kind': e['kind']} for e in quotes],
        'all_evidence_count': len(a['evidence']),
        'n_claims': len({normalize(e['text']) for e in a['evidence'] if e['kind'] in ('direct', 'listed')}),
        'flags': flags, 'uncertainties': uncertainties,
        'source_urls': urls[:3],
        'ask_when_calling': SERVICES[a['service']]['ask'],
        **({'merged_ids': [index.rows[j].get('unique_id') for j in dups]} if dups else {}),
        **dict(zip(('price_info', 'price_context'), price_evidence(row, a['service']))),
    }


def _dedupe(cards):
    """Merge listings with the same normalized name within DEDUP_KM (or same name and no distance)."""
    kept = []
    for c in cards:
        dup = next((k for k in kept if c['facility_id'] in k.get('merged_ids', [])
                    or (_name_key(k['name']) == _name_key(c['name'])
                        and (k['distance_km'] is None or c['distance_km'] is None
                             or abs(k['distance_km'] - c['distance_km']) <= DEDUP_KM))), None)
        if dup:
            if c['facility_id'] not in dup.setdefault('merged_ids', []):
                dup['merged_ids'].append(c['facility_id'])
            if 'possible_duplicate_listing' not in dup['flags']:
                dup['flags'].append('possible_duplicate_listing')
                dup['uncertainties'].append(FLAG_TEXT['possible_duplicate_listing'])
        else:
            kept.append(c)
    return kept


def _why_not(c, last):
    """One sentence on why a runner-up is below the last recommendation."""
    if c['tier'] == 'weak':
        return 'Not recommended: ' + (c['tier_reasons'][0] if c['tier_reasons'] else 'weak evidence') + '.'
    if TIERS.index(c['tier']) > TIERS.index(last['tier']):
        return f"{c['tier'].capitalize()} evidence ({c['tier_reasons'][0]}) versus {last['tier']} for those above."
    if c['distance_km'] is not None and last['distance_km'] is not None and c['distance_km'] > last['distance_km']:
        return f"Same {c['tier']} evidence, but farther ({c['distance_km']} km vs {last['distance_km']} km)."
    return f"Same {c['tier']} evidence and distance, but fewer supporting claims ({c['n_claims']} vs {last['n_claims']})."


URGENCY_TEXT = {'default': 'balanced', 'time_critical': 'time-critical: travel time weighs most',
                'can_travel': 'can travel: evidence and readiness weigh most'}


def _preference_rank(index, service, prefs, pool, k, origin):
    """Re-order the same recommendable pool by preference score. With cost='low_cost', facilities whose record says
    nothing about cost are listed separately (never scored as expensive)."""
    rows = {}
    base = min((drive_minutes(c['distance_km'], c.get('state'), origin.get('state'))[0] for c in pool
                if c['distance_km'] is not None), default=0.0)
    for c in pool:
        i = index.by_id.get(text(c['facility_id']), [None])[0]
        rows[c['facility_id']] = index.rows[i] if i is not None else {}
        c['preference'] = preference_score(c, rows[c['facility_id']], service, prefs, origin.get('state'), base)
        c['cost'] = c['preference']['cost']
    key = lambda c: (-c['preference']['score'], c['distance_km'] if c['distance_km'] is not None else 1e9, c['facility_id'])
    ranked = sorted(pool, key=key)
    unknown = []
    if prefs['cost'] == 'low_cost':
        documented = [c for c in ranked if c['cost']['category'] != 'unknown']
        # nothing documents cost: fall back to everyone (labelled in why_ranked) and list nothing separately
        unknown = [c for c in ranked if c['cost']['category'] == 'unknown'][:k] if documented else []
        ranked = documented or ranked
    results = ranked[:k]
    label = URGENCY_TEXT[prefs['urgency']] + ('; low cost preferred' if prefs['cost'] == 'low_cost' else '')
    for rank, c in enumerate(results, 1):
        c['rank'] = rank
        c.pop('why_not', None)
        p = c['preference']
        parts = ', '.join(f"{x['name']} {x['value']:.2f}×{x['weight']:.2f}" for x in p['components'] if x['value'] is not None)
        c['why_ranked'] = (f"#{rank} of {len(pool)} credible options by your preferences ({label}): score {p['score']:.2f} "
                           f"= {parts} (weights sum to 1).")
        if prefs['cost'] == 'low_cost' and c['cost']['category'] == 'unknown':
            c['why_ranked'] += ' No option nearby documents its cost, so cost could not be used.'
    for c in unknown:
        c.pop('rank', None)
        c['note'] = 'Cost not documented in the record (it may still be free or covered): ask when calling.'
    shown = {c['facility_id'] for c in results + unknown}
    runner_ups = []
    for c in [c for c in ranked if c['facility_id'] not in shown][:MAX_RUNNER_UPS]:
        c.pop('rank', None)
        last = results[-1]
        diffs = sorted(((lc['value'] or 0) * lc['weight'] - (cc['value'] or 0) * cc['weight'], lc['name'])
                       for lc, cc in zip(last['preference']['components'], c['preference']['components']))
        c['why_not'] = (f"Lower preference score ({c['preference']['score']:.2f} vs {last['preference']['score']:.2f}); "
                        f"biggest gap: {diffs[-1][1]}.")
        runner_ups.append(c)
    return results, runner_ups, unknown


def _tradeoff_note(top, default_first):
    """When the preference pick is farther than the default pick, say what was given up and what was gained (quoted)."""
    if top['facility_id'] == default_first['facility_id'] or top['distance_km'] is None or default_first['distance_km'] is None:
        return None
    extra_km = round(top['distance_km'] - default_first['distance_km'], 1)
    if extra_km < 1:
        return None
    gains = []
    comps = {x['name']: x for x in top['preference']['components']}
    if 'cost' in comps and comps['cost']['quote']:
        gains.append(f"its record documents affordable care: “{comps['cost']['quote']}” [{comps['cost']['field']}]")
    for s in comps['readiness'].get('signals', [])[:2]:
        gains.append(f"{s['signal']}: “{s['quote']}” [{s['field']}]")
    if TIERS.index(top['tier']) < TIERS.index(default_first['tier']):
        gains.append(f"{top['tier']} rather than {default_first['tier']} evidence")
    lost = f"{default_first['name']} is {extra_km} km nearer"
    if comps.get('cost') and (default_first.get('cost') or cost_info({})).get('category') == 'unknown':
        lost += ' (its cost is not documented, which does not mean it is expensive)'
    return f"{lost}. This option was preferred because " + ('; '.join(gains) if gains else 'its overall score was higher') + '.'


PRICE_UNIT_TEXT = {'per_session': 'per session', 'per_scan': 'per scan', 'per_day': 'per day',
                   'per_procedure': 'per procedure', 'per_month': 'per month', 'package': 'package', 'unspecified': 'unit not stated'}


def _price_text(p):
    if not p:
        return 'no price documented'
    unit = PRICE_UNIT_TEXT[p['unit']]
    if p['bound'] == 'free':
        return f'free ({unit})'
    if p['bound'] == 'exact':
        amount = f"₹{p['amount_min_inr']:,}"
    elif p['bound'] == 'range':
        amount = f"₹{p['amount_min_inr']:,}–{p['amount_max_inr']:,}"
    elif p['bound'] == 'upper_bound':
        amount = f"up to ₹{p['amount_max_inr']:,}"
    else:
        amount = f"from ₹{p['amount_min_inr']:,}"
    return f'{amount} {unit}' + (f" ({'; '.join(p['conditions'])})" if p['conditions'] else '')


def _sort_mode(index, service, mode, eligible, k, max_extra_km, bands, origin_pt):
    """'nearest' or 'price' ordering over the same credible (Strong/Moderate) candidates within the last band.

    Branch-resolved = no sign that the record mixes facilities. Location-certain = location confidence is not 'low'.
    Candidates failing either are never ranked: they are listed separately with the reason, so a distance or price is
    never claimed for a branch we cannot pin down.
    """
    pool = sorted([s for s in eligible if s[1] <= bands[-1]],
                  key=lambda s: (round(s[1], 1), s[0], text(index.rows[s[2]].get('unique_id'))))
    cards = _dedupe([_card(index, s[2], s[3], s[1]) for s in pool])
    ranked, unresolved, uncertain = [], [], []
    for c in cards:
        if 'possible_merged_record' in c['flags']:
            c['not_ranked_because'] = 'the record may mix several facilities or branches, so its distance and price cannot be attributed'
            unresolved.append(c)
        elif c['location_confidence'] == 'low':
            c['not_ranked_because'] = 'address, PIN code and map position disagree, so the straight-line distance is uncertain'
            i = index.by_id.get(text(c['facility_id']), [None])[0]
            loc = index.locations[i] if i is not None else {'position': None}
            pts = [loc['position']] + loc.get('alternatives', []) if loc['position'] else []
            if pts:
                c['distance_range_km'] = [round(min(haversine_km(origin_pt, p) for p in pts), 1), c['distance_km']]
            uncertain.append(c)
        else:
            ranked.append(c)
    # Records whose farthest plausible position is beyond the last band but whose nearest one is inside it (e.g. a
    # "Nagpur" listing mapped to Mumbai): never ranked, but shown so their claims and quoted prices are not lost.
    seen = {c['facility_id'] for c in cards}
    for s in eligible:
        if s[1] <= bands[-1] or index.locations[s[2]]['confidence'] != 'low':
            continue
        loc = index.locations[s[2]]
        near_km = min(haversine_km(origin_pt, p) for p in [loc['position']] + loc.get('alternatives', []))
        if near_km > bands[-1] or text(index.rows[s[2]].get('unique_id')) in seen:
            continue
        c = _card(index, s[2], s[3], s[1])
        c['distance_range_km'] = [round(near_km, 1), c['distance_km']]
        if 'possible_merged_record' in c['flags']:
            c['not_ranked_because'] = ('the record may mix several facilities or branches and its location is uncertain, so '
                                       'its distance and price cannot be attributed')
            unresolved.append(c)
        else:
            c['not_ranked_because'] = 'address, PIN code and map position disagree, so the straight-line distance is uncertain'
            uncertain.append(c)
    unresolved.sort(key=lambda c: c.get('distance_range_km', [c['distance_km']])[0])
    uncertain.sort(key=lambda c: c.get('distance_range_km', [c['distance_km']])[0])
    if not ranked:
        # Nothing can be pinned to one branch with a certain location: rank nothing, keep every flagged card and its reason.
        for c in unresolved + uncertain:
            c.pop('rank', None)
        reason = (f"None of the {len(unresolved) + len(uncertain)} credible options nearby can be pinned to one branch with a "
                  'certain location, so no distance or price order is claimed. They are listed with the reason; call to '
                  'confirm the address before travelling.')
        out = {'mode': mode, 'max_extra_km': max_extra_km, 'nearest_credible_km': None, 'ranked_count': 0,
               'no_ranked_reason': reason, 'branch_unresolved': unresolved[:3], 'location_uncertain': uncertain[:3],
               'band_km': None, 'safety_note': None, 'results': [], 'runner_ups': [],
               'method': (f'{mode.capitalize()} mode: Strong/Moderate evidence only (same gate as default); only branch-resolved '
                          'options with a certain location are ranked. Here none qualifies.')}
        if mode == 'price':
            out.update(price_comparison={'unit': None, 'budget_km': None, 'comparable_quotes': 0, 'ranked_by': None,
                                         'cheapest_claim': False, 'note': reason},
                       price_unknown_nearby=[], price_not_comparable=[])
        return out
    nearest_km = ranked[0]['distance_km']
    budget = round(nearest_km + max_extra_km, 1)
    out = {'mode': mode, 'max_extra_km': max_extra_km, 'nearest_credible_km': nearest_km, 'ranked_count': len(ranked)}
    safety_note = None

    if mode == 'nearest':
        order = list(ranked)
        if service in EMERGENCY_SERVICES:
            # Safety tie-break: a record that also says the service is NOT available never goes first while a clean
            # option exists within max_extra_km of it.
            clean = [c for c in order if 'contradictory_claims' not in c['flags']]
            if clean and 'contradictory_claims' in order[0]['flags'] and clean[0]['distance_km'] <= order[0]['distance_km'] + max_extra_km:
                order.remove(clean[0])
                order.insert(0, clean[0])
                clean[0]['safety_reordered'] = True
            safety_note = ('Nearest by straight line. In an emergency, call ahead with the questions on each card, or use the '
                           '108 ambulance: a nearer facility that cannot confirm the service is not the safer choice.')
        results = order[:k]
        last = results[-1]['distance_km']
        runner_ups = [dict(c, why_not=f"{round(c['distance_km'] - last, 1)} km farther than #{len(results)}") for c in order[k:k + MAX_RUNNER_UPS]]
        near = lambda c: c['distance_km'] is not None and c.get('distance_range_km', [c['distance_km']])[0] <= last + max_extra_km
        for rank, c in enumerate(results, 1):
            c['rank'] = rank
            c['why_ranked'] = (f"#{rank} by straight-line distance among {len(ranked)} credible, branch-resolved options"
                               + (' (moved up: the nearer record also says the service is not available)' if c.get('safety_reordered') else '')
                               + '.')
        strong_nearby = next((c for c in ranked if c['tier'] == 'strong' and c['distance_km'] <= results[0]['distance_km'] + max_extra_km), None)
        if service in EMERGENCY_SERVICES and results[0]['tier'] != 'strong' and strong_nearby and strong_nearby is not results[0]:
            out['stronger_option_nearby'] = {'facility_id': strong_nearby['facility_id'], 'name': strong_nearby['name'],
                                             'distance_km': strong_nearby['distance_km'],
                                             'note': 'Strong evidence, within max_extra_km of the nearest option'}
        out.update(method=('Nearest mode: Strong/Moderate evidence only (same gate as default); credible branch-resolved '
                           'options ordered by conservative straight-line distance (farthest plausible position when a '
                           'location is uncertain). Unresolved branches and uncertain locations are listed, not ranked.'),
                   branch_unresolved=[c for c in unresolved if near(c)][:3],
                   location_uncertain=[c for c in uncertain if near(c)][:3])
    else:
        in_budget = [c for c in ranked if c['distance_km'] <= budget]
        # Like-for-like: same service, same stated unit AND the same conditions (eligibility, scheme, inclusions, ...).
        # A concessional BPL rate and an ordinary rate share a unit but not who can get them, so they are never ranked
        # against each other.
        groups = {}
        for c in in_budget:
            p = c['price_info']
            if p and p['bound'] != 'free' and p['unit'] != 'unspecified':
                groups.setdefault(comparison_key(p), []).append(c)
        key = max(groups, key=lambda g: (len(groups[g]), -groups[g][0]['distance_km'])) if groups else None
        comparable = [c for c in groups.get(key, [])
                      if all(price_comparable(c['price_info'], d['price_info']) for d in groups.get(key, []))]
        unit, conditions = key[:2] if key else (None, ())
        pinned = None
        if service in EMERGENCY_SERVICES:
            # Urgent service: the nearest safe option (certain location, no "not available" statement) stays first;
            # price only orders the options after it.
            pinned = next((c for c in in_budget if 'contradictory_claims' not in c['flags']), None)
            safety_note = ('Urgent service: the nearest safe option is kept first; price only orders the options after it. '
                           'In an emergency, call ahead with the questions on each card, or use the 108 ambulance.')
        inf = float('inf')
        price_key = lambda c: (c['price_info']['amount_max_inr'] if c['price_info']['amount_max_inr'] is not None else inf,
                               c['price_info']['amount_min_inr'] or 0, c['distance_km'])
        cheapest = False
        if len(comparable) >= 2:
            comparable.sort(key=price_key)
            top = comparable[0]
            cheapest = all(surely_cheaper(top['price_info'], c['price_info']) for c in comparable[1:])
            ordered = comparable
            ranked_by = 'price'
            cond_text = f" under the same conditions ({'; '.join(conditions)})" if conditions else ''
            if cheapest:
                note = (f"{top['name']} documents a lower {PRICE_UNIT_TEXT[unit]} price than every other like-for-like quote "
                        f"within {budget} km{cond_text}.")
            else:
                note = (f"Ordered by documented {PRICE_UNIT_TEXT[unit]} price (upper amount first) among {len(comparable)} "
                        f"like-for-like quotes{cond_text}. No option is called the cheapest: the documented ranges overlap.")
        else:
            ordered = in_budget
            ranked_by = 'distance'
            note = (f"{len(comparable)} like-for-like documented price(s) within {budget} km (same service, unit and "
                    'conditions): prices cannot be ranked, so options are shown nearest first with every documented price '
                    'labelled on its card.')
        if pinned is not None:
            ordered = [pinned] + [c for c in ordered if c is not pinned]
            pinned['safety_pinned'] = True
        results = ordered[:k]
        shown = {c['facility_id'] for c in results}
        for rank, c in enumerate(results, 1):
            c['rank'] = rank
            if c.get('safety_pinned'):
                c['why_ranked'] = (f"#{rank}: nearest safe option for an urgent service ({c['distance_km']} km), kept first "
                                   f"whatever its price ({_price_text(c['price_info'])}).")
            elif ranked_by == 'price':
                c['why_ranked'] = f"#{rank}: {_price_text(c['price_info'])}, {c['distance_km']} km"
            else:
                c['why_ranked'] = f"#{rank} by distance within {budget} km ({_price_text(c['price_info'])})."
        last = results[-1]
        runner_ups = [dict(c, why_not=(f"documented price {_price_text(c['price_info'])} is not lower" if ranked_by == 'price'
                                       else f"{round(c['distance_km'] - last['distance_km'], 1)} km farther"))
                      for c in ordered[k:]][:MAX_RUNNER_UPS]

        def why_not_comparable(c):
            p = c['price_info']
            if p['bound'] == 'free':
                return 'free only with conditions: ' + ('; '.join(p['conditions']) or 'none stated')
            if p['unit'] == 'unspecified':
                return 'unit not stated'
            if p['unit'] != unit:
                return f"different unit ({PRICE_UNIT_TEXT[p['unit']]})"
            return ('different eligibility or conditions: ' + ('; '.join(p['conditions']) or 'no conditions stated')
                    + ' vs ' + ('; '.join(conditions) or 'no conditions stated'))

        within = lambda c: c['distance_km'] is not None and c.get('distance_range_km', [c['distance_km']])[0] <= budget
        out.update(
            method=(f'Price mode: Strong/Moderate evidence only (same gate as default); among credible branch-resolved options '
                    f'within {max_extra_km} km of the nearest one ({budget} km), documented patient prices for this service are '
                    'compared only when they state the same unit and the same conditions (eligibility, scheme, inclusions); '
                    'upper/lower bounds are kept, never filled in. Urgent services keep the nearest safe option first. '
                    'Options without a comparable price stay visible.'),
            price_comparison={'unit': unit, 'conditions': list(conditions), 'budget_km': budget,
                              'comparable_quotes': len(comparable), 'ranked_by': ranked_by, 'cheapest_claim': cheapest,
                              'safety_pinned': pinned['facility_id'] if pinned is not None else None, 'note': note},
            price_unknown_nearby=[c for c in in_budget if not c['price_info'] and c['facility_id'] not in shown][:k],
            price_not_comparable=[dict(c, not_comparable_because=why_not_comparable(c))
                                  for c in in_budget if c['price_info'] and c not in comparable
                                  and c['facility_id'] not in shown][:k],
            branch_unresolved=[c for c in unresolved if within(c)][:3],
            location_uncertain=[c for c in uncertain if within(c)][:3])
        for c in results + runner_ups:      # label every shown quote: compared like-for-like, or why not
            if c['price_info']:
                c['price_comparable'] = c in comparable or any(c['facility_id'] == x['facility_id'] for x in comparable)
                if not c['price_comparable']:
                    c['not_comparable_because'] = why_not_comparable(c)
    for c in runner_ups + out.get('branch_unresolved', []) + out.get('location_uncertain', []) + \
            out.get('price_unknown_nearby', []) + out.get('price_not_comparable', []):
        c.pop('rank', None)
    far = max(c['distance_km'] for c in results)
    out['band_km'] = next((b for b in bands if far <= b), bands[-1])
    out['safety_note'] = safety_note
    out['results'], out['runner_ups'] = results, runner_ups
    return out


def _apply_variant(a, strong_min_claims):
    """Sensitivity variant: require more distinct claims for Strong."""
    if strong_min_claims and a['tier'] == 'strong':
        n = len({normalize(e['text']) for e in a['evidence'] if e['kind'] == 'direct'})
        if n < strong_min_claims:
            return {**a, 'tier': 'moderate', 'reasons': a['reasons'] + [f'fewer than {strong_min_claims} distinct claims']}
    return a


def recommend(index, service_text, location_text, k=MAX_RESULTS, bands=BANDS_KM, strong_min_claims=None, review=True,
              preferences=None, sort_mode='default', max_extra_km=25):
    """preferences: optional {'urgency': 'default'|'time_critical'|'can_travel', 'cost': 'any'|'low_cost'}.
    sort_mode: 'default' (evidence, then distance), 'nearest' (straight-line distance among credible, branch-resolved
    options) or 'price' (documented comparable service prices within max_extra_km of the nearest credible option).
    Without preferences and with sort_mode='default' the output is exactly the evidence-then-distance ranking."""
    prefs = normalize_preferences(preferences)
    if sort_mode not in SORT_MODES:
        raise ValueError(f'sort_mode must be one of {SORT_MODES}')
    if sort_mode != 'default' and prefs['applied']:
        raise ValueError('use either preferences or sort_mode, not both')
    if isinstance(max_extra_km, bool) or not isinstance(max_extra_km, (int, float)) or not 0 <= max_extra_km <= 300:
        raise ValueError('max_extra_km must be a number between 0 and 300')
    service = resolve_service(service_text)
    if not service:
        return {'error': f'Unknown service "{service_text}". Choose one of: '
                         + ', '.join(s['label'] for s in SERVICES.values())}
    origin = index.gaz.resolve(location_text)
    if 'error' in origin:
        return {'error': origin['error']}
    o = (origin['lat'], origin['lon'])

    scored = []          # (tier_rank, distance, i, assessment)
    no_distance = 0
    for i, a in enumerate(index.assessments(service)):
        if a['tier'] == 'none':
            continue
        a = _apply_variant(a, strong_min_claims)
        dist = _distance(o, index.locations[i])
        if dist is None:
            no_distance += a['tier'] in ('strong', 'moderate')
            continue
        if a['tier'] == 'strong' and index.locations[i]['confidence'] == 'low':
            a = {**a, 'tier': 'moderate', 'reasons': a['reasons'] + ['location data conflicts, so treated as moderate'],
                 'flags': sorted(set(a['flags']) | {'location_conflict'})}
        if a['tier'] == 'strong' and a.get('merge_reasons') and all(
                foreign_mentions([e], index.rows[i], index.city_regex, city_key)
                for e in a['evidence'] if e['kind'] == 'direct'):
            a = {**a, 'tier': 'moderate', 'reasons': a['reasons'] + ['every claim may describe another facility']}
        scored.append((TIERS.index(a['tier']), dist, i, a))

    eligible = [s for s in scored if s[0] <= 1]
    order = lambda s: (s[0], round(s[1], 1), -len(s[3]['evidence']), text(index.rows[s[2]].get('unique_id')))
    counts = {b: {t: sum(1 for s in scored if s[1] <= b and TIERS[s[0]] == t) for t in ('strong', 'moderate', 'weak')}
              for b in bands}

    # An approximate location (a PIN known only to its area, +-25 km, or a whole district) cannot support a
    # 25 km answer: start at the first band at least twice the location's uncertainty. City names are exempt:
    # their "precision" is the city's size, and a city user is somewhere inside it.
    coarse = origin['kind'] in ('pin', 'pin_area', 'district')
    start = [b for b in bands if not coarse or b >= 2 * origin['precision_km']] or [bands[-1]]
    if start[0] != bands[0]:
        origin = {**origin, 'notes': origin['notes'] + [
            f"Your location is only known to about ±{origin['precision_km']:g} km, so the search starts at {start[0]} km."]}

    chosen_band, results, extra, runner_ups = None, [], None, []
    found = []          # recommendable cards, nearest band first; each remembers the band it was found in
    for b in start:
        in_band = sorted([s for s in scored if s[1] <= b], key=order)
        if not any(s[0] <= 1 for s in in_band):
            continue
        chosen_band = b
        cards = _dedupe([_card(index, s[2], s[3], s[1]) for s in in_band if s[0] <= 1])
        known = {c['facility_id'] for c in found} | {m for c in found for m in c.get('merged_ids', [])}
        for c in cards:
            if c['facility_id'] not in known and not set(c.get('merged_ids', [])) & known:
                c['found_within_km'] = b
                found.append(c)
        found = _dedupe(found)
        if coarse and len(found) < k and b != bands[-1]:
            continue   # approximate location: widen until there are k options to compare (or the last band)
        if coarse and b != start[0] and len({c['found_within_km'] for c in found[:k]}) > 1:
            origin = {**origin, 'notes': origin['notes'] + [
                f'Fewer than {k} credible options nearby; widened the search to {b} km because the location is approximate. '
                'Nearer options are listed first.']}
        results = found[:k]
        weak = _dedupe([_card(index, s[2], s[3], s[1]) for s in in_band if s[0] == 2])
        shown = {_name_key(c['name']) for c in results}
        pool = found[k:] + [w for w in weak if _name_key(w['name']) not in shown]
        for c in pool[:MAX_RUNNER_UPS]:
            c['why_not'] = _why_not(c, results[-1])
            runner_ups.append(c)
        if all(c['tier'] != 'strong' for c in results):
            nxt = bands[bands.index(b) + 1] if b != bands[-1] else None
            stronger = sorted([s for s in eligible if s[0] == 0 and s[1] > b and (nxt is None or s[1] <= nxt)],
                              key=order)
            if stronger:
                extra = _card(index, stronger[0][2], stronger[0][3], stronger[0][1])
                extra['note'] = 'Farther away, but stronger evidence than the options above.'
        break

    beyond, weak_nearby = None, []
    if not results:
        far = sorted(eligible, key=lambda s: (round(s[1], 1), s[0]))
        if far:
            beyond = _card(index, far[0][2], far[0][3], far[0][1])
            beyond['note'] = f'Nearest facility with credible evidence, beyond {bands[-1]} km.'
        weak = sorted([s for s in scored if s[0] == 2 and s[1] <= bands[1]], key=order)[:3]
        weak_nearby = [_card(index, s[2], s[3], s[1]) for s in weak]

    n_in_band = len([s for s in eligible if chosen_band and s[1] <= chosen_band])
    for rank, c in enumerate(results, 1):
        c['rank'] = rank
        if c.get('found_within_km', chosen_band) != chosen_band:
            c['why_ranked'] = (f"#{rank}: found within {c['found_within_km']} km, the nearest search radius with credible "
                               f"evidence; listed before options found only after widening to {chosen_band} km.")
        else:
            c['why_ranked'] = (f"#{rank} of {n_in_band} facilities with credible evidence within {chosen_band} km; "
                               f"ordered by evidence strength, then distance.")

    default_ids = [c['facility_id'] for c in results]
    cost_unknown, tradeoff = [], None
    if prefs['applied'] and results:
        pool = list(found)
        if prefs['urgency'] == 'can_travel' and chosen_band != bands[-1]:
            nxt = bands[bands.index(chosen_band) + 1]
            nearest = min(c['distance_km'] for c in pool if c['distance_km'] is not None)
            cap = min(nxt, max(2 * nearest, nearest + CAN_TRAVEL_EXTRA_KM))   # further, but within reason
            known = {c['facility_id'] for c in pool} | {m for c in pool for m in c.get('merged_ids', [])}
            for s in sorted([s for s in eligible if chosen_band < s[1] <= cap], key=order):
                c = _card(index, s[2], s[3], s[1])
                if c['facility_id'] not in known and not set(c.get('merged_ids', [])) & known:
                    c['found_within_km'] = nxt
                    pool.append(c)
            pool = _dedupe(pool)
        default_first = next((c for c in found if c['facility_id'] == default_ids[0]), None) if default_ids else None
        results, runner_ups, cost_unknown = _preference_rank(index, service, prefs, pool, k, origin)
        if extra and extra['facility_id'] in {c['facility_id'] for c in results + cost_unknown}:
            extra = None
        tradeoff = _tradeoff_note(results[0], default_first) if results and default_first else None

    mode = None
    if sort_mode != 'default' and results:
        mode = _sort_mode(index, service, sort_mode, eligible, k, max_extra_km, bands, o)
        results, runner_ups, extra = mode.pop('results'), mode.pop('runner_ups'), None
        chosen_band, n_in_band = mode['band_km'], mode['ranked_count']

    first_ask = SERVICES[service]['ask'][0]
    if mode and not results and mode.get('no_ranked_reason'):
        advice = mode['no_ranked_reason']
    elif not results:
        advice = ('No facility with credible evidence within reach. Call the nearest option shown before travelling, '
                  'and check with the district hospital or 108 ambulance service.')
    elif n_in_band <= 2 or chosen_band >= 75:
        advice = (f'Few alternatives within reach ({n_in_band} within {chosen_band} km): call the first option before '
                  f'setting off and ask: "{first_ask}" Keep the second option ready in case it cannot confirm.')
    else:
        advice = (f'{n_in_band} facilities with credible evidence within {chosen_band} km. Call the first option; '
                  f'if it cannot confirm, move to the next one.')

    result = {
        'contract_version': CONTRACT_VERSION,
        'service': service, 'service_label': SERVICES[service]['label'], 'service_scope': SERVICES[service]['scope'],
        'origin': origin, 'band_km': chosen_band, 'action_advice': advice, 'results': results, 'runner_ups': runner_ups,
        'farther_stronger': extra, 'nearest_beyond': beyond, 'weak_nearby': weak_nearby,
        'considered': {'by_band': counts, 'recommendable_without_location': no_distance},
        'parameters': {'bands_km': list(bands), 'max_results': k, 'strong_min_claims': strong_min_claims},
        'preferences': prefs, 'sort_mode': sort_mode,
        **({'sort': mode} if mode else {}),
        **({'preference_model': decision_assumptions(prefs), 'default_order_ids': default_ids,
            'cost_unknown_options': cost_unknown, 'tradeoff_note': tradeoff} if prefs['applied'] else {}),
        'method': (mode['method'] if mode else
                   'Strong/Moderate evidence only; nearest distance band (' + '/'.join(map(str, bands)) + ' km) that has '
                   'any; within it, stronger evidence first, then shorter straight-line distance.'),
        'disclaimer': ('Based on public web listings, not verified by the facility. Availability today is unknown: '
                       'call before travelling. Distances are straight-line, not road travel time.'),
    }
    if review:
        from .review import review as run_review
        result['review'] = run_review(index, result)
    return result
''')

**`carematch/strategy_compare.py`** · The first choice for each ranking strategy and its source evidence.

In [10]:
load_embedded('strategy_compare', r'''"""Explain how three ranking rules change the first option for exactly the same search."""

from .recommend import recommend


ORDERS = {'default': 'Best evidence', 'nearest': 'Nearest', 'price': 'Compare listed prices'}


def _first_option(mode, result):
    cards = result.get('results', [])
    sort = result.get('sort') or {}
    prices = sort.get('price_comparison') or {}
    option = dict(mode=mode, label=ORDERS[mode], facility_id=None, name=None, tier=None,
                  distance_km=None, citation=None, price_citation=None, flags=[], price_basis=prices.get('ranked_by'),
                  comparable_quotes=prices.get('comparable_quotes'), safety_pinned=bool(prices.get('safety_pinned')), reason=sort.get('no_ranked_reason') or '')
    if result.get('error') or (result.get('review') or {}).get('passed') is False:
        option['reason'] = 'Comparison unavailable for this order.'
        return option
    if not cards:
        return option
    card = cards[0]
    option.update({key: card[key] for key in ('facility_id', 'name', 'tier', 'distance_km', 'flags')})
    evidence = next(iter(card.get('evidence', [])), None)
    if evidence:
        option['citation'] = dict(field=evidence['field'], quote=evidence.get('verbatim') or evidence['quote'],
                                  facility_id=card['facility_id'], source_urls=card.get('source_urls', []))
    if mode == 'price':
        option['reason'] = prices.get('note') or option['reason']
        price = card.get('price_info')
        if price:
            option['price_citation'] = {key: price[key] for key in ('field', 'quote')}
    return option


def compare_strategies(index, service, place, active_result, max_extra_km=25):
    """Return the first choice for each standard order. Preferences on the active card stay separate."""
    active_mode = active_result.get('sort_mode', 'default')
    standard = not (active_result.get('preferences') or {}).get('applied')
    options = []
    for mode in ORDERS:
        if mode == active_mode and standard:
            result = active_result
        else:
            result = recommend(index, service, place, sort_mode=mode, **({'max_extra_km': max_extra_km} if mode == 'price' else {}))
        options.append(_first_option(mode, result))

    evidence, nearest, price = options
    changed = bool(evidence['facility_id'] and nearest['facility_id'] and evidence['facility_id'] != nearest['facility_id'])
    difference = None
    if changed:
        difference = round(evidence['distance_km'] - nearest['distance_km'], 1)
        if difference:
            relation = 'closer to' if difference > 0 else 'farther from'
            movement = f"{abs(difference):g} km {relation} the search centre"
        else:
            movement = 'at the same displayed distance'
        summary = (f"{nearest['name']} is {movement} by straight line. "
                   f"The records give {nearest['tier'].title()} evidence there and {evidence['tier'].title()} evidence at {evidence['name']}.")
    elif evidence['facility_id'] and nearest['facility_id']:
        summary = f"{evidence['name']} comes first for both evidence and distance."
    elif any(o['facility_id'] for o in options):
        summary = 'The orders differ on whether a branch can be ranked. Check the branch and location notes.'
    else:
        summary = 'None of these orders returned a first choice for this search.'
    if price['facility_id']:
        if price['safety_pinned']:
            summary += ' The urgent-care rule keeps the closest eligible option without a conflicting service claim first when available.'
        elif price['price_basis'] == 'distance':
            summary += ' Price comparison falls back to distance.'
        elif price['price_basis'] == 'price' and price['facility_id'] != nearest['facility_id']:
            summary += f" Comparing documented prices puts {price['name']} first."
        elif price['price_basis'] not in ('price', 'distance') and price['reason']:
            summary += ' ' + price['reason']
    origin = active_result.get('origin') or {}
    precision = origin.get('precision_km')
    if difference and precision and origin.get('kind') != 'coordinates' and abs(difference) <= 2 * precision:
        summary += f" The starting point is approximate (about {precision:g} km). Your closest option may differ."
    return dict(summary=summary, options=options, top_changed=changed, straight_line_difference_km=difference)
''')

**`carematch/render.py`** · Plain-text rendering of a result (used below and in the CLI).

In [11]:
load_embedded('render', r'''"""Plain-text rendering of a recommendation result (CLI and notebook)."""

TIER_WORD = {'strong': 'STRONG evidence', 'moderate': 'MODERATE evidence', 'weak': 'WEAK evidence'}


def _card_text(c, header):
    lines = [header]
    dist = f"{c['distance_km']} km" if c['distance_km'] is not None else 'distance unknown'
    meta = ', '.join(x for x in (c['type'], c['operator'], f"{c['beds']} beds" if c['beds'] else None) if x)
    lines.append(f"   {dist} · {TIER_WORD.get(c['tier'], c['tier'])} · {meta}")
    lines.append(f"   {c['address'] or c['city']}{', PIN ' + c['pin'] if c['pin'] else ''}")
    lines.append(f"   Why: {'; '.join(c['tier_reasons'])}")
    for e in c['evidence']:
        lines.append(f"     “{e['quote'][:160]}”  [{e['field']}]")
    if c.get('why_ranked'):
        lines.append(f"   Order: {c['why_ranked']}")
    for u in c['uncertainties']:
        lines.append(f"   ! {u}")
    if c['distance_basis'] and c['distance_basis'] != 'facility coordinates':
        lines.append(f"   ! distance measured from {c['distance_basis']}")
    lines.append(f"   Call: {c['phone'] or 'no phone listed'}" + (f" · {c['website']}" if c['website'] else ''))
    lines.append(f"   Ask: {c['ask_when_calling'][0]}")
    if c.get('note'):
        lines.append(f"   Note: {c['note']}")
    return '\n'.join(lines)


def to_text(result):
    if 'error' in result:
        return 'Error: ' + result['error']
    o = result['origin']
    out = [f"{result['service_label']} near {o['label']}  (location precision ~{o['precision_km']} km)"]
    out += [f"  note: {n}" for n in o['notes']]
    if result.get('service_scope'):
        out.append(f"  what the evidence can show: {result['service_scope']}")
    if result.get('action_advice'):
        out.append(f"What to do: {result['action_advice']}")
    if result['results']:
        out.append(f"Searched up to {result['band_km']} km.")
        for c in result['results']:
            out.append(_card_text(c, f"\n{c['rank']}. {c['name']}"))
    else:
        out.append(f"\nNo facility with credible evidence within {300} km.")
    if result['farther_stronger']:
        out.append(_card_text(result['farther_stronger'], f"\n+ {result['farther_stronger']['name']}"))
    if result['nearest_beyond']:
        out.append(_card_text(result['nearest_beyond'], f"\n> {result['nearest_beyond']['name']}"))
    for c in result['weak_nearby']:
        out.append(_card_text(c, f"\n? (weak, unverified) {c['name']}"))
    b = result['considered']['by_band']
    out.append('\nConsidered: ' + '; '.join(f"≤{k} km: {v['strong']} strong / {v['moderate']} moderate / {v['weak']} weak"
                                            for k, v in b.items()))
    out.append(result['disclaimer'])
    return '\n'.join(out)
''')

**`carematch/datasource.py`** · Loads the three tables from Spark (Databricks), a SQL warehouse (the app), a Volume snapshot or local CSVs, converting every value to the same text so results are identical everywhere.

In [12]:
load_embedded('datasource', r'''"""Load the source tables from wherever they are, as rows the engine understands.

Sources, in the order `load()` tries them:
  spark      - a Databricks notebook (`spark` passed in)
  warehouse  - a Databricks App: SQL warehouse attached as an app resource (env DATABRICKS_WAREHOUSE_ID)
  volume     - snapshot CSVs written by notebooks/00_export_data.py to /Volumes/workspace/carematch/exports
  local      - data/*.csv on a laptop (or CAREMATCH_DATA_DIR)

The engine was calibrated on the CSV export, so every value is converted to the text that export contains:
None/NaN -> '', numbers -> str(number). Rows are dicts keyed by the source column names.
"""
import csv
import io
import math
import os
import sys
import time

SOURCE_SCHEMA = 'databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset'
FACILITIES_SQL = f'SELECT * FROM {SOURCE_SCHEMA}.facilities'
# One row per PIN x district x state; identical to notebooks/00_export_data.py.
PINCODES_SQL = f"""
SELECT pincode, TRIM(district) AS district, TRIM(statename) AS statename,
       COUNT(*) AS n_offices,
       COUNT(TRY_CAST(latitude AS DOUBLE)) AS n_geocoded,
       AVG(TRY_CAST(latitude AS DOUBLE))  AS lat,
       AVG(TRY_CAST(longitude AS DOUBLE)) AS lon,
       MIN(TRY_CAST(latitude AS DOUBLE))  AS lat_min, MAX(TRY_CAST(latitude AS DOUBLE))  AS lat_max,
       MIN(TRY_CAST(longitude AS DOUBLE)) AS lon_min, MAX(TRY_CAST(longitude AS DOUBLE)) AS lon_max
FROM {SOURCE_SCHEMA}.india_post_pincode_directory
GROUP BY pincode, TRIM(district), TRIM(statename)
"""
EXPORT_DIR = '/Volumes/workspace/carematch/exports'
FILES = ('facilities.csv', 'pincodes.csv')
# The SDK retries failed requests for 5 minutes by default, which made a missing permission look like a hung app.
# Bounded retries surface the real error quickly; the warehouse gets longer because a stopped one must start first.
SDK_RETRY_SECONDS = 30
WAREHOUSE_RETRY_SECONDS = 180


def cell(value):
    """A source value as the CSV export writes it."""
    if value is None:
        return ''
    if isinstance(value, float):
        return '' if math.isnan(value) else str(value)
    return str(value)


def to_rows(columns, records):
    return [{c: cell(v) for c, v in zip(columns, rec)} for rec in records]


def _read_csv(text_or_file):
    csv.field_size_limit(sys.maxsize)
    fh = io.StringIO(text_or_file) if isinstance(text_or_file, str) else text_or_file
    return list(csv.DictReader(fh))


# ---- individual sources ----------------------------------------------------------------------------------------

def load_local(directory=None):
    directory = directory or os.environ.get('CAREMATCH_DATA_DIR') or os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))), 'data')
    out = []
    for name in FILES:
        with open(os.path.join(directory, name), newline='', encoding='utf-8') as fh:
            out.append(_read_csv(fh))
    return out[0], out[1]


def load_spark(spark):
    def query(sql_text):
        df = spark.sql(sql_text)
        return to_rows(df.columns, df.collect())
    return query(FACILITIES_SQL), query(PINCODES_SQL)


def load_warehouse(warehouse_id=None):
    """Read through a SQL warehouse with the app's own credentials (Databricks Apps inject them)."""
    from databricks import sql
    from databricks.sdk.core import Config

    warehouse_id = warehouse_id or os.environ.get('DATABRICKS_WAREHOUSE_ID')
    if not warehouse_id:
        raise RuntimeError('DATABRICKS_WAREHOUSE_ID is not set (attach a SQL warehouse resource to the app)')
    cfg = Config(retry_timeout_seconds=SDK_RETRY_SECONDS)
    host = cfg.host.replace('https://', '').rstrip('/')
    with sql.connect(server_hostname=host, http_path=f'/sql/1.0/warehouses/{warehouse_id}',
                     credentials_provider=lambda: cfg.authenticate,
                     _retry_stop_after_attempts_duration=WAREHOUSE_RETRY_SECONDS,
                     _socket_timeout=WAREHOUSE_RETRY_SECONDS) as conn:
        def query(sql_text):
            with conn.cursor() as cur:
                cur.execute(sql_text)
                return to_rows([d[0] for d in cur.description], cur.fetchall())
        return query(FACILITIES_SQL), query(PINCODES_SQL)


def load_volume(directory=EXPORT_DIR):
    """Snapshot CSVs in a Unity Catalog Volume: a plain path in notebooks, the Files API in apps."""
    if os.path.isdir(directory):
        return load_local(directory)
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.core import Config
    w = WorkspaceClient(config=Config(retry_timeout_seconds=SDK_RETRY_SECONDS))
    out = []
    for name in FILES:
        data = w.files.download(f'{directory}/{name}').contents.read()
        out.append(_read_csv(data.decode('utf-8')))
    return out[0], out[1]


NFHS_SQL = f'SELECT * FROM {SOURCE_SCHEMA}.nfhs_5_district_health_indicators'


def load_nfhs(spark=None, directory=None):
    """NFHS-5 district indicators (planner view only; never used to rank facilities for a patient)."""
    if spark is not None:
        df = spark.sql(NFHS_SQL)
        return to_rows(df.columns, df.collect())
    directory = directory or os.environ.get('CAREMATCH_DATA_DIR') or os.path.join(
        os.path.dirname(os.path.dirname(os.path.abspath(__file__))), 'data')
    with open(os.path.join(directory, 'nfhs5.csv'), newline='', encoding='utf-8') as fh:
        return _read_csv(fh)


# ---- entry point -----------------------------------------------------------------------------------------------

def load(spark=None, order=None):
    """Return (facility_rows, pincode_rows, info). Tries sources in order; info says which one worked and why others failed."""
    if order is None:
        if spark is not None:
            order = ['spark', 'volume']
        elif os.environ.get('DATABRICKS_APP_NAME') or os.environ.get('DATABRICKS_WAREHOUSE_ID'):
            order = ['warehouse', 'volume']
        else:
            order = ['local']
    loaders = {'spark': lambda: load_spark(spark), 'warehouse': load_warehouse, 'volume': load_volume, 'local': load_local}
    errors = []
    for name in order:
        started = time.perf_counter()
        try:
            facilities, pincodes = loaders[name]()
        except Exception as exc:  # report every failed source, then try the next one
            errors.append(f'{name}: {type(exc).__name__}: {exc}')
            continue
        info = {'source': name, 'facility_rows': len(facilities), 'pincode_rows': len(pincodes),
                'load_seconds': round(time.perf_counter() - started, 1), 'failed_sources': errors}
        return facilities, pincodes, info
    raise RuntimeError('Could not load the source tables. ' + ' | '.join(errors))
''')

**`carematch/planner.py`** · District access gaps for planners: distance from every PIN area to the nearest credible facility, joined with NFHS-5 need. Used only in Part 6; NFHS never affects a patient's ranking.

In [13]:
load_embedded('planner', r'''"""Access gaps for planners: how far each district's people are from a facility with credible evidence.

Unit of analysis: India Post PIN areas with a reliable centre (spread <= 60 km), weighted by their number of post
offices as a rough stand-in for settlements. For each area and service we take the straight-line distance to the
nearest facility whose evidence is Strong or Moderate (same rules as recommend()), then summarise by district and
put NFHS-5 district indicators next to it.

NFHS figures describe need in a district. They never enter recommend(): a district's need says nothing about the
quality of any one facility.
"""
import csv
import difflib
import math
import re
import statistics
import sys

from .geo import PIN_SPREAD_OK, canonical_state
from .textutil import haversine_km, normalize, text

NEAR_KM = 50                 # an area farther than this from the nearest credible facility counts as "far"
FAR_SHARE = 0.5              # a district is an access gap when at least half its post offices are far
FEW_HOSPITALS = 3            # fewer hospital listings (any service) than this near a district: likely a data gap
LOW_CSECTION_PCT = 5.0       # UN EmOC monitoring guidelines: a population caesarean rate below 5% signals unmet need
KM_PER_DEG = 111.2
MIN_KM_PER_DEG_LON = 88.0    # 111.2 * cos(37.5 deg), India's northern edge: a safe lower bound for longitude

NFHS_FIELDS = {
    'csection_pct': 'births_delivered_by_csection_5y_pct',
    'institutional_birth_pct': 'institutional_birth_5y_pct',
    'high_sugar_women_pct': 'w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct',
    'high_sugar_men_pct': 'm15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct',
    'insured_pct': 'hh_member_covered_health_insurance_pct',
}


class PointIndex:
    """Nearest-neighbour and radius counts on a lat/lon grid (standard library only)."""

    def __init__(self, points, cell_deg=0.5):
        self.cell = cell_deg
        self.grid = {}
        for p in points:          # p = (lat, lon, key)
            self.grid.setdefault(self._key(p[0], p[1]), []).append(p)

    def _key(self, lat, lon):
        return int(math.floor(lat / self.cell)), int(math.floor(lon / self.cell))

    def _ring(self, centre, r):
        ci, cj = centre
        for i in range(ci - r, ci + r + 1):
            for j in range(cj - r, cj + r + 1):
                if max(abs(i - ci), abs(j - cj)) == r:
                    yield from self.grid.get((i, j), ())

    def nearest(self, lat, lon, max_km=3500):
        if not self.grid:
            return None, None
        centre, best, best_key = self._key(lat, lon), None, None
        step_km = self.cell * MIN_KM_PER_DEG_LON
        for r in range(int(max_km / step_km) + 2):
            if best is not None and (r - 1) * step_km > best:
                break
            for p in self._ring(centre, r):
                d = haversine_km((lat, lon), (p[0], p[1]))
                if best is None or d < best:
                    best, best_key = d, p[2]
        return best, best_key

    def keys_within(self, lat, lon, km):
        centre = self._key(lat, lon)
        out = set()
        for r in range(int(km / (self.cell * MIN_KM_PER_DEG_LON)) + 2):
            for p in self._ring(centre, r):
                if haversine_km((lat, lon), (p[0], p[1])) <= km:
                    out.add(p[2])
        return out


# ---- inputs from the engine ----------------------------------------------------------------------------------------

def credible_points(index, service):
    """(lat, lon, facility group) for every positioned facility with Strong or Moderate evidence for the service."""
    pts = []
    for i, a in enumerate(index.assessments(service)):
        pos = index.locations[i]['position']
        if a['tier'] in ('strong', 'moderate') and pos:
            pts.append((pos[0], pos[1], index.dup_group[i]))
    return pts


def hospital_points(index):
    """Every positioned hospital listing, whatever it claims: how well the dataset covers an area at all."""
    return [(index.locations[i]['position'][0], index.locations[i]['position'][1], index.dup_group[i])
            for i, r in enumerate(index.rows)
            if index.locations[i]['position'] and normalize(r.get('facilityTypeId')) == 'hospital']


def districts(index):
    """{(district, state): [(lat, lon, offices), ...]} from PIN areas with a reliable centre."""
    out = {}
    for entries in index.gaz.pins.values():
        for e in entries:
            if e['centre'] and e['spread_km'] is not None and e['spread_km'] <= PIN_SPREAD_OK and e['district'] and e['state']:
                out.setdefault((e['district'], e['state']), []).append((e['centre'][0], e['centre'][1], max(1, e['n_offices'])))
    return out


def _weighted_median(pairs):
    pairs = sorted(pairs)
    total, run = sum(w for _, w in pairs), 0
    for v, w in pairs:
        run += w
        if run >= total / 2:
            return v
    return None


# ---- NFHS -------------------------------------------------------------------------------------------------------------

def nfhs_number(value):
    """NFHS cell -> (value, small_sample). '*' = suppressed; '(12.3)' = based on 25-49 cases."""
    s = text(value).strip()
    if not s or s == '*':
        return None, False
    small = s.startswith('(')
    try:
        return float(s.strip('()').strip()), small
    except ValueError:
        return None, False


def _dkey(name):
    return re.sub(r'[^a-z0-9]+', ' ', normalize(name).replace('&', ' and ')).strip()


def _skey(state):
    return canonical_state(state) or _dkey(state)


def match_nfhs(district_keys, nfhs_rows):
    """Map (district, state) -> NFHS row. Exact name within state, then a close spelling within state."""
    by_state = {}
    for row in nfhs_rows:
        by_state.setdefault(_skey(row.get('state_ut')), {})[_dkey(row.get('district_name'))] = row
    mapping, how = {}, {'exact': 0, 'close_spelling': 0, 'unmatched': 0}
    for district, state in district_keys:
        options = by_state.get(_skey(state)) or {}
        k = _dkey(district)
        if k in options:
            mapping[(district, state)] = options[k]
            how['exact'] += 1
            continue
        close = difflib.get_close_matches(k, list(options), n=1, cutoff=0.85)
        if close:
            mapping[(district, state)] = options[close[0]]
            how['close_spelling'] += 1
        else:
            how['unmatched'] += 1
    return mapping, how


# ---- analysis ---------------------------------------------------------------------------------------------------------

def district_table(index, services, nfhs_rows=None):
    """One row per district: per-service access figures, dataset coverage, and NFHS need indicators."""
    areas = districts(index)
    hospitals = PointIndex(hospital_points(index))
    credible = {s: PointIndex(credible_points(index, s)) for s in services}
    nfhs, match_stats = match_nfhs(list(areas), nfhs_rows or [])
    area_distances = {s: [] for s in services}     # every PIN area: (km to nearest credible facility, post offices)

    table = []
    for (district, state), pts in sorted(areas.items(), key=lambda kv: (kv[0][1], kv[0][0])):
        centre = (statistics.median(p[0] for p in pts), statistics.median(p[1] for p in pts))
        row = {'district': district, 'state': state, 'pin_areas': len(pts), 'post_offices': sum(p[2] for p in pts),
               'lat': round(centre[0], 4), 'lon': round(centre[1], 4),
               'hospitals_listed_within_50km': len(hospitals.keys_within(centre[0], centre[1], NEAR_KM))}
        offices = row['post_offices']
        for s in services:
            near = [(credible[s].nearest(lat, lon)[0], w) for lat, lon, w in pts]
            area_distances[s].extend(near)
            known = [(d, w) for d, w in near if d is not None]
            row[f'{s}_median_km'] = round(_weighted_median(known), 1) if known else None
            row[f'{s}_share_far'] = round(sum(w for d, w in near if d is None or d > NEAR_KM) / offices, 3)
            row[f'{s}_credible_within_50km'] = len(credible[s].keys_within(centre[0], centre[1], NEAR_KM))
        source = nfhs.get((district, state))
        row['nfhs_district'] = text(source.get('district_name')).strip() if source else None
        for name, col in NFHS_FIELDS.items():
            value, small = nfhs_number(source.get(col)) if source else (None, False)
            row[name] = value
            row[f'{name}_small_sample'] = small
        table.append(row)
    national = {s: coverage(area_distances[s], table, s) for s in services}
    return table, match_stats, national


def coverage(area_distances, table, service, thresholds=(25, 50, 100)):
    """Share of all post offices within each straight-line distance of a credible facility, plus gap-district count."""
    total = sum(w for _, w in area_distances)
    out = {f'within_{km}km': round(sum(w for d, w in area_distances if d is not None and d <= km) / total, 3)
           for km in thresholds} if total else {}
    out['districts_with_access_gap'] = sum(1 for r in table if r[f'{service}_share_far'] >= FAR_SHARE)
    out['districts'] = len(table)
    return out


def gap_kind(row):
    """Why there is no credible facility nearby: the dataset barely covers the area, or it lists hospitals that do not
    document this service."""
    return 'data gap (few listings at all)' if row['hospitals_listed_within_50km'] < FEW_HOSPITALS else \
        'service gap (hospitals listed, none documents this service)'


def maternal_priorities(table):
    """Districts where most people are far from credible C-section care AND NFHS shows the operation is rarely done."""
    rows = [r for r in table if r['c_section_share_far'] >= FAR_SHARE
            and r['csection_pct'] is not None and r['csection_pct'] < LOW_CSECTION_PCT]
    return sorted(rows, key=lambda r: (r['csection_pct'], -(r['c_section_median_km'] or 0)))


def dialysis_priorities(table, top_share=0.25):
    """Districts far from credible dialysis AND in the top quarter for raised blood sugar (diabetes drives kidney failure)."""
    def sugar(r):
        vals = [v for v in (r['high_sugar_women_pct'], r['high_sugar_men_pct']) if v is not None]
        return sum(vals) / len(vals) if vals else None
    scored = [(sugar(r), r) for r in table if sugar(r) is not None]
    if not scored:
        return []
    cut = sorted(v for v, _ in scored)[int(len(scored) * (1 - top_share))]
    rows = [r for v, r in scored if v >= cut and r['dialysis_share_far'] >= FAR_SHARE]
    for r in rows:
        r['_sugar'] = round(sugar(r), 1)
    return sorted(rows, key=lambda r: -(r['dialysis_median_km'] or 0))


def write_csv(rows, path, fields=None):
    csv.field_size_limit(sys.maxsize)
    fields = fields or [k for k in rows[0] if not k.startswith('_')]
    with open(path, 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=fields, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)
''')

**`carematch/calls.py`** · Branch-specific verification leads and budgeted maximum coverage.

In [14]:
load_embedded('calls', r'''"""Who to phone first: verification call list + budgeted maximum-coverage plan (one service, e.g. c_section).

Question (district NGO coordinator): "If I can make only 10 calls this week, which facilities should I verify first,
and which PIN areas that currently lack a credible referral option would gain one for each confirmed facility?"

Definitions (shared with the map, see docs/call_plan.md):
  PIN universe   unique 6-digit PIN codes with a reliable centre: Gazetteer.pin_info(pin)['centre'], the same point the
                 app uses when a user types that PIN. 19,586 PINs, 14,008 reliable.
  covered        a PIN with at least one Strong or Moderate facility for the service within RADIUS_KM straight-line.
                 A facility whose position is uncertain covers a PIN only if every candidate position is within range
                 (the same conservative distance recommend() uses). Duplicate listings count once.
  candidate      a facility whose service evidence is still unverified (tier Weak) but that shows obstetric or surgical
                 leads, can be pinned to one branch, and is a hospital (or has >= 20 beds). Verifying a Moderate one
                 would raise confidence but adds no coverage under this metric, so Moderate/Strong are not candidates.
  gain           newly covered unique PINs, assuming the facility confirms it provides the service.
"""
import re

from .geo import city_key
from .planner import PointIndex
from .services import SERVICES, SPECIALTY_NAME
from .textutil import fragments, haversine_km, normalize, split_camel, text

RADIUS_KM = 50
MIN_BEDS = 20
BRANCH_MISMATCH_KM = 50

OBSTETRIC_LEAD = re.compile(r'\bobstetric|\bmaternity\b|\blabou?r (room|ward|suite|room)|\bdeliveries\b'
                            r'|\b(normal|vaginal|safe|institutional|painless|baby|child) ?deliver|\bdelivery (room|suite|ward)'
                            r'|\bhigh[- ]risk pregnan|\bantenatal|\bbirthing|\bgyn(a)?ecolog', re.I)
SURGICAL_LEAD = re.compile(r'\boperation theat(re|er)|\bmodular (ot|operation)|\b(ot|o\.t\.)\b|\ban(a)?esthe|\bblood bank'
                           r'|\bblood storage|\bsurgical (unit|ward|care)|\bicu\b|\bnicu\b', re.I)
OBSTETRIC_SPECIALTIES = {'gynecologyAndObstetrics', 'obstetricsAndMaternityCare', 'obstetricsAndGynaecology',
                         'obstetricsAndGynecology', 'maternalFetalMedicineOrPerinatology', 'gynecology'}


# ---- PIN universe and current coverage -------------------------------------------------------------------------------

def pin_universe(index):
    """(reliable {pin: info}, excluded [pin]) using the app's PIN centre."""
    reliable, excluded = {}, []
    for pin in sorted(index.gaz.pins):
        info = index.gaz.pin_info(pin)
        if info['reliable']:
            info = {**info, 'offices': sum(e['n_offices'] for e in index.gaz.pins[pin])}
            reliable[pin] = info
        else:
            excluded.append(pin)
    return reliable, excluded


def positions(index, i):
    loc = index.locations[i]
    return [loc['position']] + loc.get('alternatives', []) if loc['position'] else []


def reachable_pins(pin_index, pins, pts, radius_km):
    """PINs within radius of every candidate position of one facility (conservative)."""
    if not pts:
        return set()
    near = pin_index.keys_within(pts[0][0], pts[0][1], radius_km)
    return {p for p in near if all(haversine_km(pins[p]['centre'], q) <= radius_km for q in pts)}


def current_coverage(index, service='c_section', radius_km=RADIUS_KM, pins=None):
    """Covered PIN set, per-PIN nearest credible facility, and the PIN spatial index."""
    pins = pins or pin_universe(index)[0]
    pin_index = PointIndex([(i['centre'][0], i['centre'][1], p) for p, i in pins.items()])
    credible, seen_groups = [], set()
    for i, a in enumerate(index.assessments(service)):
        if a['tier'] in ('strong', 'moderate') and index.locations[i]['position']:
            credible.append(i)
    covered = {}
    for i in credible:
        g = index.dup_group[i]
        for p in reachable_pins(pin_index, pins, positions(index, i), radius_km):
            covered.setdefault(p, set()).add(g)
        seen_groups.add(g)
    fac_index = PointIndex([(index.locations[i]['position'][0], index.locations[i]['position'][1], i) for i in credible])
    nearest = {}
    tiers = index.assessments(service)
    for p, info in pins.items():
        km, i = fac_index.nearest(*info['centre'])
        nearest[p] = (round(km, 1) if km is not None else None, tiers[i]['tier'] if i is not None else None,
                      index.rows[i].get('unique_id') if i is not None else None)
    return {'covered': set(covered), 'nearest': nearest, 'pin_index': pin_index, 'pins': pins,
            'credible_facilities': len(seen_groups)}


# ---- A line: candidates ----------------------------------------------------------------------------------------------

def _branch(name):
    parts = re.split(r'\s*(?:,|\s-\s|\s–\s|\()\s*', text(name), maxsplit=1)
    return parts[1].rstrip(')').strip() if len(parts) > 1 else ''


def _phone(row):
    official = text(row.get('officialPhone'))
    if official and normalize(official) not in ('null', 'none'):
        return official
    phones = fragments(row.get('phone_numbers'))
    return phones[0] if phones else None


def _city_centres(index):
    """(city key, state) -> [(row index, position)] for every positioned facility (used to check a branch name)."""
    out = {}
    for i, r in enumerate(index.rows):
        pos = index.locations[i]['position']
        if pos:
            out.setdefault((city_key(r.get('address_city')), index.locations[i]['state']), []).append((i, pos))
    return out


def _place_far_away(index, i, place, cities):
    """True if `place` names a known town/district in the facility's own state whose centre (from other listings,
    else the PIN directory) is far from the facility's position. Same-name places in other states are ignored."""
    key = city_key(place)
    pos = index.locations[i]['position']
    state = index.locations[i]['state']
    if not key or not pos:
        return None
    others = [p for j, p in cities.get((key, state), []) if j != i]
    if len(others) < 2:
        others = []         # one other listing is not enough to define a town centre
    if others:
        lat = sorted(p[0] for p in others)[len(others) // 2]
        lon = sorted(p[1] for p in others)[len(others) // 2]
        return haversine_km(pos, (lat, lon)) > BRANCH_MISMATCH_KM
    for d in index.gaz.districts.get(normalize(place)) or []:
        if d['state'] == state:
            return haversine_km(pos, d['centre']) > BRANCH_MISMATCH_KM
    return None


def leads(row, assessment):
    """Obstetric / surgical leads with the column and verbatim text, strongest first."""
    out = []
    for e in assessment['evidence']:
        if e['field'] == 'specialties':
            continue
        if e['kind'] == 'mention':
            out.append(('service named in description', e['field'], e['text']))
        elif e['kind'] == 'indirect':
            out.append(('service mentioned with referral/partner wording', e['field'], e['text']))
    for field in ('procedure', 'capability', 'equipment', 'description'):
        for f in fragments(row.get(field)):
            if OBSTETRIC_LEAD.search(f):
                out.append(('obstetric care', field, f))
            elif SURGICAL_LEAD.search(f):
                out.append(('surgical readiness', field, f))
    specs = sorted({s for s in fragments(row.get('specialties')) if s in OBSTETRIC_SPECIALTIES})
    for s in specs:
        out.append(('obstetric specialty listed', 'specialties', s))
    seen, unique = set(), []
    for kind, field, quote in out:
        if (field, quote) not in seen:
            seen.add((field, quote))
            unique.append((kind, field, quote))
    order = {'service named in description': 0, 'service mentioned with referral/partner wording': 1,
             'obstetric care': 2, 'surgical readiness': 3, 'obstetric specialty listed': 4}
    return sorted(unique, key=lambda x: (order[x[0]], len(x[2])))


def call_questions(row, branch, service):
    where = f"{text(row.get('name'))}" + (f" — the {branch} branch" if branch else f" in {text(row.get('address_city'))}")
    if service == 'c_section':
        return [f'Am I speaking to {where}?',
                'Are caesarean sections performed at this branch, or are patients transferred to a partner hospital?',
                'At night or on a holiday, is an obstetrician and an anaesthetist available for an emergency caesarean?',
                'Is blood available on site, and which number should we call in an emergency?']
    return [f'Am I speaking to {where}?'] + SERVICES[service]['ask']


def candidates(index, service='c_section', cov=None, radius_km=RADIUS_KM):
    """(candidates, data_check) lists of dicts. Each candidate carries the uncovered PINs it would add."""
    cov = cov or current_coverage(index, service, radius_km)
    cities = _city_centres(index)
    best_by_group, data_check = {}, []
    for i, a in enumerate(index.assessments(service)):
        if a['tier'] != 'weak':
            continue
        row, loc = index.rows[i], index.locations[i]
        ftype = normalize(row.get('facilityTypeId'))
        beds = re.search(r'\d+', text(row.get('capacity')))
        beds = int(beds.group(0)) if beds and 0 < int(beds.group(0)) <= 5000 else None
        if ftype != 'hospital' and not (beds and beds >= MIN_BEDS):
            continue
        if re.search(SPECIALTY_NAME, text(row.get('name')), re.I) or re.search(
                r'\b(school|college|institute) of nursing\b|\bnursing (school|college|institute)\b', text(row.get('name')), re.I):
            continue    # ayurveda / dental / eye centres and nursing schools are not C-section candidates
        found = leads(row, a)
        text_leads = [x for x in found if x[1] != 'specialties']
        has_obstetric = any(k in ('service named in description', 'service mentioned with referral/partner wording',
                                  'obstetric care') for k, _, _ in found)
        has_surgical = any(k == 'surgical readiness' for k, _, _ in found)
        has_specialty = any(k == 'obstetric specialty listed' for k, _, _ in found)
        if not (has_obstetric or (has_specialty and has_surgical)):
            continue
        branch = _branch(row.get('name'))
        problems = []
        if not loc['position']:
            problems.append('no usable location')
        if loc['confidence'] == 'low':
            problems.append('address, PIN and map position disagree')
        if index.mixing[i]:
            problems.append('record mixes facilities: ' + '; '.join(index.mixing[i]))
        for place, label in ((branch, 'branch name'), (row.get('address_city'), 'listed city')):
            if place and _place_far_away(index, i, place, cities):
                problems.append(f'{label} "{text(place)}" is more than {BRANCH_MISMATCH_KM} km from the mapped position')
        phone = _phone(row)
        base = {
            'facility_id': row.get('unique_id'), 'name': text(row.get('name')), 'branch': branch or text(row.get('address_city')),
            'city': text(row.get('address_city')), 'state': loc['state'], 'pin': loc['pin'],
            'lat': round(loc['position'][0], 5) if loc['position'] else None,
            'lon': round(loc['position'][1], 5) if loc['position'] else None,
            'location_confidence': loc['confidence'], 'phone': phone,
            'phone_status': 'ok' if phone else 'needs_contact', 'tier': a['tier'],
            'lead_count': len(text_leads) + (1 if has_specialty else 0),
            'leads': found[:5], 'call_questions': call_questions(row, branch, service),
            'data_flags': problems, 'beds': beds, '_i': i,
        }
        if problems:
            data_check.append(base)
            continue
        base['new_pins'] = reachable_pins(cov['pin_index'], cov['pins'], positions(index, i), radius_km) - cov['covered']
        g = index.dup_group[i]
        prev = best_by_group.get(g)
        if prev is None or (base['lead_count'], base['facility_id']) > (prev['lead_count'], prev['facility_id']):
            best_by_group[g] = base
    cands = sorted(best_by_group.values(), key=lambda c: (-len(c['new_pins']), -c['lead_count'], c['facility_id']))
    return cands, sorted(data_check, key=lambda c: c['facility_id'])


# ---- B line: budgeted maximum coverage -------------------------------------------------------------------------------

def greedy_plan(cands, budget=10, exclude=()):
    """Each round pick the candidate adding the most still-uncovered PINs (ties: more leads, then facility_id).
    Stops early when no candidate adds anything. Returns ordered rows."""
    pool = sorted([c for c in cands if c['phone_status'] == 'ok' and c['facility_id'] not in set(exclude)],
                  key=lambda c: c['facility_id'])
    covered, rows = set(), []
    for rank in range(1, budget + 1):
        best, best_new, best_key = None, set(), (0, 0)
        for c in pool:            # strict '>' keeps the first facility_id on ties -> deterministic
            new = c['new_pins'] - covered
            key = (len(new), c['lead_count'])
            if key > best_key:
                best, best_new, best_key = c, new, key
        if best is None or not best_new:
            break
        covered |= best_new
        pool.remove(best)
        rows.append({'rank': rank, 'candidate': best, 'new_pins': best_new, 'marginal_gain': len(best_new),
                     'cumulative_gain': len(covered)})
    return rows


def topk_baseline(cands, k):
    """Pick the k candidates with the largest individual gain, ignoring overlap; return unique PINs actually added."""
    pool = sorted([c for c in cands if c['phone_status'] == 'ok'],
                  key=lambda c: (-len(c['new_pins']), -c['lead_count'], c['facility_id']))[:k]
    union = set().union(*(c['new_pins'] for c in pool)) if pool else set()
    return {'k': k, 'facility_ids': [c['facility_id'] for c in pool], 'sum_individual_gains': sum(len(c['new_pins']) for c in pool),
            'unique_new_pins': len(union)}


def upper_bound(cands, k):
    """No k-set can add more than the sum of the k largest individual gains."""
    gains = sorted((len(c['new_pins']) for c in cands if c['phone_status'] == 'ok'), reverse=True)
    return sum(gains[:k])
''')

**`carematch/lead_model.py`** · Region-held-out classification of records for verification.

In [15]:
load_embedded('lead_model', r'''"""C-section verification leads: which facilities look like ones that document caesarean delivery, but do not?

Positive-unlabeled setup. Facilities whose records give Strong evidence for C-section are proxy positives; every other
record is unlabeled (it may well offer the service without saying so). A logistic regression learns what distinguishes
the proxy positives using only features that cannot spell out the label:

  * no C-section or maternity vocabulary: every claim fragment, description sentence or specialty that matches BLOCK is
    removed before any feature is computed, so adding or deleting such text cannot change a facility's features;
  * nothing derived from the label: no evidence tier, claim kind or count of service claims is a feature.

Evaluation holds out whole regions of India (leave-one-region-out) and keeps every facility group (duplicate listings,
same-name branches) inside a single fold, so a facility's twin never sits on the other side of the split.

Scores rank facilities for a planner's verification calls. They are not evidence that a facility offers the service.
numpy and scikit-learn are imported lazily so the engine itself stays standard-library only.
"""
import math
import re
from collections import Counter

from .geo import canonical_state
from .services import INCOMPATIBLE_TYPES
from .textutil import fragments, normalize, source_urls, split_camel, text

SERVICE = 'c_section'
CURRENT_YEAR = 2026
USE_NOTE = ('Planner verification lead from a statistical model, not evidence that this facility offers C-section. '
            'Confirm by phone before any referral.')

# Service and maternity vocabulary. Any text that matches is invisible to the model.
BLOCK = re.compile(
    r'caesar|cesar|\bc[- ]?sections?\b|\blscs\b|cemonc|obstet|gyn(a)?ec|matern|deliver|labou?r\s*(room|ward|suite)|'
    r'pregnan|natal|neonat|newborn|\bnicu\b|\bsncu\b|perinat|fetal|foetal|infertil|fertility|\bivf\b|reproductive|'
    r'midwi|birth|anaesth|anesth|blood\s*bank|\bwomen|\bmother|\bobg\b|\bob[/ -]?gyn', re.I)

# Generic capability signals, matched only in fragments that survived BLOCK. None of them names the service.
INDICATORS = {
    'icu': (r'\bicu\b|\biccu\b|intensive care|critical care', 'mentions an ICU or critical care'),
    'operation_theatre': (r'operation theat|operating (room|theat)|\bot\b|\bots\b|modular theat|surgical suite',
                          'mentions an operation theatre'),
    'ventilator': (r'ventilator', 'mentions ventilators'),
    'round_the_clock': (r'24\s*[x/×]\s*7|24 hours|round[- ]the[- ]clock|24-hour', 'mentions round-the-clock service'),
    'ambulance': (r'ambulance', 'mentions an ambulance'),
    'imaging_ct_mri': (r'\bct\b|computed tomography|\bmri\b|magnetic resonance', 'mentions CT or MRI'),
    'ultrasound': (r'ultraso|sonograph', 'mentions ultrasound'),
    'laparoscopy': (r'laparoscop|endoscop', 'mentions laparoscopy or endoscopy'),
    'dialysis': (r'dialy', 'mentions dialysis'),
    'laboratory': (r'laborator|\blab\b|pathology', 'mentions a laboratory'),
    'inpatient': (r'\bwards?\b|\bbeds?\b|inpatient|\bipd\b|admission', 'mentions inpatient beds or wards'),
    'insurance': (r'insurance|cashless|ayushman|pm-?jay|cghs|\btpa\b', 'mentions insurance or cashless schemes'),
    'surgery': (r'surger|surgical|operat(e|ion|ive)', 'mentions surgery'),
}
_INDICATOR_RE = {k: re.compile(p, re.I) for k, (p, _) in INDICATORS.items()}
CLAIM_FIELDS = ('procedure', 'equipment', 'capability')

# Specialty families (first match wins). Counting families instead of 2,900 individual enum values keeps coefficients
# stable and explanations readable ("lists 6 surgical specialties").
SPECIALTY_FAMILIES = [
    ('dental', r'dent|odont|maxillofacial|oral'),
    ('eye', r'ophthalm|retina|cornea|glaucoma|cataract|strabism|ocul|refractive|vitreo'),
    ('alternative', r'ayurved|homeo|unani|siddha|naturopath|yoga'),
    ('emergency_critical', r'emergency|critical care|trauma|intensiv'),
    ('surgical', r'surg|urolog|orthop'),
    ('pediatric', r'pediatric|paediatric|child'),
    ('diagnostic', r'radiolog|patholog|imaging|laborator|nuclear'),
    ('internal_medicine', r'internal medicine|family medicine|general medicine|general practice'),
    ('medical_specialty', r'cardio|nephro|neuro|gastro|pulmon|endocrin|oncolog|hematolog|rheumat|infectious|hepat|'
                          r'dermat|psychiat|allerg|geriatric|sleep'),
]
_FAMILY_RE = [(k, re.compile(p, re.I)) for k, p in SPECIALTY_FAMILIES]
FAMILY_LABEL = {'dental': 'dental', 'eye': 'eye-care', 'alternative': 'traditional-medicine',
                'emergency_critical': 'emergency or critical-care', 'surgical': 'surgical', 'pediatric': 'paediatric',
                'diagnostic': 'diagnostic', 'internal_medicine': 'general-medicine', 'medical_specialty': 'medical',
                'other': 'other'}

REGIONS = {
    'North': ['Jammu And Kashmir', 'Ladakh', 'Himachal Pradesh', 'Punjab', 'Chandigarh', 'Haryana', 'Delhi',
              'Uttarakhand', 'Uttar Pradesh', 'Rajasthan'],
    'West': ['Gujarat', 'Maharashtra', 'Goa', 'The Dadra And Nagar Haveli And Daman And Diu'],
    'Central': ['Madhya Pradesh', 'Chhattisgarh'],
    'East': ['Bihar', 'Jharkhand', 'Odisha', 'West Bengal'],
    'Northeast': ['Assam', 'Arunachal Pradesh', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Tripura', 'Sikkim'],
    'South': ['Andhra Pradesh', 'Telangana', 'Karnataka', 'Kerala', 'Tamil Nadu', 'Puducherry', 'Lakshadweep',
              'Andaman And Nicobar Islands'],
}
STATE_REGION = {s: r for r, states in REGIONS.items() for s in states}


def blocked(value):
    """True when the text names the service or maternity care, read both as written and with camelCase split
    (the split catches "gynecologyAndObstetrics"; the raw form keeps acronyms such as "CEmONC" whole)."""
    raw = text(value)
    return bool(BLOCK.search(raw) or BLOCK.search(split_camel(raw)))


def _number(value, low, high):
    m = re.search(r'\d+(\.\d+)?', text(value).replace(',', ''))
    v = float(m.group(0)) if m else None
    return v if v is not None and low <= v <= high else None


def _sentences(value):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text(value)) if s.strip()]


def specialty_family(name):
    words = split_camel(name).lower()
    return next((k for k, rx in _FAMILY_RE if rx.search(words)), 'other')


def facility_features(row):
    """(features, citations) for one record. citations[name] = (field, verbatim text) behind a non-zero feature."""
    f, cite = {}, {}
    ftype = normalize(row.get('facilityTypeId'))
    f['type_hospital'] = float(ftype == 'hospital')
    f['type_clinic'] = float(ftype == 'clinic')
    f['type_other'] = float(ftype not in ('hospital', 'clinic'))
    if ftype:
        cite['type_hospital' if ftype == 'hospital' else 'type_clinic' if ftype == 'clinic' else 'type_other'] = \
            ('facilityTypeId', text(row.get('facilityTypeId')))
    op = normalize(row.get('operatorTypeId'))
    f['operator_public'] = float(op == 'public')
    f['operator_private'] = float(op == 'private')
    if op in ('public', 'private'):
        cite[f'operator_{op}'] = ('operatorTypeId', text(row.get('operatorTypeId')))

    for name, field, low, high in (('beds', 'capacity', 1, 5000), ('doctors', 'numberDoctors', 1, 2000)):
        v = _number(row.get(field), low, high)
        f[f'log_{name}'] = math.log1p(v) if v else 0.0
        f[f'{name}_missing'] = float(v is None)
        if v:
            cite[f'log_{name}'] = (field, text(row.get(field)))
    year = _number(row.get('yearEstablished'), 1800, CURRENT_YEAR)
    f['log_age'] = math.log1p(CURRENT_YEAR - year) if year else 0.0
    f['age_missing'] = float(year is None)
    if year:
        cite['log_age'] = ('yearEstablished', text(row.get('yearEstablished')))

    f['has_phone'] = float(bool(fragments(row.get('phone_numbers')) or text(row.get('officialPhone')) not in ('', 'null')))
    f['has_website'] = float(text(row.get('officialWebsite')) not in ('', 'null'))
    f['log_sources'] = math.log1p(len(source_urls(row.get('source_urls'))))
    followers = _number(row.get('engagement_metrics_n_followers'), 0, 1e9)
    f['log_followers'] = math.log1p(followers) if followers else 0.0

    specs = [s for s in dict.fromkeys(fragments(row.get('specialties'))) if not blocked(s)]
    families = {}
    for s in specs:
        families.setdefault(specialty_family(s), []).append(s)
    for fam in [k for k, _ in SPECIALTY_FAMILIES] + ['other']:
        members = families.get(fam, [])
        f[f'spec_{fam}'] = math.log1p(len(members))
        if members:
            cite[f'spec_{fam}'] = ('specialties', ', '.join(members[:6]) + (' …' if len(members) > 6 else ''))

    kept = []
    for field in CLAIM_FIELDS:
        items = [x for x in fragments(row.get(field)) if not blocked(x)]
        f[f'log_{field}_items'] = math.log1p(len(items))
        kept += [(field, x) for x in items]
    sentences = [s for s in _sentences(row.get('description')) if not blocked(s)]
    f['log_description_words'] = math.log1p(sum(len(s.split()) for s in sentences))
    kept += [('description', s) for s in sentences]
    for name, rx in _INDICATOR_RE.items():
        hit = next(((field, x) for field, x in kept if rx.search(x)), None)
        f[f'ind_{name}'] = float(hit is not None)
        if hit:
            cite[f'ind_{name}'] = hit
    return f, cite


def feature_label(name, row, value=None):
    if name.startswith('spec_'):
        n = round(math.expm1(value)) if value is not None else None
        return f"lists {n if n is not None else 'some'} {FAMILY_LABEL[name[5:]]} specialt{'y' if n == 1 else 'ies'}"
    if name.startswith('ind_'):
        return INDICATORS[name[4:]][1]
    labels = {
        'type_hospital': 'listed as a hospital', 'type_clinic': 'listed as a clinic', 'type_other': 'other facility type',
        'operator_public': 'public operator', 'operator_private': 'private operator',
        'log_beds': f"{text(row.get('capacity'))} beds listed", 'beds_missing': 'bed count missing',
        'log_doctors': f"{text(row.get('numberDoctors'))} doctors listed", 'doctors_missing': 'doctor count missing',
        'log_age': f"established {text(row.get('yearEstablished'))}", 'age_missing': 'founding year missing',
        'has_phone': 'phone number listed', 'has_website': 'official website listed',
        'log_sources': 'number of source pages', 'log_followers': 'social media followers',
        'log_specialties': 'number of specialties listed', 'log_description_words': 'length of description',
    }
    if name in labels:
        return labels[name]
    m = re.match(r'log_(\w+)_items', name)
    return f'number of {m.group(1)} items' if m else name


# ---- groups and geography ---------------------------------------------------------------------------------------------

def facility_groups(index):
    """Group id per row: duplicate listings (engine) joined with records that share a normalized name (branches)."""
    parent = list(range(len(index.rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    first = {}
    for i, r in enumerate(index.rows):
        for key in (('dup', index.dup_group[i]), ('name', re.sub(r'[^a-z0-9]+', ' ', normalize(r.get('name'))).strip())):
            if key[0] == 'name' and key[1] in ('', 'null', 'none'):
                continue
            if key in first:
                union(first[key], i)
            else:
                first[key] = i
    return [find(i) for i in range(len(index.rows))]


def row_region(index, i):
    state = index.locations[i].get('state') or canonical_state(index.rows[i].get('address_stateOrRegion'))
    return STATE_REGION.get(state, 'Unknown')


def group_regions(index, groups):
    """Region of each row's group (majority region of its members), so a group never straddles folds."""
    members = {}
    for i, g in enumerate(groups):
        members.setdefault(g, []).append(row_region(index, i))
    vote = {g: (Counter(r for r in regs if r != 'Unknown').most_common(1) or [('Unknown', 0)])[0][0]
            for g, regs in members.items()}
    return [vote[g] for g in groups], sum(1 for i, g in enumerate(groups) if row_region(index, i) != vote[g])


# ---- data, model, evaluation ------------------------------------------------------------------------------------------

def build_dataset(index):
    """Feature matrix inputs, proxy labels and fold keys for every facility."""
    feats, cites = zip(*(facility_features(r) for r in index.rows))
    names = list(feats[0])
    tiers = [a['tier'] for a in index.assessments(SERVICE)]
    groups = facility_groups(index)
    regions, moved = group_regions(index, groups)
    return {'names': names, 'X': [[f[n] for n in names] for f in feats], 'citations': list(cites),
            'y': [int(t == 'strong') for t in tiers], 'tiers': tiers, 'groups': groups, 'regions': regions,
            'rows_moved_to_group_region': moved}


def make_model(C=0.3):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=5000))


def cross_validate(data, labels=None, C=0.3):
    """Leave-one-region-out out-of-fold scores, plus each row's held-out model (for explanations)."""
    import numpy as np
    X, y = np.asarray(data['X']), np.asarray(labels if labels is not None else data['y'])
    regions = np.asarray(data['regions'])
    oof, fold_model = np.full(len(y), np.nan), {}
    for region in sorted(set(regions)):
        test = regions == region
        train = ~test
        if y[train].sum() == 0 or y[train].sum() == train.sum():
            continue
        model = make_model(C).fit(X[train], y[train])
        oof[test] = model.predict_proba(X[test])[:, 1]
        fold_model[region] = model
    return oof, fold_model


def baseline_scores(data):
    """Simple rule: hospitals first, then more listed (non-maternity) specialties."""
    names = data['names']
    h = names.index('type_hospital')
    spec = [j for j, n in enumerate(names) if n.startswith('spec_')]
    return [x[h] * 100 + sum(math.expm1(x[j]) for j in spec) for x in data['X']]


def metrics(y, scores, ks=(50, 100, 200), min_positives=1):
    import numpy as np
    from sklearn.metrics import average_precision_score, roc_auc_score
    y, scores = np.asarray(y), np.asarray(scores, dtype=float)
    keep = ~np.isnan(scores)
    y, scores = y[keep], scores[keep]
    out = {'n': int(len(y)), 'positives': int(y.sum()), 'prevalence': round(float(y.mean()), 4) if len(y) else None}
    if y.sum() < min_positives:
        out['note'] = f'fewer than {min_positives} positives: AUC not reported'
    elif 0 < y.sum() < len(y):
        out['roc_auc'] = round(float(roc_auc_score(y, scores)), 3)
        out['pr_auc'] = round(float(average_precision_score(y, scores)), 3)
        order = np.argsort(-scores, kind='stable')
        for k in ks:
            if k <= len(y):
                out[f'precision_at_{k}'] = round(float(y[order[:k]].mean()), 3)
    return out


def explain(model, x, names, row, citations, top=4):
    """Top positive contributions (standardized value × coefficient) with the facility text behind each one.

    Only features the record actually has count as reasons (value > 0 with a positive coefficient). A positive
    contribution from something the record lacks ("not a clinic") is real but cannot be cited, so it is left out.
    """
    import numpy as np
    scaler, lr = model.named_steps['standardscaler'], model.named_steps['logisticregression']
    x = np.asarray(x, dtype=float)
    coef = lr.coef_[0]
    contrib = (x - scaler.mean_) / scaler.scale_ * coef
    reasons, cited = [], []
    for j in np.argsort(-contrib):
        if contrib[j] <= 0 or len(reasons) >= top:
            break
        name = names[j]
        if coef[j] <= 0 or x[j] <= 0 or name.endswith('_missing') or name == 'type_other':
            continue
        reasons.append(f'{feature_label(name, row, x[j])} (+{contrib[j]:.2f})')
        if name in citations:
            field, quote = citations[name]
            cited.append(f'{field}: "{quote[:160]}"')
    return reasons, cited


def leads(index, data, oof, fold_model, n=200):
    """Unlabeled facilities (Weak or no C-section evidence) ranked by held-out score, with reasons and cited text."""
    import numpy as np
    assessed = index.assessments(SERVICE)
    pool = [i for i, t in enumerate(data['tiers'])
            if t in ('weak', 'none') and not np.isnan(oof[i])
            and normalize(index.rows[i].get('facilityTypeId')) not in INCOMPATIBLE_TYPES]
    scores = np.sort(np.asarray([oof[i] for i in pool]))
    out = []
    for rank, i in enumerate(sorted(pool, key=lambda i: (-oof[i], text(index.rows[i].get('unique_id'))))[:n], 1):
        pct = float(np.searchsorted(scores, oof[i], side='right') / len(scores))
        row, loc = index.rows[i], index.locations[i]
        reasons, cited = explain(fold_model[data['regions'][i]], data['X'][i], data['names'], row, data['citations'][i])
        phones = fragments(row.get('phone_numbers'))
        a = assessed[i]
        out.append({
            'rank': rank, 'facility_id': text(row.get('unique_id')), 'name': text(row.get('name')),
            'city': text(row.get('address_city')), 'district': loc.get('district'), 'state': loc.get('state'),
            'lat': round(loc['position'][0], 5) if loc.get('position') else None,
            'lon': round(loc['position'][1], 5) if loc.get('position') else None,
            'phone': text(row.get('officialPhone')) if text(row.get('officialPhone')) not in ('', 'null') else (phones[0] if phones else None),
            'current_c_section_tier': a['tier'],
            'current_evidence_note': '; '.join(a['reasons']) or 'no C-section claim in the record',
            'lead_score': round(float(oof[i]), 4), 'score_percentile_among_unlabeled': round(pct, 4),
            'held_out_region': data['regions'][i], 'reasons': ' | '.join(reasons), 'cited_text': ' | '.join(cited),
            'use_note': USE_NOTE,
        })
    return out


def coefficients(data, C=0.3):
    """Standardized coefficients of a model fitted on every facility (for the report only)."""
    model = make_model(C).fit(data['X'], data['y'])
    coef = model.named_steps['logisticregression'].coef_[0]
    return sorted(({'feature': n, 'coefficient': round(float(c), 3)} for n, c in zip(data['names'], coef)),
                  key=lambda r: -abs(r['coefficient']))
''')

In [16]:
from _carematch_submission.datasource import load, load_nfhs
from _carematch_submission.textutil import is_empty, normalize
from _carematch_submission.services import SERVICES
from _carematch_submission.recommend import CareIndex, recommend
from _carematch_submission.render import to_text
from _carematch_submission.planner import district_table, maternal_priorities, dialysis_priorities, gap_kind

## Part 2 · Load the data

Three tables from `databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset`: `facilities`
(10,088 rows × 51 columns), `india_post_pincode_directory` (aggregated to one row per PIN × district) and
`nfhs_5_district_health_indicators` (706 districts). Then every facility is assessed once for all nine services.


In [17]:
started = time.perf_counter()
if ON_DATABRICKS:
    facility_rows, pincode_rows, load_info = load(spark=spark)
    nfhs_rows = load_nfhs(spark=spark)
else:
    facility_rows, pincode_rows, load_info = load(order=['local'])
    nfhs_rows = load_nfhs()
index = CareIndex(facility_rows, pincode_rows)
for service in SERVICES:
    index.assessments(service)
print(load_info)
print(f'{len(nfhs_rows)} NFHS-5 districts')
print(f'Index built and all {len(SERVICES)} services assessed in {time.perf_counter() - started:.1f} s')


{'source': 'local', 'facility_rows': 10088, 'pincode_rows': 21162, 'load_seconds': 0.4, 'failed_sources': []}
706 NFHS-5 districts
Index built and all 9 services assessed in 30.5 s


## Part 3 · Exploration: how much can each record be trusted?

The records were extracted automatically from web pages (Practo, JustDial, Facebook, hospital sites). Three problems
shape everything that follows: **claims are not facts**, **locations contradict each other**, and **some records merge
two facilities**. The cells below measure each one on the full table.


In [18]:
cols = ['name', 'facilityTypeId', 'operatorTypeId', 'specialties', 'procedure', 'equipment', 'capability',
        'description', 'phone_numbers', 'latitude', 'address_zipOrPostcode', 'source_urls', 'capacity', 'numberDoctors',
        'yearEstablished']
n = len(facility_rows)
show([{'column': c, 'filled': f"{sum(not is_empty(r.get(c)) for r in facility_rows) / n:.0%}"} for c in cols],
     ['column', 'filled'])
print()
types = Counter(normalize(r.get('facilityTypeId')) or '(empty)' for r in facility_rows)
operators = Counter(normalize(r.get('operatorTypeId')) or '(empty)' for r in facility_rows)
print('Facility type:', ', '.join(f'{k} {v:,}' for k, v in types.most_common(5)))
print('Operator:     ', ', '.join(f'{k} {v:,}' for k, v in operators.most_common(3)))
print('\nScale fields (beds, doctors) are filled for a minority of rows, so they can only cap a tier, never rank.')


column                 filled
---------------------  ------
name                   99%   
facilityTypeId         99%   
operatorTypeId         92%   
specialties            99%   
procedure              91%   
equipment              76%   
capability             99%   
description            99%   
phone_numbers          97%   
latitude               99%   
address_zipOrPostcode  99%   
source_urls            99%   
capacity               25%   
numberDoctors          36%   
yearEstablished        48%   

Facility type: hospital 5,637, clinic 3,782, dentist 490, (empty) 67, null 59
Operator:      private 8,842, null 688, public 469

Scale fields (beds, doctors) are filled for a minority of rows, so they can only cap a tier, never rank.


In [19]:
conf = Counter(loc['confidence'] for loc in index.locations)
basis = Counter(loc['basis'] or 'no usable position' for loc in index.locations)
flags = Counter(f for loc in index.locations for f in loc['flags'])
print('Location confidence:', dict(conf.most_common()))
print('\nPosition used for distance:')
for k, v in basis.most_common():
    print(f'  {v:6,}  {k}')
print('\nLocation flags:', dict(flags.most_common(6)))


Location confidence: {'high': 8017, 'low': 933, 'medium': 679, 'approximate': 459}

Position used for distance:
   9,530  facility coordinates
     434  PIN code centre (record coordinates contradict PIN and city)
      91  no usable position
      25  PIN code centre (approximate)
       8  city centre (approximate)

Location flags: {'location_conflict': 834, 'coords_wrong_used_pin': 434, 'state_field_not_a_state': 345, 'pin_not_in_directory': 206, 'pin_missing_or_malformed': 167, 'pin_inconsistent': 131}


In [20]:
groups = Counter(index.dup_group)
dup_groups = sum(1 for c in groups.values() if c > 1)
dup_rows = sum(c for c in groups.values() if c > 1)
mixed = sum(1 for m in index.mixing if m)
print(f'Duplicate listings: {dup_rows} rows in {dup_groups} groups (same name and same PIN or within 2 km); shown once, sources kept.')
print(f'Records that look merged with another facility: {mixed:,} of {len(facility_rows):,} ({mixed / len(facility_rows):.0%}).')
example = next((facility_rows[i].get('name'), m) for i, m in enumerate(index.mixing) if m)
print('Example:', example[0], '->', example[1][0] if isinstance(example[1], list) else example[1])


Duplicate listings: 30 rows in 15 groups (same name and same PIN or within 2 km); shown once, sources kept.
Records that look merged with another facility: 1,217 of 10,088 (12%).
Example: Aravind Eye Hospital -> 3 statements describe another facility ("Delhi Government Hospitals")


In [21]:
rows = []
for key, spec in SERVICES.items():
    tiers = Counter(a['tier'] for a in index.assessments(key))
    rows.append({'service': spec['label'], 'strong': tiers['strong'], 'moderate': tiers['moderate'],
                 'weak': tiers['weak'], 'none': tiers['none']})
show(rows, ['service', 'strong', 'moderate', 'weak', 'none'])
print('\nOnly Strong and Moderate are ever recommended. Weak = only referral/outsourcing, a passing mention, or a related department.')

print('\nWhat each kind of claim looks like (real text):')
seen = {}
for key in ('c_section', 'mri', 'dialysis', 'cardiac_surgery'):
    for a in index.assessments(key):
        for e in a['evidence']:
            seen.setdefault(e['kind'], (SERVICES[key]['label'], e['field'], e['text']))
for kind in ('direct', 'listed', 'indirect', 'negative', 'mention', 'related'):
    if kind in seen:
        label, field, quote = seen[kind]
        print(f'  {kind:9s} [{label}, {field}] "{quote[:110]}"')


service                         strong  moderate  weak  none
------------------------------  ------  --------  ----  ----
Dialysis                        600     435       1513  7540
C-section (caesarean delivery)  126     313       4716  4933
NICU (newborn intensive care)   278     429       1046  8335
Trauma care                     623     701       4198  4566
Emergency care                  1295    1477      2180  5136
Burn care                       38      143       1761  8146
Cardiac surgery                 275     326       3383  6104
MRI scan                        602     318       3691  5477
CT scan                         947     483       3155  5503

Only Strong and Moderate are ever recommended. Weak = only referral/outsourcing, a passing mention, or a related department.

What each kind of claim looks like (real text):
  direct    [C-section (caesarean delivery), procedure] "Caesarean Section"
  listed    [C-section (caesarean delivery), capability] "Maternity care progr

## Part 4 · Recommendations

Five queries chosen to show the behaviours that matter: plenty of strong options (Patna), a trade-off between near and
better evidenced (Darbhanga), an honest long-distance answer (Gaya), a remote northeast district (Kohima), and an honest
"nothing credible within 300 km" (Port Blair). Each answer carries its own quality review.


In [22]:
DEMO = [('Dialysis', 'Patna'), ('MRI scan', 'Darbhanga'), ('Burn care', 'Gaya'), ('Trauma care', 'Kohima'), ('Burn care', 'Port Blair')]

for service, place in DEMO:
    result = recommend(index, service, place)
    print('=' * 100)
    print(to_text(result))
    if 'review' in result:
        checks = result['review']['checks']
        print(f"Quality review: {sum(c['passed'] for c in checks)}/{len(checks)} checks passed")
    print()


Dialysis near Patna, Bihar  (location precision ~3.0 km)
  note: Distances are measured from the centre of the listed facilities in this city.
  what the evidence can show: Records show a dialysis unit or dialysis treatment; whether a slot is free today must be confirmed.
What to do: 24 facilities with credible evidence within 25 km. Call the first option; if it cannot confirm, move to the next one.
Searched up to 25 km.

1. Nectar Hospital
   0.8 km · STRONG evidence · hospital, private, 2200 beds
   A- 36, P C Colony Rd, Kankarbagh, PIN 800020
   Why: 3 distinct claims across equipment, procedure
     “Performs dialysis treatment”  [procedure]
     “Provides dialysis treatment in a dedicated dialysis unit”  [procedure]
     “Dialysis unit”  [equipment]
   Order: #1 of 24 facilities with credible evidence within 25 km; ordered by evidence strength, then distance.
   Call: +919155990993 · nectarhospital.in
   Ask: Is the dialysis unit running today, and is a slot free?

2. Patna Dialys

## Part 5 · Evaluation

**Golden set**: 38 queries covering the nine services in metros, tier-2 cities, rural PIN codes and remote or northeast
districts, including places where the honest answer is "nothing credible nearby". Automatic metrics are recomputed here;
the review labels come from the layered review in the repo (`eval/review_sheet.csv`): two independent model reviewers
judged every top-1 quote, and a team member adjudicated every disagreement plus a random sample of agreements.


In [23]:
# Golden queries, embedded from eval/queries.csv: (id, service, place, place type)
GOLDEN = [('q01', 'dialysis', 'Delhi', 'metro'), ('q02', 'dialysis', 'Patna', 'tier2'), ('q03', 'dialysis', '855107', 'rural_pin'), ('q04', 'dialysis', 'Kohima', 'remote'), ('q05', 'emergency C-section', 'Mumbai', 'metro'), ('q06', 'emergency C-section', 'Lucknow', 'tier2'), ('q07', 'emergency C-section', 'Aurangabad, Bihar', 'rural_district'), ('q08', 'emergency C-section', 'Mon', 'remote'), ('q09', 'NICU', 'Chennai', 'metro'), ('q10', 'NICU', 'Ranchi', 'tier2'), ('q11', 'NICU', '835207', 'rural_pin'), ('q12', 'NICU', 'Aizawl', 'remote'), ('q13', 'trauma', 'Kolkata', 'metro'), ('q14', 'trauma', 'Indore', 'tier2'), ('q15', 'trauma', '271801', 'rural_pin'), ('q16', 'trauma', 'Kohima', 'remote'), ('q17', 'emergency', 'Hyderabad', 'metro'), ('q18', 'emergency', 'Bhubaneswar', 'tier2'), ('q19', 'emergency', '764020', 'rural_pin'), ('q20', 'emergency', 'Tawang', 'remote'), ('q21', 'burn care', 'Bengaluru', 'metro'), ('q22', 'burn care', 'Gaya', 'tier2'), ('q23', 'burn care', '852201', 'rural_pin'), ('q24', 'burn care', 'Mon', 'remote'), ('q25', 'cardiac surgery', 'Ahmedabad', 'metro'), ('q26', 'cardiac surgery', 'Bhopal', 'tier2'), ('q27', 'cardiac surgery', '822101', 'rural_pin'), ('q28', 'cardiac surgery', 'Shillong', 'remote'), ('q29', 'MRI', 'Pune', 'metro'), ('q30', 'MRI', 'Darbhanga', 'tier2'), ('q31', 'MRI', '473001', 'rural_pin'), ('q32', 'MRI', 'Leh', 'remote'), ('q33', 'CT scan', 'Kolkata', 'metro'), ('q34', 'CT scan', 'Varanasi', 'tier2'), ('q35', 'CT scan', '845401', 'rural_pin'), ('q36', 'CT scan', 'Kargil', 'remote'), ('q37', 'burn care', 'Port Blair', 'remote'), ('q38', 'cardiac surgery', 'Port Blair', 'remote')]

runs, latencies = [], []
for qid, service, place, place_type in GOLDEN:
    t = time.perf_counter()
    r = recommend(index, service, place)
    latencies.append((time.perf_counter() - t) * 1000)
    runs.append((qid, service, place, place_type, r))

errors = [q for q, *_, r in runs if 'error' in r]
answered = [x for x in runs if x[4].get('results')]
failed_checks = Counter(c['name'] for *_, r in runs for c in r.get('review', {}).get('checks', []) if not c['passed'])
quotes = sum(len(c['evidence']) for *_, r in runs for c in r.get('results', []))
dups = [q for q, *_, r in runs if len({c['facility_id'] for c in r.get('results', [])}) != len(r.get('results', []))]
tiers = Counter(c['tier'] for *_, r in runs for c in r.get('results', []))
lat = sorted(latencies)
print(f'Queries: {len(runs)}  errors: {len(errors)}  answered within 300 km: {len(answered)}  honest no-answer: {len(runs) - len(answered) - len(errors)}')
counts = [len(r.get('review', {}).get('checks', [])) for *_, r in runs]
span = f'{min(counts)}' if min(counts) == max(counts) else f'{min(counts)}–{max(counts)}'
print(f'All quality checks passed ({span} checks per answer): '
      f'{sum(1 for *_, r in runs if r.get("review", {}).get("passed"))}/{len(runs)}'
      + (f'  failures: {dict(failed_checks)}' if failed_checks else ''))
print(f'Quotes shown in recommendations: {quotes} (each verified verbatim against its source column by the review)')
print(f'Duplicate facility inside a top 3: {len(dups)}')
print(f'Recommendations: {sum(tiers.values())} ({tiers["strong"]} Strong, {tiers["moderate"]} Moderate)')
print(f'Latency with a warm index: p50 {lat[len(lat) // 2]:.0f} ms, p95 {lat[int(len(lat) * 0.95) - 1]:.0f} ms')

print('\nMedian straight-line distance to the top recommendation:')
by_type = {}
for qid, service, place, place_type, r in answered:
    by_type.setdefault(place_type, []).append(r['results'][0]['distance_km'])
show([{'place type': k, 'queries answered': len(v), 'median km': round(statistics.median(v), 1)} for k, v in by_type.items()],
     ['place type', 'queries answered', 'median km'])


Queries: 38  errors: 0  answered within 300 km: 36  honest no-answer: 2
All quality checks passed (8 checks per answer): 38/38
Quotes shown in recommendations: 248 (each verified verbatim against its source column by the review)
Duplicate facility inside a top 3: 0
Recommendations: 104 (89 Strong, 15 Moderate)
Latency with a warm index: p50 25 ms, p95 115 ms

Median straight-line distance to the top recommendation:
place type      queries answered  median km
--------------  ----------------  ---------
metro           9                 2.4      
tier2           9                 1.3      
rural_pin       8                 113.5    
remote          9                 116.9    
rural_district  1                 113.9    


In [24]:
def top_ids(r):
    return [c['facility_id'] for c in r.get('results', [])]

baseline = {q: top_ids(r) for q, *_, r in runs}
variants = {'distance bands 20/60/120/250 km': dict(bands=(20, 60, 120, 250)),
            'distance bands 30/90/180/350 km': dict(bands=(30, 90, 180, 350)),
            'Strong requires >= 3 distinct claims': dict(strong_min_claims=3)}
rows = []
for name, kw in variants.items():
    same_top1, jacc = 0, []
    for qid, service, place, place_type in GOLDEN:
        ids = top_ids(recommend(index, service, place, review=False, **kw))
        base = baseline[qid]
        same_top1 += (ids[:1] == base[:1])
        jacc.append(len(set(ids) & set(base)) / len(set(ids) | set(base)) if (ids or base) else 1.0)
    rows.append({'variant': name, 'same top-1': f'{same_top1 / len(GOLDEN):.0%}', 'top-3 overlap (Jaccard)': f'{statistics.mean(jacc):.2f}'})
show(rows, ['variant', 'same top-1', 'top-3 overlap (Jaccard)'])
print('\nThe answer barely depends on the exact distance cut-offs and more on how much evidence "Strong" demands,')
print('which is why the product shows tiers and the quotes themselves rather than a precise score.')


variant                               same top-1  top-3 overlap (Jaccard)
------------------------------------  ----------  -----------------------
distance bands 20/60/120/250 km       95%         0.90                   
distance bands 30/90/180/350 km       97%         0.96                   
Strong requires >= 3 distinct claims  74%         0.64                   

The answer barely depends on the exact distance cut-offs and more on how much evidence "Strong" demands,
which is why the product shows tiers and the quotes themselves rather than a precise score.


In [25]:
# Review labels, embedded from eval/review_sheet.csv
REVIEW = {'q01': {'service': 'dialysis', 'place': 'Delhi', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q02': {'service': 'dialysis', 'place': 'Patna', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q03': {'service': 'dialysis', 'place': '855107', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q04': {'service': 'dialysis', 'place': 'Kohima', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q05': {'service': 'emergency C-section', 'place': 'Mumbai', 'model_quote_supports': 'yes', 'model_one_facility': 'duplicate (now merged and flagged)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'duplicate', 'models_agree': 'no', 'human_quote_supports': 'partly', 'human_one_facility': 'duplicate', 'human_note': 'Two questions: (1) does the quote prove the hospital performs C-sections? yes. (2) does it prove emergency readiness now (surgeon, anaesthetist, blood)? unknown, no evidence: confirm by phone. Scored "partly" for an emergency-C-section request. Metro: more alternatives exist, still confirm real-time capability. Duplicate row now merged.'}, 'q06': {'service': 'emergency C-section', 'place': 'Lucknow', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'no', 'human_quote_supports': 'partly', 'human_one_facility': 'yes', 'human_note': 'Two questions: (1) does the quote prove the hospital performs C-sections? yes. (2) does it prove emergency readiness now (surgeon, anaesthetist, blood)? unknown, no evidence: confirm by phone. Scored "partly" for an emergency-C-section request. Tier-2 city: verify more strictly; fewer alternatives and blood/anaesthesia cover may be limited.'}, 'q07': {'service': 'emergency C-section', 'place': 'Aurangabad, Bihar', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'no', 'human_quote_supports': 'partly', 'human_one_facility': 'yes', 'human_note': 'Two questions: (1) does the quote prove the hospital performs C-sections? yes. (2) does it prove emergency readiness now (surgeon, anaesthetist, blood)? unknown, no evidence: confirm by phone. Scored "partly" for an emergency-C-section request.'}, 'q08': {'service': 'emergency C-section', 'place': 'Mon', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'no', 'human_quote_supports': 'partly', 'human_one_facility': 'yes', 'human_note': 'Two questions: (1) does the quote prove the hospital performs C-sections? yes. (2) does it prove emergency readiness now (surgeon, anaesthetist, blood)? unknown, no evidence: confirm by phone. Scored "partly" for an emergency-C-section request. Remote: few alternatives, verify before travelling.'}, 'q09': {'service': 'NICU', 'place': 'Chennai', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q10': {'service': 'NICU', 'place': 'Ranchi', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q11': {'service': 'NICU', 'place': '835207', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q12': {'service': 'NICU', 'place': 'Aizawl', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q13': {'service': 'trauma', 'place': 'Kolkata', 'model_quote_supports': 'partly', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q14': {'service': 'trauma', 'place': 'Indore', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q15': {'service': 'trauma', 'place': '271801', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q16': {'service': 'trauma', 'place': 'Kohima', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q17': {'service': 'emergency', 'place': 'Hyderabad', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q18': {'service': 'emergency', 'place': 'Bhubaneswar', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q19': {'service': 'emergency', 'place': '764020', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': 'yes', 'human_one_facility': 'yes', 'human_note': 'Spot check: judgement correct.'}, 'q20': {'service': 'emergency', 'place': 'Tawang', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'no', 'human_quote_supports': 'yes', 'human_one_facility': 'yes', 'human_note': 'Agree with Claude: emergency care quote is direct; "24/7" and "24x7" are the same claim.'}, 'q21': {'service': 'burn care', 'place': 'Bengaluru', 'model_quote_supports': 'yes', 'model_one_facility': 'merged (now flagged)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'merged', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q22': {'service': 'burn care', 'place': 'Gaya', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q23': {'service': 'burn care', 'place': '852201', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': 'yes', 'human_one_facility': 'yes', 'human_note': 'Spot check: judgement correct.'}, 'q24': {'service': 'burn care', 'place': 'Mon', 'model_quote_supports': 'partly', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q25': {'service': 'cardiac surgery', 'place': 'Ahmedabad', 'model_quote_supports': 'partly', 'model_one_facility': 'merged (now flagged)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'merged', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q26': {'service': 'cardiac surgery', 'place': 'Bhopal', 'model_quote_supports': 'partly', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'partly', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q27': {'service': 'cardiac surgery', 'place': '822101', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q28': {'service': 'cardiac surgery', 'place': 'Shillong', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q29': {'service': 'MRI', 'place': 'Pune', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q30': {'service': 'MRI', 'place': 'Darbhanga', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q31': {'service': 'MRI', 'place': '473001', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q32': {'service': 'MRI', 'place': 'Leh', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q33': {'service': 'CT scan', 'place': 'Kolkata', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': 'yes', 'human_one_facility': 'yes', 'human_note': 'Spot check: correct. A "partnership" (e.g. PET-CT with Neuberg Diagnostics) is not outsourcing, but equipment availability must be confirmed by phone.'}, 'q34': {'service': 'CT scan', 'place': 'Varanasi', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}, 'q35': {'service': 'CT scan', 'place': '845401', 'model_quote_supports': 'yes', 'model_one_facility': 'yes (no merge signal)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'yes', 'models_agree': 'yes', 'human_quote_supports': 'yes', 'human_one_facility': 'yes', 'human_note': 'Spot check: correct. PIN 845401 is only known to its area (last digits ambiguous), so the answer should offer 3 options over a wider radius.'}, 'q36': {'service': 'CT scan', 'place': 'Kargil', 'model_quote_supports': 'yes', 'model_one_facility': 'merged (now flagged)', 'xcheck_quote_supports': 'yes', 'xcheck_one_facility': 'merged', 'models_agree': 'yes', 'human_quote_supports': '', 'human_one_facility': '', 'human_note': ''}}

def count(col):
    return dict(Counter(v[col] for v in REVIEW.values() if v[col]).most_common())

print(f'Top-1 answers reviewed: {len(REVIEW)}')
print('Quote supports the service   model A:', count('model_quote_supports'), '| model B:', count('xcheck_quote_supports'))
print('One real facility            model A:', count('model_one_facility'), '| model B:', count('xcheck_one_facility'))
agree = sum(1 for v in REVIEW.values() if v['models_agree'] == 'yes')
print(f'Models agree on both questions: {agree}/{len(REVIEW)}')
human = {k: v for k, v in REVIEW.items() if v['human_quote_supports']}
print(f'\nHuman adjudication ({len(human)} rows: every disagreement plus a random sample of agreements):')
for k, v in human.items():
    print(f"  {k} {v['service']:<16} {v['place']:<14} quote supports: {v['human_quote_supports']:<7} one facility: {v['human_one_facility']:<9} {v['human_note'][:70]}")
print('\nNo reviewer found a quote that contradicts the service. "Partly" means the quote proves the service is offered but')
print('not that it is available right now (e.g. emergency C-section readiness): the card states exactly that and says what to confirm by phone.')


Top-1 answers reviewed: 36
Quote supports the service   model A: {'yes': 32, 'partly': 4} | model B: {'yes': 27, 'partly': 9}
One real facility            model A: {'yes (no merge signal)': 32, 'merged (now flagged)': 3, 'duplicate (now merged and flagged)': 1} | model B: {'yes': 32, 'merged': 3, 'duplicate': 1}
Models agree on both questions: 31/36

Human adjudication (9 rows: every disagreement plus a random sample of agreements):
  q05 emergency C-section Mumbai         quote supports: partly  one facility: duplicate Two questions: (1) does the quote prove the hospital performs C-sectio
  q06 emergency C-section Lucknow        quote supports: partly  one facility: yes       Two questions: (1) does the quote prove the hospital performs C-sectio
  q07 emergency C-section Aurangabad, Bihar quote supports: partly  one facility: yes       Two questions: (1) does the quote prove the hospital performs C-sectio
  q08 emergency C-section Mon            quote supports: partly  one facility: y

## Part 6 · Impact: where the referral network has the largest gaps

For every India Post PIN area with a reliable centre (15,174 areas, 702 districts, weighted by number of post offices as a
rough stand-in for settlements), we measure the straight-line distance to the nearest facility with **Strong or
Moderate** evidence for each service, using exactly the rules above. A district is an **access gap** when at least half of
its post offices are more than 50 km away. We then put NFHS-5 need next to it: the caesarean rate (UN emergency-obstetric-care
guidelines treat a population rate below 5% as unmet need) and institutional births for maternal care, and the share of
adults with raised blood sugar for dialysis. Where fewer than 3 hospitals of any kind are listed within 50 km, the gap is
most likely in the **data**; otherwise it is a **service** gap.


In [26]:
started = time.perf_counter()
table, nfhs_match, national = district_table(index, list(SERVICES), nfhs_rows)
print(f'{len(table)} districts in {time.perf_counter() - started:.0f} s; NFHS matched: {nfhs_match}')
print('\nShare of post offices within each straight-line distance of a facility with credible evidence:')
rows = [{'service': SERVICES[s]['label'], '<=25 km': f"{v['within_25km']:.0%}", '<=50 km': f"{v['within_50km']:.0%}",
         '<=100 km': f"{v['within_100km']:.0%}", 'access-gap districts': f"{v['districts_with_access_gap']}/{v['districts']}"}
        for s, v in sorted(national.items(), key=lambda kv: -kv[1]['within_50km'])]
show(rows, ['service', '<=25 km', '<=50 km', '<=100 km', 'access-gap districts'])


702 districts in 2 s; NFHS matched: {'exact': 537, 'close_spelling': 35, 'unmatched': 130}

Share of post offices within each straight-line distance of a facility with credible evidence:
service                         <=25 km  <=50 km  <=100 km  access-gap districts
------------------------------  -------  -------  --------  --------------------
Emergency care                  52%      84%      99%       123/702             
CT scan                         38%      70%      95%       214/702             
Trauma care                     36%      67%      91%       269/702             
Dialysis                        35%      65%      91%       268/702             
NICU (newborn intensive care)   30%      59%      90%       308/702             
MRI scan                        29%      58%      90%       308/702             
Cardiac surgery                 24%      49%      82%       380/702             
C-section (caesarean delivery)  21%      46%      80%       405/702             
Bur

In [27]:
maternal = maternal_priorities(table)
dialysis = dialysis_priorities(table)
for r in maternal + dialysis:
    r['gap'] = 'data' if gap_kind(r).startswith('data') else 'service'
for r in maternal:
    r['far'] = f"{r['c_section_share_far']:.0%}"
for r in dialysis:
    r['far'] = f"{r['dialysis_share_far']:.0%}"

print(f'MATERNAL PRIORITIES: {len(maternal)} districts far from documented C-section care with an NFHS caesarean rate below 5%')
show(maternal, ['district', 'state', 'c_section_median_km', 'far', 'csection_pct', 'institutional_birth_pct',
                'hospitals_listed_within_50km', 'gap'],
     ['district', 'state', 'median km', 'post offices >50 km', 'C-section %', 'inst. births %', 'hospitals <50 km', 'gap'], limit=15)
print(f'\nDIALYSIS PRIORITIES: {len(dialysis)} districts far from documented dialysis, top quarter for raised blood sugar')
show(dialysis, ['district', 'state', 'dialysis_median_km', 'far', '_sugar', 'hospitals_listed_within_50km', 'gap'],
     ['district', 'state', 'median km', 'post offices >50 km', 'raised sugar %', 'hospitals <50 km', 'gap'], limit=12)


MATERNAL PRIORITIES: 27 districts far from documented C-section care with an NFHS caesarean rate below 5%
district               state          median km  post offices >50 km  C-section %  inst. births %  hospitals <50 km  gap    
---------------------  -------------  ---------  -------------------  -----------  --------------  ----------------  -------
Mon                    Nagaland       85.7       100%                 1.4          21.4            0                 data   
Tuensang               Nagaland       137.8      100%                 1.5          34.8            2                 data   
Phek                   Nagaland       204.2      100%                 1.7          32.2            0                 data   
Kiphire                Nagaland       177.5      100%                 2.1          34.8            0                 data   
Peren                  Nagaland       197.5      100%                 2.4          43.5            2                 data   
South West Garo Hil

In [28]:
cs = national['c_section']
by_state = Counter(r['state'] for r in maternal)
data_gaps = sum(r['gap'] == 'data' for r in maternal)
service_gaps = [r['district'] + ', ' + r['state'] for r in maternal if r['gap'] == 'service']
print(f"1. Documented C-section care is within 50 km of only {cs['within_50km']:.0%} of post offices; {cs['districts_with_access_gap']} of {cs['districts']} districts are access gaps.")
print(f"2. {len(maternal)} districts are both far from it and show unmet need; most are in " +
      ', '.join(f'{s} ({n})' for s, n in by_state.most_common(4)) + '.')
contrast = [r for r in maternal if r['institutional_birth_pct'] is not None]
low = min(contrast, key=lambda r: r['institutional_birth_pct'])
high = max(contrast, key=lambda r: r['institutional_birth_pct'])
print(f"3. Two different problems: in {low['district']} only {low['institutional_birth_pct']:.0f}% of births are in a facility (women do not reach care);")
print(f"   in {high['district']} {high['institutional_birth_pct']:.0f}% are, yet only {high['csection_pct']}% are caesareans (facilities that rarely operate).")
print(f"4. {data_gaps} of {len(maternal)} maternal priorities have fewer than 3 hospitals of any kind listed nearby: the dataset (88% private) probably misses")
print(f"   their public hospitals. Service gaps (hospitals listed, none documents C-section): {'; '.join(service_gaps) or 'none'}.")
print(f"5. {len(dialysis)} districts combine poor documented dialysis access with high raised-blood-sugar prevalence;"
      f" {sum(r['gap'] == 'service' for r in dialysis)} of them are service gaps.")


1. Documented C-section care is within 50 km of only 46% of post offices; 405 of 702 districts are access gaps.
2. 27 districts are both far from it and show unmet need; most are in Nagaland (8), Meghalaya (5), Rajasthan (3), Uttar Pradesh (3).
3. Two different problems: in Mon only 21% of births are in a facility (women do not reach care);
   in Karauli 98% are, yet only 2.8% are caesareans (facilities that rarely operate).
4. 21 of 27 maternal priorities have fewer than 3 hospitals of any kind listed nearby: the dataset (88% private) probably misses
   their public hospitals. Service gaps (hospitals listed, none documents C-section): Mokokchung, Nagaland; East Garo Hills, Meghalaya; Alirajpur, Madhya Pradesh; South West Khasi Hills, Meghalaya; Sahebganj, Jharkhand; Barmer, Rajasthan.
5. 36 districts combine poor documented dialysis access with high raised-blood-sugar prevalence; 20 of them are service gaps.


### Recommended actions

| # | Action | Where | Why |
| --- | --- | --- | --- |
| 1 | **Fix the data before building.** Add public facilities (district hospitals, CHCs / First Referral Units, national dialysis programme centres) with their documented services. | The "data gap" districts above | The gap may be in the listing, not on the ground. Once added, CareMatch shows them to health workers with no code change. |
| 2 | **Verify, then upgrade, surgical obstetric capacity.** Confirm by phone which listed hospitals can do an emergency caesarean (CareMatch prints the questions); where none can, prioritise the district hospital for emergency obstetric staffing (obstetrician, anaesthetist, blood storage). | The maternal "service gap" districts, and districts where births happen in facilities but caesareans are rare | High institutional births with a very low caesarean rate points to missing surgical capacity, not missing buildings. |
| 3 | **Pre-arranged referral routes and transport.** Name the nearest facility with credible evidence for each priority district (CareMatch's answer from the district centre) and agree a referral and ambulance protocol with it. | Northeast priority districts, where the nearest documented C-section care is often 100+ km away | When the nearest capable facility is hours away, a known destination saves the most time. |
| 4 | **Site the next dialysis units by need × distance.** Use the dialysis priority list to decide where new units, or verification of existing ones, go first. | The dialysis "service gap" districts | Patients on dialysis travel two to three times a week; distance decides whether they keep up. |


## Part 7 · Which choice changes, and where should a planner call first?

Three short demonstrations built on the same engine. None of them changes what a patient is recommended.


**Same search, three orders.** Best evidence, nearest and compare-listed-prices can put different facilities first.
The comparison names each first choice with its quote, and says when the starting point is too approximate to trust
a small distance difference.


In [29]:
from _carematch_submission.strategy_compare import compare_strategies
service, place = 'dialysis', 'Delhi'
comparison = compare_strategies(index, service, place, recommend(index, service, place))
print(place, service, comparison['summary'], sep=' · ')
for opt in comparison['options']:
    print(opt['label'], opt['name'], opt['distance_km'], opt['tier'])
    if opt['citation']:
        print(' ', opt['facility_id'], opt['citation']['field'], opt['citation']['quote'])

Delhi · dialysis · Moolchand Hospital is 1.6 km closer to the search centre by straight line. The records give Moderate evidence there and Strong evidence at Dashmesh Hospital. Price comparison falls back to distance. The starting point is approximate (about 10.1 km). Your closest option may differ.
Best evidence Dashmesh Hospital 9.2 strong
  afb09018-c676-418f-9151-f3b6a504614f procedure Dialysis procedures including hemodialysis and peritoneal dialysis
Nearest Moolchand Hospital 7.6 moderate
  2ddc7b9f-65ac-40d1-a1a0-6b649ff9d7b2 capability Centres of Excellence include Cancer Institute, Heart Institute, Neurosciences Institute, Orthopaedics Institute, Paediatrics Institute, Women’s Institute, Urology Institute, Renalcare & Dialysis, Gastrosciences Institute, Moolchand Diagnostics and The Moolchand Clinic
Compare listed prices Moolchand Hospital 7.6 moderate
  2ddc7b9f-65ac-40d1-a1a0-6b649ff9d7b2 capability Centres of Excellence include Cancer Institute, Heart Institute, Neuroscienc

**Ten calls, most new coverage.** For C-section within 50 km, 7,402 of 14,008 reliably located PIN areas already
have a facility with credible evidence. Each candidate below has obstetric or surgical leads but no credible C-section
claim. Choosing greedily (each call adds the most still-uncovered PINs) is compared with simply taking the candidates
with the largest individual reach, which overlap. Gains assume the facility confirms the service.


In [30]:
from _carematch_submission.calls import current_coverage, candidates, greedy_plan, topk_baseline
coverage = current_coverage(index)
leads, checks = candidates(index, cov=coverage)
plan = greedy_plan(leads, budget=10)
print('Possible extra PIN coverage if the selected facilities confirm the service, within 50 km.')
for k in (3, 5, 10):
    actual = plan[:k]
    print(k, 'calls:', sum(step['marginal_gain'] for step in actual), 'PINs; independent-gain baseline:',
          topk_baseline(leads, k)['unique_new_pins'])
if plan:
    first = plan[0]['candidate']
    followup = greedy_plan(leads, budget=2, exclude=[first['facility_id']])
    print('If the first call answers no, two calls remain:', sum(r['marginal_gain'] for r in followup), 'potential PINs.')
    for step in plan[:3]:
        print(step['candidate']['name'], step['marginal_gain'], step['candidate']['leads'][:1])

Possible extra PIN coverage if the selected facilities confirm the service, within 50 km.
3 calls: 365 PINs; independent-gain baseline: 152
5 calls: 554 PINs; independent-gain baseline: 154
10 calls: 936 PINs; independent-gain baseline: 154
If the first call answers no, two calls remain: 258 potential PINs.
SH Medical Centre 150 [('obstetric care', 'capability', 'Obstetrics and Gynecology department')]
SanthaRam Hospital 110 [('surgical readiness', 'capability', 'Blood Bank operates 24/7.')]
Anu Specialty Hospital 105 [('obstetric care', 'capability', 'Cardiology, General Medicine, Orthopaedics, Urology, Gynecology, Nephrology, Neurology, and Pulmonology departments')]


**A statistical second opinion on whom to verify.** A logistic regression is trained with Strong C-section evidence
as a proxy positive and every other record unlabeled, using no C-section or maternity wording at all, and is evaluated
on whole regions of India it never saw. It is compared with a simple rule (hospitals first, then more specialties).


In [31]:
from _carematch_submission.lead_model import build_dataset, cross_validate, baseline_scores, metrics
data = build_dataset(index)
scores, _ = cross_validate(data, C=0.3)
print('Region-held-out recovery of proxy positives. Actual availability requires a call.')
for name, predictions in [('Logistic regression', scores), ('Hospital and specialty baseline', baseline_scores(data))]:
    result = metrics(data['y'], predictions)
    print(f"{name}: ROC-AUC {result['roc_auc']}, PR-AUC {result['pr_auc']} (base rate {result['prevalence']}), "
          f"precision@50 {result['precision_at_50']}, precision@200 {result['precision_at_200']}")

Region-held-out recovery of proxy positives. Actual availability requires a call.
Logistic regression: ROC-AUC 0.839, PR-AUC 0.053 (base rate 0.0125), precision@50 0.04, precision@200 0.075
Hospital and specialty baseline: ROC-AUC 0.757, PR-AUC 0.031 (base rate 0.0125), precision@50 0.1, precision@200 0.025


## Limits, and what we would do next

- **Documented, not actual, capacity.** A facility saying it offers a service is evidence, not proof; availability today is
  never known. The app says so on every card and turns uncertainty into questions to ask by phone.
- **Straight-line distance.** Road travel, especially in hills, is longer; distances are labelled straight-line everywhere.
- **The dataset under-represents public facilities** (88% private, 5% public), so rural gaps are partly data gaps.
- **NFHS-5 is 2019–21** with older district boundaries (the match rate is printed above); it is used only for the planner
  view and never changes a patient's ranking.
- **English-only rules.** Claim classification uses English patterns calibrated on this table.
- **Prices are rarely documented.** Only a handful of records state a patient price for the service itself, so the
  price order usually falls back to distance and says so; a price is shown only as quoted, with its unit and conditions.
- **Verification leads are leads.** The call plan and the lead model choose whom to phone; neither changes what a patient
  is recommended, and coverage gains assume the facility confirms the service.

Next: add public facility registries, road travel time, a phone-confirmation loop so confirmed answers improve future
rankings, and local-language labels for health workers.

Repository: https://github.com/DDaniel-code/carematch. Design and evaluation details: `docs/system_design.md`, `docs/evidence_calibration.md`,
`docs/evaluation.md`, `docs/impact.md`.
